# Questionnaire Scoring and Feature Mapping

## Project
AI-Powered Career Guidance Platform — Psychometric Assessment & ML-Based Career Matching

## Aim
To transform the student's responses to 120 assessment questions into the 25 input features required by the trained Soft Voting Ensemble model.

## Objectives
1. Load and inspect the existing questionnaire and data dictionary.
2. Understand the answer formats and scoring rules for all 120 questions.
3. Map question responses to interest, aptitude, personality, skill, and profile features.
4. Calculate the student's assessment scores using defined scoring rules.
5. Validate the resulting 25-feature profile before passing it to the trained ML model.

## Workflow
120 Question Responses → Response Validation → Scoring → Feature Mapping → 25-Feature Student Profile → Career Prediction

## Important Note
Scoring rules and feature mappings must be derived from the existing questionnaire and its data dictionary. No scoring rules or question mappings will be assumed without verification.

## Inspect the existing files

In [3]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

print("Current project directory:", PROJECT_DIR)
print("\nRelevant files:")

for pattern in [
    "*Questions*.xlsx",
    "data_dictionary.json",
    "career_guidance_processed.csv",
    "career_guidance_raw_synthetic.csv",
    "README.md"
]:
    for file_path in sorted(PROJECT_DIR.glob(pattern)):
        print(file_path.name)

Current project directory: C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset

Relevant files:
AI_Career_Guidance_Questions.xlsx
data_dictionary.json
career_guidance_processed.csv
career_guidance_raw_synthetic.csv
README.md


In [4]:

# Step 17A: Inspect the 120-question questionnaire workbook

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd()
QUESTION_FILE = PROJECT_DIR / "AI_Career_Guidance_Questions.xlsx"

# List all sheets in the Excel workbook
excel_file = pd.ExcelFile(QUESTION_FILE)

print("Workbook:", QUESTION_FILE.name)
print("\nAvailable sheets:")
for sheet in excel_file.sheet_names:
    print("-", sheet)

# Inspect the dimensions and column names of each sheet
for sheet in excel_file.sheet_names:
    df = pd.read_excel(QUESTION_FILE, sheet_name=sheet)

    print("\n" + "=" * 70)
    print("SHEET:", sheet)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    display(df.head(5))


Workbook: AI_Career_Guidance_Questions.xlsx

Available sheets:
- Questions Dataset

SHEET: Questions Dataset
Shape: (123, 8)
Columns: ['Question_ID', 'Section', 'Question_Text', 'Answer_Type', 'Answer_Options', 'Scoring_Method', 'Feature_Mapping', 'Purpose']


,Question_ID,Section,Question_Text,Answer_Type,Answer_Options,Scoring_Method,Feature_Mapping,Purpose
0,Q001,Interest,How interested are you in writing scripts or c...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in core programming and logic
1,Q002,Interest,How interested are you in analyzing large data...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in data science and statisti...
2,Q003,Interest,How interested are you in finding vulnerabilit...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in cybersecurity and risk mi...
3,Q004,Interest,How interested are you in sketching out wirefr...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in UI/UX and user-centric de...
4,Q005,Interest,How interested are you in coordinating team me...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in business management and l...


In [5]:

# Step 17B: Audit question sections, scoring rules, and feature mappings

import pandas as pd
from pathlib import Path

question_file = Path.cwd() / "AI_Career_Guidance_Questions.xlsx"

questions_df = pd.read_excel(
    question_file,
    sheet_name="Questions Dataset"
)

# 1. Count questions by section and answer type
print("QUESTIONS BY SECTION")
display(
    questions_df.groupby("Section")
    .size()
    .reset_index(name="Question_Count")
)

print("\nQUESTIONS BY ANSWER TYPE")
display(
    questions_df.groupby("Answer_Type", dropna=False)
    .size()
    .reset_index(name="Question_Count")
)

# 2. Check missing values in important columns
required_columns = [
    "Question_ID",
    "Section",
    "Question_Text",
    "Answer_Type",
    "Answer_Options",
    "Scoring_Method",
    "Feature_Mapping",
    "Purpose"
]

print("\nMISSING VALUES")
display(
    questions_df[required_columns]
    .isna()
    .sum()
    .reset_index(name="Missing_Count")
    .rename(columns={"index": "Column"})
)

# 3. Display all questions with their scoring and mapping details
print("\nSCORING AND FEATURE MAPPING REVIEW")

display(
    questions_df[
        [
            "Question_ID",
            "Section",
            "Answer_Type",
            "Scoring_Method",
            "Feature_Mapping",
            "Purpose"
        ]
    ].to_string(index=False)
)


QUESTIONS BY SECTION


,Section,Question_Count
0,Aptitude,12
1,Education & Career Preferences,80
2,Interest,10
3,Personality,10
4,Skills,11



QUESTIONS BY ANSWER TYPE


,Answer_Type,Question_Count
0,Agreement Scale (1-5),10
1,Dropdown,3
2,MCQ,89
3,Rating scale (1-5),10
4,Skill Scale (1-5 + N/A),11



MISSING VALUES


,Column,Missing_Count
0,Question_ID,0
1,Section,0
2,Question_Text,0
3,Answer_Type,0
4,Answer_Options,0
5,Scoring_Method,120
6,Feature_Mapping,120
7,Purpose,0



SCORING AND FEATURE MAPPING REVIEW


"Question_ID                        Section             Answer_Type                                                                 Scoring_Method     Feature_Mapping                                                               Purpose\n       Q001                       Interest      Rating scale (1-5)                                                                            NaN                 NaN                       Measures interest in core programming and logic\n       Q002                       Interest      Rating scale (1-5)                                                                            NaN                 NaN            Measures interest in data science and statistical analysis\n       Q003                       Interest      Rating scale (1-5)                                                                            NaN                 NaN                Measures interest in cybersecurity and risk mitigation\n       Q004                       Interest      Rat

In [6]:

# Step 17D: Inspect the 40 psychometric assessment questions

import pandas as pd
from pathlib import Path

question_file = Path.cwd() / "AI_Career_Guidance_Questions.xlsx"

questions_df = pd.read_excel(
    question_file,
    sheet_name="Questions Dataset"
)

assessment_sections = [
    "Interest",
    "Aptitude",
    "Personality",
    "Skills"
]

assessment_questions = questions_df[
    questions_df["Section"].isin(assessment_sections)
].copy()

for section in assessment_sections:
    section_df = assessment_questions[
        assessment_questions["Section"] == section
    ]

    print("\n" + "=" * 90)
    print(f"{section.upper()} — {len(section_df)} QUESTIONS")
    print("=" * 90)

    for _, row in section_df.iterrows():
        print(f"\n{row['Question_ID']}: {row['Question_Text']}")
        print(f"Answer type: {row['Answer_Type']}")
        print(f"Answer options: {row['Answer_Options']}")
        print(f"Purpose: {row['Purpose']}")



INTEREST — 10 QUESTIONS

Q001: How interested are you in writing scripts or code to automate complex, repetitive tasks?
Answer type: Rating scale (1-5)
Answer options: 1 - Not interested at all to 5 - Extremely interested
Purpose: Measures interest in core programming and logic

Q002: How interested are you in analyzing large datasets to identify hidden trends and predict future outcomes?
Answer type: Rating scale (1-5)
Answer options: 1 - Not interested at all to 5 - Extremely interested
Purpose: Measures interest in data science and statistical analysis

Q003: How interested are you in finding vulnerabilities in computer networks and systems to prevent unauthorized access?
Answer type: Rating scale (1-5)
Answer options: 1 - Not interested at all to 5 - Extremely interested
Purpose: Measures interest in cybersecurity and risk mitigation

Q004: How interested are you in sketching out wireframes and designing the visual layout of a mobile application?
Answer type: Rating scale (1-5)
An

In [7]:
# Step 17E: Inspect the model's required input features

import json
import pandas as pd
from pathlib import Path

project_dir = Path.cwd()

# Load the training dataset and data dictionary
train_df = pd.read_csv(project_dir / "career_guidance_train.csv")

with open(project_dir / "data_dictionary.json", "r", encoding="utf-8") as file:
    data_dictionary = json.load(file)

# Identify the target and input features
target_column = "recommended_career_domain"

feature_columns = [
    column for column in train_df.columns
    if column != target_column
]

print("=" * 80)
print("MODEL INPUT FEATURE AUDIT")
print("=" * 80)

print(f"\nNumber of input features: {len(feature_columns)}")
print(f"Target column: {target_column}")
print(f"Training dataset shape: {train_df.shape}")

print("\nFEATURE NAMES AND DATA TYPES")
print("-" * 80)

for column in feature_columns:
    print(f"{column:<40} {train_df[column].dtype}")

print("\nCATEGORICAL FEATURES")
print("-" * 80)

categorical_features = train_df[feature_columns].select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

for column in categorical_features:
    print(f"{column}: {sorted(train_df[column].dropna().astype(str).unique())}")

print("\nNUMERICAL FEATURES")
print("-" * 80)

numerical_features = [
    column for column in feature_columns
    if column not in categorical_features
]

for column in numerical_features:
    print(column)

print("\nCAREER DOMAINS IN THE TRAINING DATA")
print("-" * 80)

print(
    train_df[target_column]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nDATA DICTIONARY STRUCTURE")
print("-" * 80)

if isinstance(data_dictionary, dict):
    print("Top-level keys:", list(data_dictionary.keys()))

    for key, value in data_dictionary.items():
        if isinstance(value, dict):
            print(f"\n{key}:")
            print("  Available entries:", list(value.keys())[:15])
        elif isinstance(value, list):
            print(f"\n{key}: list containing {len(value)} items")
        else:
            print(f"\n{key}: {value}")
else:
    print("Data dictionary type:", type(data_dictionary).__name__)

print("\nFeature audit completed.")

MODEL INPUT FEATURE AUDIT

Number of input features: 25
Target column: recommended_career_domain
Training dataset shape: (1920, 26)

FEATURE NAMES AND DATA TYPES
--------------------------------------------------------------------------------
age_group                                str
education_level                          str
experience_stage                         str
region_type                              str
interest_technology                      float64
interest_data                            float64
interest_problem_solving                 float64
interest_creativity                      float64
interest_people                          float64
interest_helping                         float64
aptitude_logical                         float64
aptitude_numerical                       float64
aptitude_verbal                          float64
aptitude_spatial                         float64
aptitude_critical_thinking               float64
trait_openness                        

C:\Users\susha\AppData\Local\Temp\ipykernel_19008\3147588227.py:40: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = train_df[feature_columns].select_dtypes(


In [8]:
# Step 17F: Create a proposed mapping for the 40 assessment questions

import pandas as pd

mapping_rows = [
    # Interest questions
    ("Q001", "interest_technology", "Direct rating"),
    ("Q002", "interest_data", "Direct rating"),
    ("Q003", "interest_technology", "Shared feature; mapping needs validation"),
    ("Q004", "interest_creativity", "Direct rating"),
    ("Q005", "interest_people", "Shared feature; mapping needs validation"),
    ("Q006", "interest_helping", "Direct rating"),
    ("Q007", "interest_people", "Shared feature; mapping needs validation"),
    ("Q008", "interest_creativity", "Shared feature; mapping needs validation"),
    ("Q009", "interest_data", "Shared feature; mapping needs validation"),
    ("Q010", "interest_problem_solving", "Shared feature; mapping needs validation"),

    # Aptitude questions: score only after verifying the answer key
    ("Q011", "aptitude_numerical", "Provisional; verify answer key"),
    ("Q012", "aptitude_logical", "Ambiguous wording; review before scoring"),
    ("Q013", "aptitude_numerical", "Provisional; verify answer key"),
    ("Q014", "aptitude_logical", "Provisional; verify answer key"),
    ("Q015", "aptitude_critical_thinking", "Provisional; verify answer key"),
    ("Q016", "aptitude_logical", "Loop structure needs clarification"),
    ("Q017", "aptitude_numerical", "Provisional; verify answer key"),
    ("Q018", "aptitude_critical_thinking", "Provisional; verify answer key"),
    ("Q019", "aptitude_logical", "Provisional; verify answer key"),
    ("Q020", "aptitude_numerical", "Provisional; verify answer key"),

    # Personality questions
    ("Q021", "trait_agreeableness", "Provisional; construct alignment review"),
    ("Q022", "trait_conscientiousness", "Reverse/construct alignment review"),
    ("Q023", "trait_extraversion", "Leadership is not directly measured by this feature"),
    ("Q024", "trait_conscientiousness", "Provisional; construct alignment review"),
    ("Q025", "trait_openness", "Direct conceptual alignment"),
    ("Q026", "trait_emotional_stability", "Direct conceptual alignment"),
    ("Q027", "trait_conscientiousness", "Provisional; construct alignment review"),
    ("Q028", "trait_extraversion", "Direct conceptual alignment"),
    ("Q029", "trait_openness", "Analytical thinking is not openness; review"),
    ("Q030", "trait_openness", "Adaptability is not openness; review"),

    # Skills questions
    ("Q031", "skill_programming", "Direct rating"),
    ("Q032", "skill_data_analysis", "Direct rating"),
    ("Q033", "interest_creativity", "No matching design-skill feature; proxy only"),
    ("Q034", "skill_data_analysis", "Finance skill lacks a dedicated feature"),
    ("Q035", "interest_technology", "No matching cybersecurity-skill feature; proxy only"),
    ("Q036", "skill_leadership", "Project management is not identical to leadership"),
    ("Q037", "interest_creativity", "No matching marketing-skill feature; proxy only"),
    ("Q038", "skill_communication", "Legal skill lacks a dedicated feature"),
    ("Q039", "interest_problem_solving", "No matching engineering-skill feature; proxy only"),
    ("Q040", "skill_research", "Direct rating"),
]

mapping_df = pd.DataFrame(
    mapping_rows,
    columns=["Question_ID", "Proposed_Feature", "Review_Note"]
)

# Add question text and answer format for traceability
question_details = questions_df[
    ["Question_ID", "Question_Text", "Answer_Type", "Answer_Options"]
].copy()

mapping_df = mapping_df.merge(
    question_details,
    on="Question_ID",
    how="left",
    validate="one_to_one"
)

print("Proposed mapping rows:", len(mapping_df))
print("\nProposed question-to-feature mapping:")
display(mapping_df)

print("\nQuestions mapped to each proposed feature:")
display(
    mapping_df.groupby("Proposed_Feature")["Question_ID"]
    .apply(list)
    .reset_index(name="Question_IDs")
)

Proposed mapping rows: 40

Proposed question-to-feature mapping:


,Question_ID,Proposed_Feature,Review_Note,Question_Text,Answer_Type,Answer_Options
0,Q001,interest_technology,Direct rating,How interested are you in writing scripts or c...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...
1,Q002,interest_data,Direct rating,How interested are you in analyzing large data...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...
2,Q003,interest_technology,Shared feature; mapping needs validation,How interested are you in finding vulnerabilit...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...
3,Q004,interest_creativity,Direct rating,How interested are you in sketching out wirefr...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...
4,Q005,interest_people,Shared feature; mapping needs validation,How interested are you in coordinating team me...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...
5,Q006,interest_helping,Direct rating,How interested are you in studying human anato...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...
6,Q007,interest_people,Shared feature; mapping needs validation,How interested are you in explaining difficult...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...
7,Q008,interest_creativity,Shared feature; mapping needs validation,How interested are you in analyzing search eng...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...
8,Q009,interest_data,Shared feature; mapping needs validation,How interested are you in auditing financial r...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...
9,Q010,interest_problem_solving,Shared feature; mapping needs validation,How interested are you in designing physical m...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...



Questions mapped to each proposed feature:


,Proposed_Feature,Question_IDs
0,aptitude_critical_thinking,"[Q015, Q018]"
1,aptitude_logical,"[Q012, Q014, Q016, Q019]"
2,aptitude_numerical,"[Q011, Q013, Q017, Q020]"
3,interest_creativity,"[Q004, Q008, Q033, Q037]"
4,interest_data,"[Q002, Q009]"
5,interest_helping,[Q006]
6,interest_people,"[Q005, Q007]"
7,interest_problem_solving,"[Q010, Q039]"
8,interest_technology,"[Q001, Q003, Q035]"
9,skill_communication,[Q038]


In [9]:
# Step 17G: Inspect original feature definitions and dataset documentation

import json
from pathlib import Path
import pandas as pd

project_dir = Path.cwd()

# Load the data dictionary
with open(project_dir / "data_dictionary.json", "r", encoding="utf-8") as file:
    data_dictionary = json.load(file)

print("=" * 80)
print("ORIGINAL DATA DICTIONARY — FEATURE DEFINITIONS")
print("=" * 80)

for column_info in data_dictionary["columns"]:
    if isinstance(column_info, dict):
        print("\n" + "-" * 60)
        for key, value in column_info.items():
            print(f"{key}: {value}")
    else:
        print(column_info)

# Inspect dataset documentation, if available
readme_path = project_dir / "README.md"

if readme_path.exists():
    print("\n" + "=" * 80)
    print("DATASET README")
    print("=" * 80)
    print(readme_path.read_text(encoding="utf-8"))

# Show a few original synthetic records for context
raw_path = project_dir / "career_guidance_raw_synthetic.csv"

if raw_path.exists():
    raw_df = pd.read_csv(raw_path)
    print("\n" + "=" * 80)
    print("SAMPLE SYNTHETIC RECORDS")
    print("=" * 80)
    display(raw_df.head(5))

ORIGINAL DATA DICTIONARY — FEATURE DEFINITIONS

------------------------------------------------------------
name: profile_id
type: identifier
description: Synthetic profile identifier; exclude from ML features.

------------------------------------------------------------
name: age_group
type: categorical
description: Broad age band; use cautiously and assess fairness.

------------------------------------------------------------
name: education_level
type: categorical
description: Educational background category.

------------------------------------------------------------
name: experience_stage
type: categorical
description: Current study/career stage.

------------------------------------------------------------
name: region_type
type: categorical
description: Broad region type; exclude if it creates geographic bias.

------------------------------------------------------------
name: interest_technology
type: numeric_1_to_10_raw_or_0_to_1_processed
description: Synthetic assessmen

,profile_id,age_group,education_level,experience_stage,region_type,interest_technology,interest_data,interest_problem_solving,interest_creativity,interest_people,...,trait_conscientiousness,trait_extraversion,trait_agreeableness,trait_emotional_stability,skill_programming,skill_data_analysis,skill_research,skill_communication,skill_leadership,recommended_career_domain
0,CG00001,15-18,B.Sc Mathematics,1-3 years,Rural,6.3,10.0,9.6,3.8,3.9,...,9.6,7.7,5.8,6.5,5.4,10.0,5.9,6.7,4.8,Data Science & Analytics
1,CG00002,15-18,B.Sc Mathematics,Student,Rural,5.4,9.9,6.6,6.7,7.7,...,6.8,4.8,8.9,8.0,6.9,7.9,6.4,6.2,7.4,Finance & Accounting
2,CG00003,31+,B.Tech Mechanical,Student,Rural,8.1,8.5,9.0,3.6,5.5,...,7.5,6.2,7.8,7.0,9.3,6.4,6.2,7.5,4.7,Engineering & Core Technology
3,CG00004,23-30,B.Tech Mechanical,Student,Semi-urban,8.8,8.2,9.1,5.3,6.3,...,8.7,5.3,7.0,6.7,8.8,8.1,6.8,5.3,4.9,Engineering & Core Technology
4,CG00005,15-18,BBA,Entry-level,Semi-urban,5.4,6.7,5.5,7.4,10.0,...,5.6,8.6,8.3,7.2,3.5,6.1,5.1,8.4,9.0,Business & Management


In [10]:
# Step 17H: Examine feature patterns by career domain

import pandas as pd
from pathlib import Path

project_dir = Path.cwd()
train_df = pd.read_csv(project_dir / "career_guidance_train.csv")

target = "recommended_career_domain"

numeric_features = [
    column for column in train_df.select_dtypes(include="number").columns
    if column != target
]

# Mean score of each feature for every career domain
career_feature_means = (
    train_df.groupby(target)[numeric_features]
    .mean()
    .round(2)
)

print("AVERAGE FEATURE SCORES BY CAREER DOMAIN")
print("Scores are on the original 1–10 synthetic-data scale.\n")

display(career_feature_means)

print("\nFEATURE VARIATION ACROSS CAREER DOMAINS")
print("Higher values indicate greater differences between domain averages.\n")

feature_variation = (
    career_feature_means.max() - career_feature_means.min()
).sort_values(ascending=False)

display(
    feature_variation.rename("Domain_Mean_Range")
    .to_frame()
)

print("\nDataset limitation:")
print(
    "These are patterns in synthetic data, not validated measurements "
    "of real students or evidence of real-world career suitability."
)


AVERAGE FEATURE SCORES BY CAREER DOMAIN
Scores are on the original 1–10 synthetic-data scale.



,interest_technology,interest_data,interest_problem_solving,interest_creativity,interest_people,interest_helping,aptitude_logical,aptitude_numerical,aptitude_verbal,aptitude_spatial,...,trait_openness,trait_conscientiousness,trait_extraversion,trait_agreeableness,trait_emotional_stability,skill_programming,skill_data_analysis,skill_research,skill_communication,skill_leadership
recommended_career_domain,,,,,,,,,,,,,,,,,,,,,
Business & Management,6.99,6.01,7.11,7.82,8.11,6.85,6.91,6.97,6.99,7.88,...,8.02,7.85,7.99,8.08,7.85,6.16,6.04,5.98,7.98,8.76
Creative Arts & Media,6.58,5.00,7.04,9.40,8.83,7.04,7.18,6.02,6.96,8.00,...,6.93,6.94,8.79,6.91,6.93,6.22,4.99,5.84,8.64,8.61
Cybersecurity,8.70,6.77,7.96,6.01,6.89,4.96,7.85,7.91,8.03,7.96,...,8.11,8.01,6.09,7.81,6.93,7.37,6.67,8.87,6.80,5.99
Data Science & Analytics,7.85,8.72,7.83,7.15,6.06,4.88,8.79,9.03,7.97,7.12,...,6.97,7.89,6.75,6.90,5.87,6.81,8.72,5.82,6.93,6.01
Digital Marketing,7.01,5.97,7.83,8.72,7.97,7.01,7.08,6.92,6.93,7.97,...,7.89,7.05,8.02,8.02,7.99,6.05,6.09,5.99,8.75,8.02
Education & Research,6.81,8.02,7.08,7.07,8.19,7.85,7.96,7.93,6.97,6.96,...,8.03,7.82,7.82,7.91,7.92,7.11,7.91,6.19,7.84,7.97
Engineering & Core Technology,8.04,8.03,7.80,6.14,5.96,5.20,8.84,7.88,7.90,7.79,...,7.01,8.10,5.90,6.90,5.90,8.00,7.04,7.18,5.91,4.96
Finance & Accounting,6.07,8.86,5.95,6.01,7.17,5.02,8.02,8.81,6.99,7.06,...,6.87,7.98,6.20,7.95,7.14,6.04,8.58,5.84,5.98,6.71
Healthcare & Life Sciences,7.01,6.83,7.08,6.24,8.02,7.93,8.02,7.04,6.99,6.95,...,8.04,6.98,7.93,7.01,7.88,6.04,6.13,4.89,7.03,8.13



FEATURE VARIATION ACROSS CAREER DOMAINS
Higher values indicate greater differences between domain averages.



,Domain_Mean_Range
interest_helping,3.98
skill_research,3.98
skill_data_analysis,3.90
interest_data,3.88
skill_leadership,3.80
interest_creativity,3.42
aptitude_numerical,3.12
trait_extraversion,2.92
interest_people,2.89
skill_communication,2.87



Dataset limitation:
These are patterns in synthetic data, not validated measurements of real students or evidence of real-world career suitability.


In [11]:
# Step 17I: Verify numerical feature scales and missing values

import pandas as pd
from pathlib import Path

project_dir = Path.cwd()
train_df = pd.read_csv(project_dir / "career_guidance_train.csv")
processed_df = pd.read_csv(project_dir / "career_guidance_processed.csv")

target = "recommended_career_domain"

numeric_features = [
    column
    for column in train_df.select_dtypes(include="number").columns
    if column != target
]

scale_summary = pd.DataFrame({
    "Training_Min": train_df[numeric_features].min(),
    "Training_Max": train_df[numeric_features].max(),
    "Training_Mean": train_df[numeric_features].mean().round(2),
    "Training_Missing": train_df[numeric_features].isna().sum(),
    "Processed_Min": processed_df[numeric_features].min(),
    "Processed_Max": processed_df[numeric_features].max(),
    "Processed_Missing": processed_df[numeric_features].isna().sum()
})

print("NUMERICAL FEATURE SCALE AUDIT")
display(scale_summary)

print("\nTRAINING CATEGORICAL VALUE AUDIT")

categorical_features = [
    "age_group",
    "education_level",
    "experience_stage",
    "region_type"
]

for column in categorical_features:
    print(f"\n{column}:")
    print(sorted(train_df[column].dropna().astype(str).unique()))

print("\nTARGET LABELS")
print(sorted(train_df[target].dropna().unique()))

print("\nAudit completed.")

NUMERICAL FEATURE SCALE AUDIT


,Training_Min,Training_Max,Training_Mean,Training_Missing,Processed_Min,Processed_Max,Processed_Missing
interest_technology,1.3,10.0,7.31,0,0.033333,1.0,0
interest_data,1.0,10.0,6.94,0,0.000000,1.0,0
interest_problem_solving,1.1,10.0,7.35,0,0.011111,1.0,0
interest_creativity,1.0,10.0,7.18,0,0.000000,1.0,0
interest_people,1.0,10.0,7.41,0,0.000000,1.0,0
interest_helping,1.0,10.0,6.42,0,0.000000,1.0,0
aptitude_logical,1.8,10.0,7.85,0,0.088889,1.0,0
aptitude_numerical,2.1,10.0,7.51,0,0.122222,1.0,0
aptitude_verbal,2.5,10.0,7.27,0,0.166667,1.0,0
aptitude_spatial,2.7,10.0,7.56,0,0.088889,1.0,0



TRAINING CATEGORICAL VALUE AUDIT

age_group:
['15-18', '19-22', '23-30', '31+']

education_level:
['B.Com', 'B.Des', 'B.Ed', 'B.Pharm', 'B.Sc Biology', 'B.Sc Mathematics', 'B.Sc Nursing', 'B.Sc Statistics', 'B.Tech Biotechnology', 'B.Tech Civil', 'B.Tech Computer Science', 'B.Tech Electrical', 'B.Tech IT', 'B.Tech Mechanical', 'BA', 'BA Economics', 'BA Media', 'BA Political Science', 'BBA', 'BCA', 'BFA', 'Diploma Design', 'Diploma Engineering', 'Diploma IT', 'LLB', 'M.Sc', 'MA', 'MBA', 'MBBS', 'MCA', 'PhD']

experience_stage:
['1-3 years', '4+ years', 'Entry-level', 'Student']

region_type:
['Rural', 'Semi-urban', 'Urban']

TARGET LABELS
['Business & Management', 'Creative Arts & Media', 'Cybersecurity', 'Data Science & Analytics', 'Digital Marketing', 'Education & Research', 'Engineering & Core Technology', 'Finance & Accounting', 'Healthcare & Life Sciences', 'Law & Public Service', 'Software Development', 'UI/UX Design']

Audit completed.


In [12]:
# Step 17J: Inspect questionnaire questions Q041-Q120

import pandas as pd
from pathlib import Path

project_dir = Path.cwd()
excel_path = project_dir / "AI_Career_Guidance_Questions.xlsx"

questions_df = pd.read_excel(
    excel_path,
    sheet_name="Questions Dataset"
)

remaining_questions = questions_df[
    questions_df["Question_ID"].astype(str).str.extract(r"(\d+)")[0]
    .astype(int).between(41, 120)
]

for _, row in remaining_questions.iterrows():
    print(f"\n{row['Question_ID']} | Section: {row['Section']}")
    print(f"Question: {row['Question_Text']}")
    print(f"Answer type: {row['Answer_Type']}")
    print(f"Options: {row['Answer_Options']}")
    print(f"Purpose: {row['Purpose']}")
    print("-" * 80)

print(f"\nTotal questions inspected: {len(remaining_questions)}")


Q041 | Section: Education & Career Preferences
Question: What is your current or most recently completed level of education?
Answer type: Dropdown
Options: High School, Diploma, Bachelor's, Master's, PhD, Other
Purpose: General Education
--------------------------------------------------------------------------------

Q042 | Section: Education & Career Preferences
Question: Which broad academic field does your primary degree or certification fall under?
Answer type: Dropdown
Options: Engineering, Business, Arts, Sciences, Medicine, Law, None/Other
Purpose: General Education
--------------------------------------------------------------------------------

Q043 | Section: Education & Career Preferences
Question: What is your cumulative academic performance/GPA equivalent?
Answer type: MCQ
Options: Top 10%, Top 25%, Average, Below Average, Prefer not to say
Purpose: General Education
--------------------------------------------------------------------------------

Q044 | Section: Educati

In [13]:
# Step 17J: Inspect questionnaire questions Q041-Q120

import pandas as pd
from pathlib import Path

project_dir = Path.cwd()
excel_path = project_dir / "AI_Career_Guidance_Questions.xlsx"

questions_df = pd.read_excel(
    excel_path,
    sheet_name="Questions Dataset"
)

remaining_questions = questions_df[
    questions_df["Question_ID"].astype(str).str.extract(r"(\d+)")[0]
    .astype(int).between(41, 120)
]

for _, row in remaining_questions.iterrows():
    print(f"\n{row['Question_ID']} | Section: {row['Section']}")
    print(f"Question: {row['Question_Text']}")
    print(f"Answer type: {row['Answer_Type']}")
    print(f"Options: {row['Answer_Options']}")
    print(f"Purpose: {row['Purpose']}")
    print("-" * 80)

print(f"\nTotal questions inspected: {len(remaining_questions)}")


Q041 | Section: Education & Career Preferences
Question: What is your current or most recently completed level of education?
Answer type: Dropdown
Options: High School, Diploma, Bachelor's, Master's, PhD, Other
Purpose: General Education
--------------------------------------------------------------------------------

Q042 | Section: Education & Career Preferences
Question: Which broad academic field does your primary degree or certification fall under?
Answer type: Dropdown
Options: Engineering, Business, Arts, Sciences, Medicine, Law, None/Other
Purpose: General Education
--------------------------------------------------------------------------------

Q043 | Section: Education & Career Preferences
Question: What is your cumulative academic performance/GPA equivalent?
Answer type: MCQ
Options: Top 10%, Top 25%, Average, Below Average, Prefer not to say
Purpose: General Education
--------------------------------------------------------------------------------

Q044 | Section: Educati

In [14]:
# Step 17K: Check the model's numerical input ranges

import pandas as pd
from pathlib import Path

project_dir = Path.cwd()
train_df = pd.read_csv(project_dir / "career_guidance_train.csv")

target = "recommended_career_domain"

numeric_features = [
    c for c in train_df.select_dtypes(include="number").columns
    if c != target
]

print("Minimum value across numerical features:",
      train_df[numeric_features].min().min())

print("Maximum value across numerical features:",
      train_df[numeric_features].max().max())

print("\nPer-feature minimum and maximum:")
display(pd.DataFrame({
    "Minimum": train_df[numeric_features].min(),
    "Maximum": train_df[numeric_features].max()
}))

Minimum value across numerical features: 1.0
Maximum value across numerical features: 10.0

Per-feature minimum and maximum:


,Minimum,Maximum
interest_technology,1.3,10.0
interest_data,1.0,10.0
interest_problem_solving,1.1,10.0
interest_creativity,1.0,10.0
interest_people,1.0,10.0
interest_helping,1.0,10.0
aptitude_logical,1.8,10.0
aptitude_numerical,2.1,10.0
aptitude_verbal,2.5,10.0
aptitude_spatial,2.7,10.0


In [15]:
# Step 17L: Create the feature-mapping specification

import pandas as pd

feature_mapping = pd.DataFrame([
    ("interest_technology", "Interest", "Q001, Q003, Q010"),
    ("interest_data", "Interest", "Q002"),
    ("interest_problem_solving", "Interest", "Q001, Q002, Q003, Q010"),
    ("interest_creativity", "Interest", "Q004"),
    ("interest_people", "Interest", "Q005, Q007"),
    ("interest_helping", "Interest", "Q006, Q007"),
    ("aptitude_logical", "Aptitude", "Q011-Q020: requires validated scoring"),
    ("aptitude_numerical", "Aptitude", "Q011, Q013, Q015, Q017, Q020"),
    ("aptitude_verbal", "Aptitude", "No direct validated measure yet"),
    ("aptitude_spatial", "Aptitude", "No direct validated measure yet"),
    ("aptitude_critical_thinking", "Aptitude", "Q018, Q019: requires validated scoring"),
    ("trait_openness", "Personality", "Q025, Q030: provisional"),
    ("trait_conscientiousness", "Personality", "Q024, Q027: provisional"),
    ("trait_extraversion", "Personality", "Q023, Q028: provisional"),
    ("trait_agreeableness", "Personality", "Q021: insufficient by itself"),
    ("trait_emotional_stability", "Personality", "Q026: insufficient by itself"),
    ("skill_programming", "Skills", "Q031"),
    ("skill_data_analysis", "Skills", "Q032"),
    ("skill_research", "Skills", "Q040"),
    ("skill_communication", "Skills", "Q021, Q028: proxy only"),
    ("skill_leadership", "Skills", "Q023, Q036: proxy only"),
], columns=["Model_Feature", "Feature_Group", "Candidate_Question_Mapping"])

feature_mapping["Mapping_Status"] = "Requires review"
feature_mapping["Scoring_Rule"] = "Not defined"
feature_mapping["Validation_Notes"] = ""

display(feature_mapping)

print("\nTotal model features:", len(feature_mapping))
print("Mappings are candidates, not validated scoring rules.")

,Model_Feature,Feature_Group,Candidate_Question_Mapping,Mapping_Status,Scoring_Rule,Validation_Notes
0,interest_technology,Interest,"Q001, Q003, Q010",Requires review,Not defined,
1,interest_data,Interest,Q002,Requires review,Not defined,
2,interest_problem_solving,Interest,"Q001, Q002, Q003, Q010",Requires review,Not defined,
3,interest_creativity,Interest,Q004,Requires review,Not defined,
4,interest_people,Interest,"Q005, Q007",Requires review,Not defined,
5,interest_helping,Interest,"Q006, Q007",Requires review,Not defined,
6,aptitude_logical,Aptitude,Q011-Q020: requires validated scoring,Requires review,Not defined,
7,aptitude_numerical,Aptitude,"Q011, Q013, Q015, Q017, Q020",Requires review,Not defined,
8,aptitude_verbal,Aptitude,No direct validated measure yet,Requires review,Not defined,
9,aptitude_spatial,Aptitude,No direct validated measure yet,Requires review,Not defined,



Total model features: 21
Mappings are candidates, not validated scoring rules.


In [16]:
# Step 17M: Extract aptitude questions for answer-key validation

import pandas as pd
from pathlib import Path

project_dir = Path.cwd()
excel_path = project_dir / "AI_Career_Guidance_Questions.xlsx"

questions_df = pd.read_excel(
    excel_path,
    sheet_name="Questions Dataset"
)

aptitude_questions = questions_df[
    questions_df["Question_ID"].isin(
        [f"Q{i:03d}" for i in range(11, 21)]
    )
]

for _, row in aptitude_questions.iterrows():
    print(f"\n{row['Question_ID']}: {row['Question_Text']}")
    print(f"Answer type: {row['Answer_Type']}")
    print(f"Options: {row['Answer_Options']}")
    print(f"Purpose: {row['Purpose']}")
    print("-" * 70)

print("\nTotal aptitude questions:", len(aptitude_questions))


Q011: A project requires 3 developers to complete in 12 days. If the deadline is moved up and it must be done in 4 days, how many developers of equal skill are needed in total?
Answer type: MCQ
Options: A) 6, B) 9, C) 12, D) 4
Purpose: Measures quantitative reasoning and inverse proportionality
----------------------------------------------------------------------

Q012: If all 'Servers' are 'Computers', and some 'Computers' are 'Infected', which of the following must be true?
Answer type: MCQ
Options: A) All Servers are Infected, B) Some Servers are Infected, C) Not all Computers are Servers, D) None of the above
Purpose: Measures logical deduction and syllogism
----------------------------------------------------------------------

Q013: A dataset has the values: 15, 18, 22, 27, 33. What is the next logical value in this sequence?
Answer type: MCQ
Options: A) 38, B) 39, C) 40, D) 42
Purpose: Measures numerical pattern recognition
-----------------------------------------------------

In [17]:
# Step 17O: Store aptitude answer keys and review flags

aptitude_answer_key = {
    "Q011": {
        "correct_option": "B",
        "status": "validated"
    },
    "Q012": {
        "correct_option": "D",
        "status": "validated"
    },
    "Q013": {
        "correct_option": "C",
        "status": "validated"
    },
    "Q014": {
        "correct_option": "A",
        "status": "validated"
    },
    "Q015": {
        "correct_option": "D",
        "status": "review_required"
    },
    "Q016": {
        "correct_option": "B",
        "status": "review_required"
    },
    "Q017": {
        "correct_option": "B",
        "status": "validated"
    },
    "Q018": {
        "correct_option": "A",
        "status": "validated"
    },
    "Q019": {
        "correct_option": "C",
        "status": "validated"
    },
    "Q020": {
        "correct_option": "C",
        "status": "validated"
    }
}

answer_key_df = pd.DataFrame.from_dict(
    aptitude_answer_key,
    orient="index"
).rename_axis("Question_ID").reset_index()

display(answer_key_df)

print("Total aptitude questions:", len(answer_key_df))
print(
    "Questions requiring review:",
    (answer_key_df["status"] == "review_required").sum()
)

,Question_ID,correct_option,status
0,Q011,B,validated
1,Q012,D,validated
2,Q013,C,validated
3,Q014,A,validated
4,Q015,D,review_required
5,Q016,B,review_required
6,Q017,B,validated
7,Q018,A,validated
8,Q019,C,validated
9,Q020,C,validated


Total aptitude questions: 10
Questions requiring review: 2


In [18]:
import pandas as pd
import json
from pathlib import Path

# File paths
project_dir = Path.cwd()
excel_path = project_dir / "AI_Career_Guidance_Questions.xlsx"

# Load the questionnaire
questions_df = pd.read_excel(
    excel_path,
    sheet_name="Questions Dataset"
)

# Revised Q015
q015_text = (
    "A company's annual revenue increases from ₹10 lakh to ₹12 lakh. "
    "Its profit margin decreases from 30% to 20%. Assuming profit equals "
    "revenue minus total costs, how do its total costs change?"
)

q015_options = {
    "A": "Decrease from ₹7 lakh to ₹6 lakh",
    "B": "Remain at ₹7 lakh",
    "C": "Increase from ₹7 lakh to ₹9.6 lakh",
    "D": "Increase from ₹7 lakh to ₹8 lakh"
}

# Revised Q016
q016_text = (
    "Consider the following algorithm. The outer loop runs n times, and "
    "during each outer iteration, the inner loop starts at n and halves "
    "its value until it becomes 1 or less. What is the overall time complexity?"
)

q016_options = {
    "A": "O(n)",
    "B": "O(n log n)",
    "C": "O(n²)",
    "D": "O(log n)"
}

# Update the existing rows without changing their IDs
for question_id, question_text, options in [
    ("Q015", q015_text, q015_options),
    ("Q016", q016_text, q016_options)
]:
    mask = questions_df["Question_ID"].astype(str).str.strip() == question_id

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for {question_id}; found {mask.sum()}."
        )

    questions_df.loc[mask, "Question_Text"] = question_text
    questions_df.loc[mask, "Answer_Options"] = json.dumps(
        options, ensure_ascii=False
    )

# Save back to the same Excel workbook, preserving other sheets
with pd.ExcelWriter(
    excel_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    questions_df.to_excel(
        writer,
        sheet_name="Questions Dataset",
        index=False
    )

print("Q015 and Q016 updated successfully.")
print(f"Saved to: {excel_path}")

Q015 and Q016 updated successfully.
Saved to: C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset\AI_Career_Guidance_Questions.xlsx


In [19]:
import pandas as pd
import json

# Reload the saved Excel file to verify the actual changes
questions_df = pd.read_excel(
    excel_path,
    sheet_name="Questions Dataset"
)

for question_id in ["Q015", "Q016"]:
    row = questions_df.loc[
        questions_df["Question_ID"].astype(str).str.strip() == question_id
    ].iloc[0]

    print(f"\n{question_id}")
    print("Question:", row["Question_Text"])
    print("Options:", json.loads(row["Answer_Options"]))

# Update the validated answer key
aptitude_answer_key["Q015"] = {
    "correct_option": "C",
    "status": "revised_validated"
}

aptitude_answer_key["Q016"] = {
    "correct_option": "B",
    "status": "revised_validated"
}

print("\nUpdated answer key:")
print(json.dumps(aptitude_answer_key, indent=4))


Q015
Question: A company's annual revenue increases from ₹10 lakh to ₹12 lakh. Its profit margin decreases from 30% to 20%. Assuming profit equals revenue minus total costs, how do its total costs change?
Options: {'A': 'Decrease from ₹7 lakh to ₹6 lakh', 'B': 'Remain at ₹7 lakh', 'C': 'Increase from ₹7 lakh to ₹9.6 lakh', 'D': 'Increase from ₹7 lakh to ₹8 lakh'}

Q016
Question: Consider the following algorithm. The outer loop runs n times, and during each outer iteration, the inner loop starts at n and halves its value until it becomes 1 or less. What is the overall time complexity?
Options: {'A': 'O(n)', 'B': 'O(n log n)', 'C': 'O(n²)', 'D': 'O(log n)'}

Updated answer key:
{
    "Q011": {
        "correct_option": "B",
        "status": "validated"
    },
    "Q012": {
        "correct_option": "D",
        "status": "validated"
    },
    "Q013": {
        "correct_option": "C",
        "status": "validated"
    },
    "Q014": {
        "correct_option": "A",
        "status": "va

In [20]:
# Validate the aptitude answer key

expected_questions = [f"Q{i:03d}" for i in range(11, 21)]

print("APTITUDE ANSWER KEY VALIDATION")
print("-" * 40)

for question_id in expected_questions:
    answer = aptitude_answer_key.get(question_id)

    if answer is None:
        print(f"{question_id}: MISSING ANSWER KEY")
    else:
        print(
            f"{question_id}: "
            f"Correct option = {answer['correct_option']}, "
            f"Status = {answer['status']}"
        )

missing_questions = [
    qid for qid in expected_questions
    if qid not in aptitude_answer_key
]

assert not missing_questions, (
    f"Missing answer keys: {missing_questions}"
)

assert aptitude_answer_key["Q015"]["correct_option"] == "C"
assert aptitude_answer_key["Q016"]["correct_option"] == "B"

print("\nAll 10 aptitude questions have answer keys.")
print("Q015 and Q016 corrected answers verified.")

APTITUDE ANSWER KEY VALIDATION
----------------------------------------
Q011: Correct option = B, Status = validated
Q012: Correct option = D, Status = validated
Q013: Correct option = C, Status = validated
Q014: Correct option = A, Status = validated
Q015: Correct option = C, Status = revised_validated
Q016: Correct option = B, Status = revised_validated
Q017: Correct option = B, Status = validated
Q018: Correct option = A, Status = validated
Q019: Correct option = C, Status = validated
Q020: Correct option = C, Status = validated

All 10 aptitude questions have answer keys.
Q015 and Q016 corrected answers verified.


In [21]:
def calculate_aptitude_scores(user_answers, answer_key):
    """
    Calculate provisional aptitude scores on a 1–10 scale.

    user_answers example:
    {
        "Q011": "B",
        "Q012": "D",
        ...
    }
    """

    # Questions assigned to each aptitude dimension
    aptitude_mapping = {
        "aptitude_numerical": [
            "Q011", "Q013", "Q015", "Q017", "Q020"
        ],
        "aptitude_logical": [
            "Q012", "Q014", "Q016", "Q019"
        ],
        "aptitude_critical_thinking": [
            "Q018", "Q019"
        ]
    }

    scores = {}
    details = {}

    for feature, question_ids in aptitude_mapping.items():
        answered = [
            qid for qid in question_ids
            if qid in user_answers and qid in answer_key
        ]

        if not answered:
            scores[feature] = None
            details[feature] = {
                "correct": 0,
                "answered": 0,
                "total": len(question_ids)
            }
            continue

        correct = sum(
            str(user_answers[qid]).strip().upper()
            == answer_key[qid]["correct_option"]
            for qid in answered
        )

        # Convert the correct-answer proportion to a 1–10 score
        proportion = correct / len(answered)
        scores[feature] = round(1 + 9 * proportion, 2)

        details[feature] = {
            "correct": correct,
            "answered": len(answered),
            "total": len(question_ids)
        }

    return scores, details


print("Aptitude scoring function created successfully.")

Aptitude scoring function created successfully.


In [22]:
# Sample answers for testing only — not real student responses
sample_answers = {
    "Q011": "B",
    "Q012": "D",
    "Q013": "C",
    "Q014": "A",
    "Q015": "C",
    "Q016": "B",
    "Q017": "B",
    "Q018": "A",
    "Q019": "C",
    "Q020": "C"
}

# Calculate aptitude scores
sample_scores, sample_details = calculate_aptitude_scores(
    sample_answers,
    aptitude_answer_key
)

print("SAMPLE APTITUDE SCORES")
print("-" * 40)

for feature, score in sample_scores.items():
    print(f"{feature}: {score}/10")

print("\nANSWER DETAILS")
print("-" * 40)

for feature, details in sample_details.items():
    print(f"{feature}: {details}")

SAMPLE APTITUDE SCORES
----------------------------------------
aptitude_numerical: 10.0/10
aptitude_logical: 10.0/10
aptitude_critical_thinking: 10.0/10

ANSWER DETAILS
----------------------------------------
aptitude_numerical: {'correct': 5, 'answered': 5, 'total': 5}
aptitude_logical: {'correct': 4, 'answered': 4, 'total': 4}
aptitude_critical_thinking: {'correct': 2, 'answered': 2, 'total': 2}


In [23]:
# Provisional mapping of interest questions to model features

interest_mapping = {
    "interest_technology": ["Q001", "Q003", "Q010"],
    "interest_data": ["Q002"],
    "interest_problem_solving": ["Q001", "Q002", "Q003", "Q010"],
    "interest_creativity": ["Q004"],
    "interest_people": ["Q005", "Q007"],
    "interest_helping": ["Q006", "Q007"]
}

# Display the mapping for review
print("INTEREST FEATURE MAPPING")
print("-" * 50)

for feature, question_ids in interest_mapping.items():
    print(f"{feature}: {', '.join(question_ids)}")

print("\nInterest mapping created.")
print("Note: These mappings are provisional and require validation.")

INTEREST FEATURE MAPPING
--------------------------------------------------
interest_technology: Q001, Q003, Q010
interest_data: Q002
interest_problem_solving: Q001, Q002, Q003, Q010
interest_creativity: Q004
interest_people: Q005, Q007
interest_helping: Q006, Q007

Interest mapping created.
Note: These mappings are provisional and require validation.


## Create the interest scoring function

In [24]:
def calculate_interest_scores(user_answers, mapping):
    """
    Convert interest ratings from 1–5 to the model's 1–10 scale.
    user_answers example: {"Q001": 4, "Q002": 5, ...}
    """

    scores = {}
    details = {}

    for feature, question_ids in mapping.items():
        valid_answers = []

        for qid in question_ids:
            if qid not in user_answers:
                continue

            try:
                rating = float(user_answers[qid])
            except (TypeError, ValueError):
                continue

            if 1 <= rating <= 5:
                valid_answers.append(rating)

        if not valid_answers:
            scores[feature] = None
            details[feature] = {
                "answered": 0,
                "required_mapping_questions": len(question_ids)
            }
            continue

        average_rating = sum(valid_answers) / len(valid_answers)
        scores[feature] = round((average_rating / 5) * 10, 2)

        details[feature] = {
            "average_rating_1_to_5": round(average_rating, 2),
            "answered": len(valid_answers),
            "required_mapping_questions": len(question_ids)
        }

    return scores, details


print("Interest scoring function created successfully.")

Interest scoring function created successfully.


In [25]:
# Sample interest ratings for testing only
sample_interest_answers = {
    "Q001": 4,
    "Q002": 5,
    "Q003": 4,
    "Q004": 5,
    "Q005": 3,
    "Q006": 4,
    "Q007": 4,
    "Q008": 3,
    "Q009": 2,
    "Q010": 5
}

# Calculate interest scores
sample_interest_scores, sample_interest_details = calculate_interest_scores(
    sample_interest_answers,
    interest_mapping
)

print("SAMPLE INTEREST SCORES")
print("-" * 45)

for feature, score in sample_interest_scores.items():
    print(f"{feature}: {score}/10")

print("\nSCORING DETAILS")
print("-" * 45)

for feature, details in sample_interest_details.items():
    print(f"{feature}: {details}")

SAMPLE INTEREST SCORES
---------------------------------------------
interest_technology: 8.67/10
interest_data: 10.0/10
interest_problem_solving: 9.0/10
interest_creativity: 10.0/10
interest_people: 7.0/10
interest_helping: 8.0/10

SCORING DETAILS
---------------------------------------------
interest_technology: {'average_rating_1_to_5': 4.33, 'answered': 3, 'required_mapping_questions': 3}
interest_data: {'average_rating_1_to_5': 5.0, 'answered': 1, 'required_mapping_questions': 1}
interest_problem_solving: {'average_rating_1_to_5': 4.5, 'answered': 4, 'required_mapping_questions': 4}
interest_creativity: {'average_rating_1_to_5': 5.0, 'answered': 1, 'required_mapping_questions': 1}
interest_people: {'average_rating_1_to_5': 3.5, 'answered': 2, 'required_mapping_questions': 2}
interest_helping: {'average_rating_1_to_5': 4.0, 'answered': 2, 'required_mapping_questions': 2}


In [26]:
# Inspect personality questions directly from the Excel workbook

personality_questions = questions_df[
    questions_df["Question_ID"].astype(str).str.strip().isin(
        [f"Q{i:03d}" for i in range(21, 31)]
    )
].copy()

display(
    personality_questions[
        [
            "Question_ID",
            "Question_Text",
            "Answer_Type",
            "Answer_Options",
            "Scoring_Method",
            "Feature_Mapping"
        ]
    ].sort_values("Question_ID")
)

,Question_ID,Question_Text,Answer_Type,Answer_Options,Scoring_Method,Feature_Mapping
20,Q021,I thrive in environments where I must collabor...,Agreement Scale (1-5),1 - Strongly Disagree to 5 - Strongly Agree,NaN,NaN
21,Q022,I prefer structured environments with clear gu...,Agreement Scale (1-5),1 - Strongly Disagree to 5 - Strongly Agree,NaN,NaN
22,Q023,I naturally take charge of situations and enjo...,Agreement Scale (1-5),1 - Strongly Disagree to 5 - Strongly Agree,NaN,NaN
23,Q024,I am highly detail-oriented and will spend ext...,Agreement Scale (1-5),1 - Strongly Disagree to 5 - Strongly Agree,NaN,NaN
24,Q025,I actively seek out opportunities to learn ent...,Agreement Scale (1-5),1 - Strongly Disagree to 5 - Strongly Agree,NaN,NaN
25,Q026,I perform well under tight deadlines and high-...,Agreement Scale (1-5),1 - Strongly Disagree to 5 - Strongly Agree,NaN,NaN
26,Q027,I prefer to follow established rules and proto...,Agreement Scale (1-5),1 - Strongly Disagree to 5 - Strongly Agree,NaN,NaN
27,Q028,"I enjoy engaging with clients, customers, or t...",Agreement Scale (1-5),1 - Strongly Disagree to 5 - Strongly Agree,NaN,NaN
28,Q029,I tend to rely on logical analysis and objecti...,Agreement Scale (1-5),1 - Strongly Disagree to 5 - Strongly Agree,NaN,NaN
29,Q030,I am comfortable abandoning a project I've wor...,Agreement Scale (1-5),1 - Strongly Disagree to 5 - Strongly Agree,NaN,NaN


In [27]:
# Provisional personality mapping based on question wording

personality_mapping = {
    "trait_openness": ["Q025", "Q030"],
    "trait_conscientiousness": ["Q024", "Q027"],
    "trait_extraversion": ["Q023", "Q028"],
    "trait_agreeableness": ["Q021"],
    "trait_emotional_stability": ["Q026"]
}

print("PERSONALITY FEATURE MAPPING")
print("-" * 45)

for feature, question_ids in personality_mapping.items():
    print(f"{feature}: {', '.join(question_ids)}")

print("\nMapping created for review.")
print("These mappings are provisional, not psychometrically validated.")

PERSONALITY FEATURE MAPPING
---------------------------------------------
trait_openness: Q025, Q030
trait_conscientiousness: Q024, Q027
trait_extraversion: Q023, Q028
trait_agreeableness: Q021
trait_emotional_stability: Q026

Mapping created for review.
These mappings are provisional, not psychometrically validated.


## Create the personality scoring function

In [28]:
def calculate_personality_scores(user_answers, mapping):
    """
    Convert agreement ratings from 1–5 to a provisional 1–10 score.
    """

    scores = {}
    details = {}

    for feature, question_ids in mapping.items():
        valid_answers = []

        for qid in question_ids:
            if qid not in user_answers:
                continue

            try:
                rating = float(user_answers[qid])
            except (TypeError, ValueError):
                continue

            if 1 <= rating <= 5:
                valid_answers.append(rating)

        if not valid_answers:
            scores[feature] = None
            details[feature] = {
                "answered": 0,
                "required_questions": len(question_ids)
            }
            continue

        average_rating = sum(valid_answers) / len(valid_answers)

        # Convert the 1–5 average to the model's 1–10 scale
        scores[feature] = round(average_rating * 2, 2)

        details[feature] = {
            "average_rating_1_to_5": round(average_rating, 2),
            "answered": len(valid_answers),
            "required_questions": len(question_ids)
        }

    return scores, details


print("Personality scoring function created successfully.")

Personality scoring function created successfully.


In [29]:
# Sample personality answers for testing only
sample_personality_answers = {
    "Q021": 4,
    "Q022": 3,
    "Q023": 5,
    "Q024": 4,
    "Q025": 5,
    "Q026": 3,
    "Q027": 4,
    "Q028": 4,
    "Q029": 5,
    "Q030": 4
}

# Calculate personality scores
sample_personality_scores, sample_personality_details = (
    calculate_personality_scores(
        sample_personality_answers,
        personality_mapping
    )
)

print("SAMPLE PERSONALITY SCORES")
print("-" * 45)

for feature, score in sample_personality_scores.items():
    print(f"{feature}: {score}/10")

print("\nSCORING DETAILS")
print("-" * 45)

for feature, details in sample_personality_details.items():
    print(f"{feature}: {details}")

SAMPLE PERSONALITY SCORES
---------------------------------------------
trait_openness: 9.0/10
trait_conscientiousness: 8.0/10
trait_extraversion: 9.0/10
trait_agreeableness: 8.0/10
trait_emotional_stability: 6.0/10

SCORING DETAILS
---------------------------------------------
trait_openness: {'average_rating_1_to_5': 4.5, 'answered': 2, 'required_questions': 2}
trait_conscientiousness: {'average_rating_1_to_5': 4.0, 'answered': 2, 'required_questions': 2}
trait_extraversion: {'average_rating_1_to_5': 4.5, 'answered': 2, 'required_questions': 2}
trait_agreeableness: {'average_rating_1_to_5': 4.0, 'answered': 1, 'required_questions': 1}
trait_emotional_stability: {'average_rating_1_to_5': 3.0, 'answered': 1, 'required_questions': 1}


In [30]:
# Inspect skill questions from the questionnaire

skill_questions = questions_df[
    questions_df["Question_ID"].astype(str).str.strip().isin(
        [f"Q{i:03d}" for i in range(31, 41)]
    )
].copy()

display(
    skill_questions[
        [
            "Question_ID",
            "Question_Text",
            "Answer_Type",
            "Answer_Options",
            "Scoring_Method",
            "Feature_Mapping"
        ]
    ].sort_values("Question_ID")
)

,Question_ID,Question_Text,Answer_Type,Answer_Options,Scoring_Method,Feature_Mapping
30,Q031,"How would you rate your ability to write, debu...",Skill Scale (1-5 + N/A),"1-Beginner to 5-Expert, N/A",NaN,NaN
31,Q032,How would you rate your ability to use tools l...,Skill Scale (1-5 + N/A),"1-Beginner to 5-Expert, N/A",NaN,NaN
32,Q033,How would you rate your proficiency with desig...,Skill Scale (1-5 + N/A),"1-Beginner to 5-Expert, N/A",NaN,NaN
33,Q034,How would you rate your ability to read financ...,Skill Scale (1-5 + N/A),"1-Beginner to 5-Expert, N/A",NaN,NaN
34,Q035,How would you rate your ability to configure f...,Skill Scale (1-5 + N/A),"1-Beginner to 5-Expert, N/A",NaN,NaN
35,Q036,"How would you rate your ability to plan, execu...",Skill Scale (1-5 + N/A),"1-Beginner to 5-Expert, N/A",NaN,NaN
36,Q037,How would you rate your ability to set up targ...,Skill Scale (1-5 + N/A),"1-Beginner to 5-Expert, N/A",NaN,NaN
37,Q038,How would you rate your ability to comprehend ...,Skill Scale (1-5 + N/A),"1-Beginner to 5-Expert, N/A",NaN,NaN
38,Q039,How would you rate your proficiency with core ...,Skill Scale (1-5 + N/A),"1-Beginner to 5-Expert, N/A",NaN,NaN
39,Q040,How would you rate your ability to conduct aca...,Skill Scale (1-5 + N/A),"1-Beginner to 5-Expert, N/A",NaN,NaN


In [31]:
# Provisional mapping of skill questions to model features

skill_mapping = {
    "skill_programming": ["Q031"],
    "skill_data_analysis": ["Q032"],
    "skill_research": ["Q040"],
    "skill_leadership": ["Q036"]
}

print("SKILL FEATURE MAPPING")
print("-" * 45)

for feature, question_ids in skill_mapping.items():
    print(f"{feature}: {', '.join(question_ids)}")

print("\nMapping created for review.")
print("Unmapped questions are intentionally excluded for now.")

SKILL FEATURE MAPPING
---------------------------------------------
skill_programming: Q031
skill_data_analysis: Q032
skill_research: Q040
skill_leadership: Q036

Mapping created for review.
Unmapped questions are intentionally excluded for now.


## Create the skill scoring function

In [32]:
def calculate_skill_scores(user_answers, mapping):
    """
    Convert skill ratings from 1–5 to the model's 1–10 scale.
    N/A and invalid answers are excluded.
    """

    scores = {}
    details = {}

    for feature, question_ids in mapping.items():
        valid_answers = []

        for qid in question_ids:
            if qid not in user_answers:
                continue

            answer = user_answers[qid]

            if isinstance(answer, str) and answer.strip().upper() in {
                "N/A", "NA", "NOT APPLICABLE"
            }:
                continue

            try:
                rating = float(answer)
            except (TypeError, ValueError):
                continue

            if 1 <= rating <= 5:
                valid_answers.append(rating)

        if not valid_answers:
            scores[feature] = None
            details[feature] = {
                "answered": 0,
                "required_questions": len(question_ids),
                "status": "No valid rating"
            }
            continue

        average_rating = sum(valid_answers) / len(valid_answers)

        scores[feature] = round(average_rating * 2, 2)

        details[feature] = {
            "average_rating_1_to_5": round(average_rating, 2),
            "answered": len(valid_answers),
            "required_questions": len(question_ids),
            "status": "Scored"
        }

    return scores, details


print("Skill scoring function created successfully.")

Skill scoring function created successfully.


In [33]:
# Sample skill answers for testing only
sample_skill_answers = {
    "Q031": 4,       # Programming
    "Q032": 5,       # Data analysis
    "Q036": 3,       # Project management / leadership proxy
    "Q040": "N/A"    # Research
}

# Calculate skill scores
sample_skill_scores, sample_skill_details = calculate_skill_scores(
    sample_skill_answers,
    skill_mapping
)

print("SAMPLE SKILL SCORES")
print("-" * 45)

for feature, score in sample_skill_scores.items():
    print(f"{feature}: {score}/10")

print("\nSCORING DETAILS")
print("-" * 45)

for feature, details in sample_skill_details.items():
    print(f"{feature}: {details}")
    

SAMPLE SKILL SCORES
---------------------------------------------
skill_programming: 8.0/10
skill_data_analysis: 10.0/10
skill_research: None/10
skill_leadership: 6.0/10

SCORING DETAILS
---------------------------------------------
skill_programming: {'average_rating_1_to_5': 4.0, 'answered': 1, 'required_questions': 1, 'status': 'Scored'}
skill_data_analysis: {'average_rating_1_to_5': 5.0, 'answered': 1, 'required_questions': 1, 'status': 'Scored'}
skill_research: {'answered': 0, 'required_questions': 1, 'status': 'No valid rating'}
skill_leadership: {'average_rating_1_to_5': 3.0, 'answered': 1, 'required_questions': 1, 'status': 'Scored'}


In [34]:
# Combine the scores from our sample assessments

combined_sample_profile = {}

combined_sample_profile.update(sample_interest_scores)
combined_sample_profile.update(sample_personality_scores)
combined_sample_profile.update(sample_skill_scores)

# Add aptitude scores from the earlier test
combined_sample_profile.update(sample_scores)

profile_df = pd.DataFrame(
    combined_sample_profile.items(),
    columns=["Feature", "Score"]
)

display(profile_df)

print("\nTotal features currently scored:", profile_df["Score"].notna().sum())
print("Total mapped features:", len(profile_df))
print("Features with missing scores:", profile_df["Score"].isna().sum())

,Feature,Score
0,interest_technology,8.67
1,interest_data,10.00
2,interest_problem_solving,9.00
3,interest_creativity,10.00
4,interest_people,7.00
5,interest_helping,8.00
6,trait_openness,9.00
7,trait_conscientiousness,8.00
8,trait_extraversion,9.00
9,trait_agreeableness,8.00



Total features currently scored: 17
Total mapped features: 18
Features with missing scores: 1


In [35]:
# Get the exact feature list expected by the trained model

model_features = list(X_train.columns)

# Features available in our combined sample profile
available_features = set(combined_sample_profile.keys())

missing_features = [
    feature for feature in model_features
    if feature not in available_features
]

print("Expected model features:", len(model_features))
print("Currently mapped features:", len(available_features))
print("Missing model features:", len(missing_features))

print("\nMISSING FEATURES")
print("-" * 45)

for feature in missing_features:
    print(feature)

print("\nFEATURES WITH MISSING SCORES")

for feature, score in combined_sample_profile.items():
    if score is None or pd.isna(score):
        print(feature)

NameError: name 'X_train' is not defined

In [36]:
from pathlib import Path
import pandas as pd

# Locate the training dataset
data_path = Path.cwd() / "career_guidance_train.csv"

if not data_path.exists():
    raise FileNotFoundError(
        f"Training dataset not found at: {data_path}\n"
        "Check that your notebook is in the dataset directory."
    )

# Load the training dataset
train_df = pd.read_csv(data_path)

# The target column is the career label
target_column = "career_domain"

# Identify the 25 input features
model_features = [
    col for col in train_df.columns
    if col != target_column
]

# Compare with our current questionnaire mapping
available_features = set(combined_sample_profile.keys())

missing_features = [
    feature for feature in model_features
    if feature not in available_features
]

print("Training dataset shape:", train_df.shape)
print("Expected model input features:", len(model_features))
print("Currently mapped features:", len(available_features))
print("Features not yet mapped:", len(missing_features))

print("\nMISSING MODEL FEATURES")
print("-" * 40)

for feature in missing_features:
    print(feature)

print("\nFEATURES WITH MISSING SCORES")
print("-" * 40)

for feature, score in combined_sample_profile.items():
    if score is None or pd.isna(score):
        print(feature)

Training dataset shape: (1920, 26)
Expected model input features: 26
Currently mapped features: 18
Features not yet mapped: 8

MISSING MODEL FEATURES
----------------------------------------
age_group
education_level
experience_stage
region_type
aptitude_verbal
aptitude_spatial
skill_communication
recommended_career_domain

FEATURES WITH MISSING SCORES
----------------------------------------
skill_research


In [37]:
# Inspect all training dataset columns

print("ALL TRAINING DATASET COLUMNS")
print("-" * 50)

for index, column in enumerate(train_df.columns, start=1):
    print(f"{index}. {column}")

print("\nPossible target columns:")
for column in train_df.columns:
    if any(word in column.lower() for word in ["career_domain", "target", "label"]):
        print("-", column)

ALL TRAINING DATASET COLUMNS
--------------------------------------------------
1. age_group
2. education_level
3. experience_stage
4. region_type
5. interest_technology
6. interest_data
7. interest_problem_solving
8. interest_creativity
9. interest_people
10. interest_helping
11. aptitude_logical
12. aptitude_numerical
13. aptitude_verbal
14. aptitude_spatial
15. aptitude_critical_thinking
16. trait_openness
17. trait_conscientiousness
18. trait_extraversion
19. trait_agreeableness
20. trait_emotional_stability
21. skill_programming
22. skill_data_analysis
23. skill_research
24. skill_communication
25. skill_leadership
26. recommended_career_domain

Possible target columns:
- recommended_career_domain


In [38]:
# Define the actual target column
target_column = "recommended_career_domain"

# Exact 25 input features from the training dataset
model_features = [
    col for col in train_df.columns
    if col != target_column
]

# Current combined profile features
available_features = set(combined_sample_profile.keys())

# Features not yet mapped
missing_features = [
    feature for feature in model_features
    if feature not in available_features
]

print("Total dataset columns:", len(train_df.columns))
print("Target column:", target_column)
print("Expected input features:", len(model_features))
print("Currently mapped features:", len(available_features))
print("Unmapped model features:", len(missing_features))

print("\nUNMAPPED FEATURES")
print("-" * 40)

for feature in missing_features:
    print(feature)

print("\nCATEGORICAL INPUTS TO COLLECT")
print("-" * 40)

for feature in [
    "age_group",
    "education_level",
    "experience_stage",
    "region_type"
]:
    print(feature)

Total dataset columns: 26
Target column: recommended_career_domain
Expected input features: 25
Currently mapped features: 18
Unmapped model features: 7

UNMAPPED FEATURES
----------------------------------------
age_group
education_level
experience_stage
region_type
aptitude_verbal
aptitude_spatial
skill_communication

CATEGORICAL INPUTS TO COLLECT
----------------------------------------
age_group
education_level
experience_stage
region_type


In [39]:
# Inspect valid categories used during model training

categorical_features = [
    "age_group",
    "education_level",
    "experience_stage",
    "region_type"
]

for feature in categorical_features:
    print(f"\n{feature}")
    print("-" * 45)

    print(
        train_df[feature]
        .value_counts(dropna=False)
        .to_string()
    )

    print("\nUnique values:")
    print(sorted(train_df[feature].dropna().astype(str).unique()))


age_group
---------------------------------------------
age_group
31+      494
23-30    494
15-18    487
19-22    445

Unique values:
['15-18', '19-22', '23-30', '31+']

education_level
---------------------------------------------
education_level
B.Tech IT                  338
MBA                        123
BBA                        100
B.Com                       97
BCA                         95
MCA                         86
B.Tech Computer Science     82
BA Economics                73
Diploma Design              70
BA                          63
B.Des                       63
Diploma IT                  59
BFA                         56
B.Sc Mathematics            52
B.Sc Statistics             47
B.Pharm                     40
B.Sc Nursing                38
MA                          37
M.Sc                        35
B.Sc Biology                34
LLB                         33
Diploma Engineering         32
BA Media                    32
B.Tech Electrical           32
B.Tech 

In [40]:
# Get valid categories directly from the training dataset

valid_categories = {
    feature: sorted(train_df[feature].dropna().astype(str).unique())
    for feature in [
        "age_group",
        "education_level",
        "experience_stage",
        "region_type"
    ]
}


def validate_categorical_inputs(student_profile):
    """
    Validate categorical values against the training dataset.
    """

    errors = []

    for feature, allowed_values in valid_categories.items():
        value = student_profile.get(feature)

        if value is None or pd.isna(value) or str(value).strip() == "":
            errors.append(f"{feature}: value is required")

        elif str(value).strip() not in allowed_values:
            errors.append(
                f"{feature}: invalid value '{value}'. "
                f"Allowed values: {allowed_values}"
            )

    return errors


print("Categorical validation function created successfully.")

for feature, categories in valid_categories.items():
    print(f"{feature}: {len(categories)} valid categories")

Categorical validation function created successfully.
age_group: 4 valid categories
education_level: 31 valid categories
experience_stage: 4 valid categories
region_type: 3 valid categories


In [41]:
# Test 1: Valid categorical inputs

sample_categorical_profile = {
    "age_group": "19-22",
    "education_level": "B.Tech IT",
    "experience_stage": "Student",
    "region_type": "Semi-urban"
}

errors = validate_categorical_inputs(sample_categorical_profile)

print("Test 1: Valid Profile")
print("Validation errors:", errors)
print("Result:", "PASS" if not errors else "FAIL")


# Test 2: Invalid education level

invalid_profile = sample_categorical_profile.copy()
invalid_profile["education_level"] = "B.Tech Artificial Intelligence"

errors = validate_categorical_inputs(invalid_profile)

print("\nTest 2: Invalid Profile")
print("Validation errors:", errors)
print("Result:", "PASS" if errors else "FAIL")

Test 1: Valid Profile
Validation errors: []
Result: PASS

Test 2: Invalid Profile
Validation errors: ["education_level: invalid value 'B.Tech Artificial Intelligence'. Allowed values: ['B.Com', 'B.Des', 'B.Ed', 'B.Pharm', 'B.Sc Biology', 'B.Sc Mathematics', 'B.Sc Nursing', 'B.Sc Statistics', 'B.Tech Biotechnology', 'B.Tech Civil', 'B.Tech Computer Science', 'B.Tech Electrical', 'B.Tech IT', 'B.Tech Mechanical', 'BA', 'BA Economics', 'BA Media', 'BA Political Science', 'BBA', 'BCA', 'BFA', 'Diploma Design', 'Diploma Engineering', 'Diploma IT', 'LLB', 'M.Sc', 'MA', 'MBA', 'MBBS', 'MCA', 'PhD']"]
Result: PASS


In [42]:
# Inspect the three unmapped numerical features

unmapped_features = [
    "aptitude_verbal",
    "aptitude_spatial",
    "skill_communication"
]

for feature in unmapped_features:
    print(f"\n{feature}")
    print("-" * 40)
    print("Minimum:", train_df[feature].min())
    print("Maximum:", train_df[feature].max())
    print("Mean:", round(train_df[feature].mean(), 2))
    print("Median:", train_df[feature].median())
    print("Missing values:", train_df[feature].isna().sum())


aptitude_verbal
----------------------------------------
Minimum: 2.5
Maximum: 10.0
Mean: 7.27
Median: 7.25
Missing values: 0

aptitude_spatial
----------------------------------------
Minimum: 2.7
Maximum: 10.0
Mean: 7.56
Median: 7.6
Missing values: 0

skill_communication
----------------------------------------
Minimum: 1.3
Maximum: 10.0
Mean: 7.55
Median: 7.7
Missing values: 0


In [43]:
# Search the questionnaire for relevant questions

questionnaire_df = pd.read_excel(
    "AI_Career_Guidance_Questions.xlsx",
    sheet_name="Questions Dataset"
)

keywords = [
    "verbal",
    "language",
    "reading",
    "writing",
    "spatial",
    "visual",
    "rotation",
    "shape",
    "communication",
    "speaking",
    "listening",
    "express"
]

for keyword in keywords:
    matches = questionnaire_df[
        questionnaire_df["Question_Text"]
        .astype(str)
        .str.contains(keyword, case=False, na=False)
    ]

    if not matches.empty:
        print(f"\nKeyword: {keyword}")

        for _, row in matches.iterrows():
            print(f"{row['Question_ID']}: {row['Question_Text']}")


Keyword: language
Q014: In a given code language, 'DATA' is written as 'FCVC'. How would 'CODE' be written in that same language?
Q031: How would you rate your ability to write, debug, and optimize code in at least one programming language?

Keyword: reading
Q120: How comfortable are you reading and synthesizing hundreds of pages of dense legal precedent daily?

Keyword: writing
Q001: How interested are you in writing scripts or code to automate complex, repetitive tasks?
Q123: How would you rate your ability to explain complex ideas clearly in speaking and writing?

Keyword: visual
Q004: How interested are you in sketching out wireframes and designing the visual layout of a mobile application?
Q076: Do you prefer conducting user research (interviews, surveys) or creating high-fidelity visual designs?

Keyword: rotation
Q062: Are you willing to participate in on-call rotations for emergency system outages?

Keyword: speaking
Q123: How would you rate your ability to explain complex ide

In [44]:
# Display aptitude and skill questions for mapping review

review_questions = questionnaire_df[
    questionnaire_df["Question_ID"].between("Q011", "Q040")
][
    ["Question_ID", "Section", "Question_Text", "Answer_Type", "Answer_Options"]
]

for _, row in review_questions.iterrows():
    print(f"\n{row['Question_ID']} | {row['Section']}")
    print(row["Question_Text"])
    print("Answer type:", row["Answer_Type"])
    print("Options:", row["Answer_Options"])


Q011 | Aptitude
A project requires 3 developers to complete in 12 days. If the deadline is moved up and it must be done in 4 days, how many developers of equal skill are needed in total?
Answer type: MCQ
Options: A) 6, B) 9, C) 12, D) 4

Q012 | Aptitude
If all 'Servers' are 'Computers', and some 'Computers' are 'Infected', which of the following must be true?
Answer type: MCQ
Options: A) All Servers are Infected, B) Some Servers are Infected, C) Not all Computers are Servers, D) None of the above

Q013 | Aptitude
A dataset has the values: 15, 18, 22, 27, 33. What is the next logical value in this sequence?
Answer type: MCQ
Options: A) 38, B) 39, C) 40, D) 42

Q014 | Aptitude
In a given code language, 'DATA' is written as 'FCVC'. How would 'CODE' be written in that same language?
Answer type: MCQ
Options: A) EQFG, B) ERFG, C) EQGG, D) DPEF

Q015 | Aptitude
A company's annual revenue increases from ₹10 lakh to ₹12 lakh. Its profit margin decreases from 30% to 20%. Assuming profit equals

In [45]:
# Inspect the last questions and existing question ID structure

print("Total questions:", len(questionnaire_df))
print("Last 10 question IDs:")
print(questionnaire_df["Question_ID"].tail(10).tolist())

print("\nQuestion ID data type:")
print(questionnaire_df["Question_ID"].dtype)

print("\nDuplicate question IDs:")
print(questionnaire_df.loc[
    questionnaire_df["Question_ID"].duplicated(),
    "Question_ID"
].tolist())


Total questions: 123
Last 10 question IDs:
['Q114', 'Q115', 'Q116', 'Q117', 'Q118', 'Q119', 'Q120', 'Q121', 'Q122', 'Q123']

Question ID data type:
str

Duplicate question IDs:
[]


In [46]:
# Prepare three proposed questions for missing model features

new_questions = pd.DataFrame([
    {
        "Question_ID": "Q121",
        "Section": "Aptitude",
        "Question_Text": (
            "Read a short passage and identify its main conclusion."
        ),
        "Answer_Type": "MCQ",
        "Answer_Options": (
            '{"A": "Option 1", "B": "Option 2", '
            '"C": "Option 3", "D": "Option 4"}'
        ),
        "Scoring_Method": (
            "Correct answer = 1; incorrect answer = 0. "
            "Convert proportion correct to a 1-10 score."
        ),
        "Feature_Mapping": "aptitude_verbal",
        "Purpose": (
            "Preliminary assessment of verbal comprehension "
            "and reasoning."
        )
    },
    {
        "Question_ID": "Q122",
        "Section": "Aptitude",
        "Question_Text": (
            "Identify the shape that results when a 2D shape "
            "is rotated or folded."
        ),
        "Answer_Type": "MCQ",
        "Answer_Options": (
            '{"A": "Option 1", "B": "Option 2", '
            '"C": "Option 3", "D": "Option 4"}'
        ),
        "Scoring_Method": (
            "Correct answer = 1; incorrect answer = 0. "
            "Convert proportion correct to a 1-10 score."
        ),
        "Feature_Mapping": "aptitude_spatial",
        "Purpose": (
            "Preliminary assessment of spatial reasoning."
        )
    },
    {
        "Question_ID": "Q123",
        "Section": "Skills",
        "Question_Text": (
            "How would you rate your ability to explain "
            "complex ideas clearly in speaking and writing?"
        ),
        "Answer_Type": "Skill Scale (1-5 + N/A)",
        "Answer_Options": (
            "1-Beginner to 5-Expert, N/A"
        ),
        "Scoring_Method": (
            "Convert a valid 1-5 rating to the model's 1-10 "
            "scale; N/A remains missing."
        ),
        "Feature_Mapping": "skill_communication",
        "Purpose": (
            "Collect a self-reported communication skill rating."
        )
    }
])

print("Proposed questions:", len(new_questions))
display(new_questions)

Proposed questions: 3


,Question_ID,Section,Question_Text,Answer_Type,Answer_Options,Scoring_Method,Feature_Mapping,Purpose
0,Q121,Aptitude,Read a short passage and identify its main con...,MCQ,"{""A"": ""Option 1"", ""B"": ""Option 2"", ""C"": ""Optio...",Correct answer = 1; incorrect answer = 0. Conv...,aptitude_verbal,Preliminary assessment of verbal comprehension...
1,Q122,Aptitude,Identify the shape that results when a 2D shap...,MCQ,"{""A"": ""Option 1"", ""B"": ""Option 2"", ""C"": ""Optio...",Correct answer = 1; incorrect answer = 0. Conv...,aptitude_spatial,Preliminary assessment of spatial reasoning.
2,Q123,Skills,How would you rate your ability to explain com...,Skill Scale (1-5 + N/A),"1-Beginner to 5-Expert, N/A",Convert a valid 1-5 rating to the model's 1-10...,skill_communication,Collect a self-reported communication skill ra...


In [47]:
# Define complete preliminary aptitude questions

aptitude_additions = {
    "Q121": {
        "question": (
            "Read the passage: "
            "'Regular exercise can improve cardiovascular health. "
            "It may also improve sleep quality and reduce stress. "
            "Therefore, regular exercise supports overall well-being.' "
            "What is the main conclusion?"
        ),
        "options": {
            "A": "Exercise only improves sleep quality.",
            "B": "Exercise supports overall well-being.",
            "C": "Stress is the main cause of poor health.",
            "D": "Exercise is necessary only for athletes."
        },
        "correct_option": "B",
        "feature": "aptitude_verbal"
    },

    "Q122": {
        "question": (
            "A square has vertices labeled clockwise A, B, C, and D, "
            "starting at the top-left corner. The square is rotated "
            "90 degrees clockwise. Which vertex moves to the top-right "
            "corner?"
        ),
        "options": {
            "A": "A",
            "B": "B",
            "C": "C",
            "D": "D"
        },
        "correct_option": "D",
        "feature": "aptitude_spatial"
    }
}

for question_id, question in aptitude_additions.items():
    print(f"\n{question_id}: {question['question']}")

    for option, text in question["options"].items():
        print(f"{option}) {text}")

    print("Correct answer:", question["correct_option"])
    print("Feature:", question["feature"])


Q121: Read the passage: 'Regular exercise can improve cardiovascular health. It may also improve sleep quality and reduce stress. Therefore, regular exercise supports overall well-being.' What is the main conclusion?
A) Exercise only improves sleep quality.
B) Exercise supports overall well-being.
C) Stress is the main cause of poor health.
D) Exercise is necessary only for athletes.
Correct answer: B
Feature: aptitude_verbal

Q122: A square has vertices labeled clockwise A, B, C, and D, starting at the top-left corner. The square is rotated 90 degrees clockwise. Which vertex moves to the top-right corner?
A) A
B) B
C) C
D) D
Correct answer: D
Feature: aptitude_spatial


In [48]:
# Finalize the preliminary aptitude answer keys

new_aptitude_answer_key = {
    "Q121": {
        "correct_option": "B",
        "feature": "aptitude_verbal",
        "status": "preliminary_validated"
    },
    "Q122": {
        "correct_option": "A",
        "feature": "aptitude_spatial",
        "status": "preliminary_validated"
    }
}

for question_id, answer in new_aptitude_answer_key.items():
    print(
        question_id,
        "| Correct option:", answer["correct_option"],
        "| Feature:", answer["feature"],
        "| Status:", answer["status"]
    )

Q121 | Correct option: B | Feature: aptitude_verbal | Status: preliminary_validated
Q122 | Correct option: A | Feature: aptitude_spatial | Status: preliminary_validated


In [49]:
# Define and test communication-skill scoring

def calculate_communication_score(answer):
    """
    Convert a communication skill rating from 1-5 to 1-10.
    Return None when the response is missing or N/A.
    """

    if answer is None:
        return None

    if isinstance(answer, str) and answer.strip().upper() == "N/A":
        return None

    try:
        rating = float(answer)
    except (TypeError, ValueError):
        return None

    if not rating.is_integer() or not 1 <= rating <= 5:
        return None

    return rating * 2


# Test valid and invalid responses
test_answers = [1, 3, 5, "N/A", None, 6]

for answer in test_answers:
    print(
        f"Answer: {answer!s:>4} -> "
        f"Score: {calculate_communication_score(answer)}"
    )

Answer:    1 -> Score: 2.0
Answer:    3 -> Score: 6.0
Answer:    5 -> Score: 10.0
Answer:  N/A -> Score: None
Answer: None -> Score: None
Answer:    6 -> Score: None


In [50]:
from pathlib import Path
import pandas as pd
import json

excel_path = Path.cwd() / "AI_Career_Guidance_Questions.xlsx"

# Complete the two MCQ records with their actual questions and options
for question_id, question in aptitude_additions.items():
    mask = new_questions["Question_ID"] == question_id

    new_questions.loc[mask, "Question_Text"] = question["question"]
    new_questions.loc[mask, "Answer_Options"] = json.dumps(
        question["options"],
        ensure_ascii=False
    )

# Add the verified answer keys to the scoring method
new_questions.loc[
    new_questions["Question_ID"] == "Q121", "Scoring_Method"
] = "Correct option: B; correct=1, incorrect=0; provisional single-item assessment."

new_questions.loc[
    new_questions["Question_ID"] == "Q122", "Scoring_Method"
] = "Correct option: A; correct=1, incorrect=0; provisional single-item assessment."

# Read the existing worksheet
existing_questions = pd.read_excel(
    excel_path,
    sheet_name="Questions Dataset"
)

# Prevent duplicate IDs before appending
existing_ids = set(existing_questions["Question_ID"].astype(str))
new_ids = set(new_questions["Question_ID"].astype(str))

if existing_ids.intersection(new_ids):
    raise ValueError("Duplicate question IDs found. Nothing was saved.")

# Append the new questions
updated_questions = pd.concat(
    [existing_questions, new_questions],
    ignore_index=True
)

# Preserve the existing workbook and replace only the target worksheet
with pd.ExcelWriter(
    excel_path,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    updated_questions.to_excel(
        writer,
        sheet_name="Questions Dataset",
        index=False
    )

print("Questionnaire updated successfully.")
print("Total questions:", len(updated_questions))
print("New question IDs:", new_questions["Question_ID"].tolist())
print("Saved to:", excel_path)

ValueError: Duplicate question IDs found. Nothing was saved.

In [51]:
# Reload the saved questionnaire and verify the changes

verified_questions = pd.read_excel(
    "AI_Career_Guidance_Questions.xlsx",
    sheet_name="Questions Dataset"
)

print("Total questions:", len(verified_questions))
print(
    "Unique question IDs:",
    verified_questions["Question_ID"].nunique()
)

print("\nNew questions:")
display(
    verified_questions[
        verified_questions["Question_ID"].isin(["Q121", "Q122", "Q123"])
    ][
        [
            "Question_ID",
            "Section",
            "Question_Text",
            "Scoring_Method",
            "Feature_Mapping"
        ]
    ]
)

assert len(verified_questions) == 123
assert verified_questions["Question_ID"].nunique() == 123

print("\nVerification: PASS")

Total questions: 123
Unique question IDs: 123

New questions:


,Question_ID,Section,Question_Text,Scoring_Method,Feature_Mapping
120,Q121,Aptitude,Read the passage: 'Regular exercise can improv...,"Correct option: B; correct=1, incorrect=0; pro...",aptitude_verbal
121,Q122,Aptitude,"A square has vertices labeled clockwise A, B, ...","Correct option: A; correct=1, incorrect=0; pro...",aptitude_spatial
122,Q123,Skills,How would you rate your ability to explain com...,Convert a valid 1-5 rating to the model's 1-10...,skill_communication



Verification: PASS


In [52]:
# Compare model input features with the current questionnaire mappings

target_column = "recommended_career_domain"

model_features = [
    column for column in train_df.columns
    if column != target_column
]

mapped_features = set()

for mapping in [
    interest_mapping,
    personality_mapping,
    skill_mapping
]:
    mapped_features.update(mapping.keys())

mapped_features.update([
    "aptitude_logical",
    "aptitude_numerical",
    "aptitude_critical_thinking",
    "aptitude_verbal",
    "aptitude_spatial",
    "skill_communication"
])

print("Total model input features:", len(model_features))
print("Features currently mapped:", len(mapped_features))

print("\nFeatures still requiring mapping or input:")
for feature in model_features:
    if feature not in mapped_features:
        print("-", feature)

print("\nMapped features:")
for feature in model_features:
    if feature in mapped_features:
        print("-", feature)

Total model input features: 25
Features currently mapped: 21

Features still requiring mapping or input:
- age_group
- education_level
- experience_stage
- region_type

Mapped features:
- interest_technology
- interest_data
- interest_problem_solving
- interest_creativity
- interest_people
- interest_helping
- aptitude_logical
- aptitude_numerical
- aptitude_verbal
- aptitude_spatial
- aptitude_critical_thinking
- trait_openness
- trait_conscientiousness
- trait_extraversion
- trait_agreeableness
- trait_emotional_stability
- skill_programming
- skill_data_analysis
- skill_research
- skill_communication
- skill_leadership


In [56]:
# Collect and validate the student's categorical information

def collect_student_details():
    print("=== Student Background Information ===")

    student_details = {
        "age_group": input(
            "Age group (15-18 / 19-22 / 23-30 / 31+): "
        ).strip(),

        "education_level": input(
            "Education level (enter exact category): "
        ).strip(),

        "experience_stage": input(
            "Experience stage (Student / Entry-level / "
            "1-3 years / 4+ years): "
        ).strip(),

        "region_type": input(
            "Region type (Rural / Semi-urban / Urban): "
        ).strip()
    }

    errors = validate_categorical_inputs(student_details)

    if errors:
        print("\nPlease correct these entries:")
        for error in errors:
            print("-", error)
        return None

    print("\nStudent details validated successfully.")
    return student_details

In [ ]:
student_details = collect_student_details()
print("\nCollected profile:", student_details)

In [53]:
# Enter the student's details directly

student_details = {
    "age_group": "19-22",
    "education_level": "B.Tech IT",
    "experience_stage": "Student",
    "region_type": "Semi-urban"
}

errors = validate_categorical_inputs(student_details)

if errors:
    print("Validation errors:")
    for error in errors:
        print("-", error)
else:
    print("Student details validated successfully.")
    print(student_details)

Student details validated successfully.
{'age_group': '19-22', 'education_level': 'B.Tech IT', 'experience_stage': 'Student', 'region_type': 'Semi-urban'}


In [54]:
import pandas as pd

# Recreate the validation function
valid_categories = {
    feature: sorted(train_df[feature].dropna().astype(str).unique())
    for feature in [
        "age_group",
        "education_level",
        "experience_stage",
        "region_type"
    ]
}

def validate_categorical_inputs(student_profile):
    errors = []

    for feature, allowed_values in valid_categories.items():
        value = student_profile.get(feature)

        if value is None or pd.isna(value) or str(value).strip() == "":
            errors.append(f"{feature}: value is required")

        elif str(value).strip() not in allowed_values:
            errors.append(f"{feature}: invalid value '{value}'")

    return errors


# Sample student profile
student_details = {
    "age_group": "19-22",
    "education_level": "B.Tech IT",
    "experience_stage": "Student",
    "region_type": "Semi-urban"
}

errors = validate_categorical_inputs(student_details)

if errors:
    print("Validation errors:")
    for error in errors:
        print("-", error)
else:
    print("Student details validated successfully.")
    print(student_details)

Student details validated successfully.
{'age_group': '19-22', 'education_level': 'B.Tech IT', 'experience_stage': 'Student', 'region_type': 'Semi-urban'}


In [55]:
from pathlib import Path

project_dir = Path.cwd()

required_files = [
    "career_guidance_train.csv",
    "career_guidance_test.csv",
    "artifacts/career_guidance_model.joblib",
    "AI_Career_Guidance_Questions.xlsx"
]

print("Project folder:", project_dir)

for file_name in required_files:
    file_path = project_dir / file_name
    print(f"{file_name}: {'FOUND' if file_path.exists() else 'NOT FOUND'}")

Project folder: C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset
career_guidance_train.csv: FOUND
career_guidance_test.csv: FOUND
artifacts/career_guidance_model.joblib: FOUND
AI_Career_Guidance_Questions.xlsx: FOUND


## Reload the trained model and training data
### Because the kernel was restarted, let's restore the essential variables before proceeding.

In [56]:
import pandas as pd
import joblib
from pathlib import Path

# Project paths
project_dir = Path.cwd()

train_path = project_dir / "career_guidance_train.csv"
test_path = project_dir / "career_guidance_test.csv"
model_path = project_dir / "artifacts" / "career_guidance_model.joblib"

# Load datasets
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Load the already-trained model
model = joblib.load(model_path)

# Define target and model input features
target_column = "recommended_career_domain"

model_features = [
    column for column in train_df.columns
    if column != target_column
]

career_labels = list(model.classes_)

print("Training dataset:", train_df.shape)
print("Testing dataset:", test_df.shape)
print("Model type:", type(model).__name__)
print("Number of input features:", len(model_features))
print("Number of career domains:", len(career_labels))
print("\nCareer domains:")
for career in career_labels:
    print("-", career)

Training dataset: (1920, 26)
Testing dataset: (480, 26)
Model type: VotingClassifier
Number of input features: 25
Number of career domains: 12

Career domains:
- Business & Management
- Creative Arts & Media
- Cybersecurity
- Data Science & Analytics
- Digital Marketing
- Education & Research
- Engineering & Core Technology
- Finance & Accounting
- Healthcare & Life Sciences
- Law & Public Service
- Software Development
- UI/UX Design


In [57]:
# Verify the model's expected feature order

print("Model input feature order:\n")

for index, feature in enumerate(model_features, start=1):
    print(f"{index:02d}. {feature}")

print("\nTotal features:", len(model_features))

# Check whether the model stores feature names
if hasattr(model, "feature_names_in_"):
    print(
        "\nModel feature names match training columns:",
        list(model.feature_names_in_) == model_features
    )
else:
    print("\nModel does not expose feature_names_in_.")

Model input feature order:

01. age_group
02. education_level
03. experience_stage
04. region_type
05. interest_technology
06. interest_data
07. interest_problem_solving
08. interest_creativity
09. interest_people
10. interest_helping
11. aptitude_logical
12. aptitude_numerical
13. aptitude_verbal
14. aptitude_spatial
15. aptitude_critical_thinking
16. trait_openness
17. trait_conscientiousness
18. trait_extraversion
19. trait_agreeableness
20. trait_emotional_stability
21. skill_programming
22. skill_data_analysis
23. skill_research
24. skill_communication
25. skill_leadership

Total features: 25

Model feature names match training columns: True


In [58]:
# Check which scoring functions are available in the current kernel

required_functions = [
    "calculate_interest_scores",
    "calculate_aptitude_scores",
    "calculate_personality_scores",
    "calculate_skill_scores",
    "calculate_communication_score",
    "validate_categorical_inputs"
]

for function_name in required_functions:
    exists = function_name in globals()
    print(f"{function_name}: {'AVAILABLE' if exists else 'NOT DEFINED'}")

calculate_interest_scores: AVAILABLE
calculate_aptitude_scores: AVAILABLE
calculate_personality_scores: AVAILABLE
calculate_skill_scores: AVAILABLE
calculate_communication_score: AVAILABLE
validate_categorical_inputs: AVAILABLE


In [59]:
# Inspect the existing sample scoring outputs

scoring_outputs = {
    "Interest": sample_interest_scores,
    "Aptitude": sample_scores,
    "Personality": sample_personality_scores,
    "Skills": sample_skill_scores
}

for section, scores in scoring_outputs.items():
    print(f"\n{section} scores")
    print("-" * 40)

    for feature, value in scores.items():
        print(f"{feature}: {value}")

# Combine the available scores
combined_scores = {}

for scores in scoring_outputs.values():
    combined_scores.update(scores)

# Check coverage against the model inputs
categorical_features = [
    "age_group",
    "education_level",
    "experience_stage",
    "region_type"
]

available_features = set(combined_scores) | set(categorical_features)

missing_features = [
    feature for feature in model_features
    if feature not in available_features
]

print("\nFeatures still missing from the profile:")
for feature in missing_features:
    print("-", feature)

print("\nMissing feature count:", len(missing_features))


Interest scores
----------------------------------------
interest_technology: 8.67
interest_data: 10.0
interest_problem_solving: 9.0
interest_creativity: 10.0
interest_people: 7.0
interest_helping: 8.0

Aptitude scores
----------------------------------------
aptitude_numerical: 10.0
aptitude_logical: 10.0
aptitude_critical_thinking: 10.0

Personality scores
----------------------------------------
trait_openness: 9.0
trait_conscientiousness: 8.0
trait_extraversion: 9.0
trait_agreeableness: 8.0
trait_emotional_stability: 6.0

Skills scores
----------------------------------------
skill_programming: 8.0
skill_data_analysis: 10.0
skill_research: None
skill_leadership: 6.0

Features still missing from the profile:
- aptitude_verbal
- aptitude_spatial
- skill_communication

Missing feature count: 3


In [60]:
# Answer key for the new aptitude questions
new_aptitude_answer_key = {
    "Q121": "B",
    "Q122": "A"
}


def calculate_new_feature_scores(user_answers):
    """
    Calculate provisional scores for Q121, Q122 and Q123.
    Aptitude scores use the same 1-10 convention as existing scoring.
    """

    scores = {}

    # Verbal aptitude: Q121
    answer = user_answers.get("Q121")
    if answer in ["A", "B", "C", "D"]:
        scores["aptitude_verbal"] = (
            10.0 if answer == new_aptitude_answer_key["Q121"] else 1.0
        )
    else:
        scores["aptitude_verbal"] = None

    # Spatial aptitude: Q122
    answer = user_answers.get("Q122")
    if answer in ["A", "B", "C", "D"]:
        scores["aptitude_spatial"] = (
            10.0 if answer == new_aptitude_answer_key["Q122"] else 1.0
        )
    else:
        scores["aptitude_spatial"] = None

    # Communication skill: Q123
    scores["skill_communication"] = calculate_communication_score(
        user_answers.get("Q123")
    )

    return scores


# Test using sample answers only
sample_new_answers = {
    "Q121": "B",
    "Q122": "A",
    "Q123": 4
}

sample_new_feature_scores = calculate_new_feature_scores(
    sample_new_answers
)

print("New feature scores:")
for feature, value in sample_new_feature_scores.items():
    print(f"{feature}: {value}")

New feature scores:
aptitude_verbal: 10.0
aptitude_spatial: 10.0
skill_communication: 8.0


In [61]:
# Combine background details and all sample assessment scores

combined_profile = {
    **student_details,
    **sample_interest_scores,
    **sample_scores,
    **sample_new_feature_scores,
    **sample_personality_scores,
    **sample_skill_scores
}

# Keep exactly the model's expected feature order
student_profile_df = pd.DataFrame(
    [{feature: combined_profile.get(feature) for feature in model_features}],
    columns=model_features
)

print("Profile shape:", student_profile_df.shape)

print("\nComplete profile:")
display(student_profile_df.T.rename(columns={0: "Sample value"}))

# Identify missing or invalid values
missing_values = student_profile_df.columns[
    student_profile_df.isna().any()
].tolist()

print("\nFeatures with missing values:", missing_values)
print("Missing feature count:", len(missing_values))

Profile shape: (1, 25)

Complete profile:


,Sample value
age_group,19-22
education_level,B.Tech IT
experience_stage,Student
region_type,Semi-urban
interest_technology,8.67
interest_data,10.0
interest_problem_solving,9.0
interest_creativity,10.0
interest_people,7.0
interest_helping,8.0



Features with missing values: ['skill_research']
Missing feature count: 1


In [62]:
# Validate a student profile before sending it to the model

def validate_student_profile(profile_df):
    errors = []

    if list(profile_df.columns) != model_features:
        errors.append("Feature names or feature order do not match the model.")

    if len(profile_df) != 1:
        errors.append("Expected exactly one student profile.")

    if profile_df[model_features].isna().any().any():
        missing = profile_df.columns[
            profile_df.isna().any()
        ].tolist()
        errors.append(f"Missing values: {missing}")

    for feature in [
        "age_group",
        "education_level",
        "experience_stage",
        "region_type"
    ]:
        if feature in profile_df.columns:
            value = profile_df.iloc[0][feature]
            if value not in valid_categories[feature]:
                errors.append(f"Invalid category for {feature}: {value}")

    numeric_features = [
        feature for feature in model_features
        if feature not in [
            "age_group",
            "education_level",
            "experience_stage",
            "region_type"
        ]
    ]

    for feature in numeric_features:
        value = profile_df.iloc[0][feature]

        if pd.notna(value):
            try:
                numeric_value = float(value)
                if not 1 <= numeric_value <= 10:
                    errors.append(
                        f"{feature} must be between 1 and 10."
                    )
            except (TypeError, ValueError):
                errors.append(f"{feature} must be numeric.")

    return errors


profile_errors = validate_student_profile(student_profile_df)

if profile_errors:
    print("Profile is NOT ready for prediction:")
    for error in profile_errors:
        print("-", error)
else:
    print("Profile is ready for prediction.")

Profile is NOT ready for prediction:
- Missing values: ['skill_research']


In [63]:
# Step 43: Fill the missing sample research-skill score

sample_skill_scores["skill_research"] = 6.0  # Sample rating: 3/5 × 2

# Rebuild the complete student profile in the model's exact feature order
combined_profile = {
    **student_details,
    **sample_interest_scores,
    **sample_scores,
    **sample_new_feature_scores,
    **sample_personality_scores,
    **sample_skill_scores
}

student_profile_df = pd.DataFrame(
    [{feature: combined_profile.get(feature) for feature in model_features}],
    columns=model_features
)

# Validate again
profile_errors = validate_student_profile(student_profile_df)

if profile_errors:
    print("Profile is NOT ready for prediction:")
    for error in profile_errors:
        print("-", error)
else:
    print("Profile is ready for prediction.")
    print("Profile shape:", student_profile_df.shape)
    print("Missing values:", student_profile_df.isna().sum().sum())

Profile is ready for prediction.
Profile shape: (1, 25)
Missing values: 0


In [64]:
# Step 44: Predict the recommended career domain

# Generate predictions
predicted_career = model.predict(student_profile_df)[0]

# Get prediction probabilities
prediction_probabilities = model.predict_proba(student_profile_df)[0]

# Sort careers by probability, from highest to lowest
ranked_indices = prediction_probabilities.argsort()[::-1]

print("CAREER GUIDANCE RESULTS")
print("=" * 40)
print("Recommended Career Domain:", predicted_career)

print("\nTop 3 Career Recommendations:")
for rank, index in enumerate(ranked_indices[:3], start=1):
    career = career_labels[index]
    probability = prediction_probabilities[index] * 100

    print(f"{rank}. {career} — {probability:.2f}% model score")

CAREER GUIDANCE RESULTS
Recommended Career Domain: Data Science & Analytics

Top 3 Career Recommendations:
1. Data Science & Analytics — 42.79% model score
2. Education & Research — 29.67% model score
3. Engineering & Core Technology — 11.18% model score


In [65]:
# Step 46: Verify prediction consistency

predicted_career = model.predict(student_profile_df)[0]
prediction_probabilities = model.predict_proba(student_profile_df)[0]

top_index = prediction_probabilities.argmax()
top_career = model.classes_[top_index]
top_score = prediction_probabilities[top_index] * 100

print("Predicted career:", predicted_career)
print("Highest-scoring career:", top_career)
print(f"Highest model score: {top_score:.2f}%")
print("Prediction consistent:", predicted_career == top_career)

Predicted career: Data Science & Analytics
Highest-scoring career: Data Science & Analytics
Highest model score: 42.79%
Prediction consistent: True


In [66]:
# Step 47: Inspect the trained model for explainability support

print("Model type:", type(model).__name__)
print("\nEnsemble components:")

for name, estimator in model.named_estimators_.items():
    print(f"- {name}: {type(estimator).__name__}")

print("\nFeature importance support:")

for name, estimator in model.named_estimators_.items():
    supports_importance = (
        hasattr(estimator, "feature_importances_")
        or hasattr(estimator, "coef_")
    )
    print(f"- {name}: {'Yes' if supports_importance else 'No'}")

Model type: VotingClassifier

Ensemble components:
- gnb: Pipeline
- lr: Pipeline
- mlp: Pipeline
- extra_trees: Pipeline

Feature importance support:
- gnb: No
- lr: No
- mlp: No
- extra_trees: No


In [67]:
# Step 48: Inspect each pipeline's internal steps

for name, pipeline in model.named_estimators_.items():
    print(f"\n{name.upper()}")

    for step_name, step_object in pipeline.steps:
        print(f"  {step_name}: {type(step_object).__name__}")

        if hasattr(step_object, "feature_importances_"):
            print("  Feature importance: Available")

        if hasattr(step_object, "coef_"):
            print("  Coefficients: Available")


GNB
  preprocessor: ColumnTransformer
  classifier: GaussianNB

LR
  preprocessor: ColumnTransformer
  classifier: LogisticRegression
  Coefficients: Available

MLP
  preprocessor: ColumnTransformer
  classifier: MLPClassifier

EXTRA_TREES
  preprocessor: ColumnTransformer
  classifier: ExtraTreesClassifier
  Feature importance: Available


In [68]:
# Step 49: Extract feature importance from Extra Trees

extra_trees_pipeline = model.named_estimators_["extra_trees"]
extra_trees_classifier = extra_trees_pipeline.named_steps["classifier"]
preprocessor = extra_trees_pipeline.named_steps["preprocessor"]

# Get the transformed feature names after preprocessing
transformed_features = preprocessor.get_feature_names_out()

# Extract importance values
importance_values = extra_trees_classifier.feature_importances_

# Create a sorted feature-importance table
feature_importance_df = pd.DataFrame({
    "Feature": transformed_features,
    "Importance": importance_values
}).sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print("Top 15 Features — Extra Trees Component")
display(feature_importance_df.head(15))

Top 15 Features — Extra Trees Component


,Feature,Importance
0,numeric__interest_helping,0.038490
1,numeric__interest_creativity,0.035169
2,numeric__interest_data,0.032157
3,numeric__skill_leadership,0.032031
4,numeric__skill_research,0.031714
5,numeric__skill_data_analysis,0.029505
6,numeric__trait_extraversion,0.026446
7,numeric__aptitude_numerical,0.025344
8,numeric__skill_communication,0.023139
9,numeric__trait_emotional_stability,0.021953


In [69]:
# Step 50: Compare global feature importance with student scores

student_values = student_profile_df.iloc[0]

comparison_df = feature_importance_df.copy()

# Remove preprocessing prefixes such as "numeric__"
comparison_df["Feature"] = comparison_df["Feature"].str.replace(
    r"^[^_]+__", "", regex=True
)

comparison_df["Student Score"] = comparison_df["Feature"].map(
    student_values.to_dict()
)

comparison_df = comparison_df.sort_values(
    "Importance", ascending=False
).reset_index(drop=True)

print("Top 15 Features and Sample Student Scores")
display(comparison_df.head(15))

Top 15 Features and Sample Student Scores


,Feature,Importance,Student Score
0,interest_helping,0.038490,8.0
1,interest_creativity,0.035169,10.0
2,interest_data,0.032157,10.0
3,skill_leadership,0.032031,6.0
4,skill_research,0.031714,6.0
5,skill_data_analysis,0.029505,10.0
6,trait_extraversion,0.026446,9.0
7,aptitude_numerical,0.025344,10.0
8,skill_communication,0.023139,8.0
9,trait_emotional_stability,0.021953,6.0


In [70]:
# Step 51: Compare Extra Trees with the ensemble prediction

# Ensemble probabilities
ensemble_probabilities = model.predict_proba(student_profile_df)[0]
ensemble_classes = model.classes_

# Extra Trees probabilities
extra_trees_probabilities = extra_trees_pipeline.predict_proba(
    student_profile_df
)[0]
extra_trees_classes = extra_trees_pipeline.classes_

# Create comparison table
comparison = pd.DataFrame({
    "Career Domain": ensemble_classes,
    "Ensemble Score (%)": ensemble_probabilities * 100
})

extra_trees_scores = dict(zip(
    extra_trees_classes,
    extra_trees_probabilities * 100
))

comparison["Extra Trees Score (%)"] = comparison["Career Domain"].map(
    extra_trees_scores
)

comparison = comparison.sort_values(
    "Ensemble Score (%)", ascending=False
).reset_index(drop=True)

print("Top 3 Career Domains: Ensemble vs Extra Trees")
display(comparison.head(3).round(2))

Top 3 Career Domains: Ensemble vs Extra Trees


,Career Domain,Ensemble Score (%),Extra Trees Score (%)
0,Data Science & Analytics,42.79,NaN
1,Education & Research,29.67,NaN
2,Engineering & Core Technology,11.18,NaN


In [71]:
# Step 52: Inspect career labels from both models

print("Ensemble classes:")
for career in model.classes_:
    print(repr(career))

print("\nExtra Trees pipeline classes:")
for career in extra_trees_pipeline.classes_:
    print(repr(career))

print("\nClass counts:")
print("Ensemble:", len(model.classes_))
print("Extra Trees:", len(extra_trees_pipeline.classes_))

Ensemble classes:
'Business & Management'
'Creative Arts & Media'
'Cybersecurity'
'Data Science & Analytics'
'Digital Marketing'
'Education & Research'
'Engineering & Core Technology'
'Finance & Accounting'
'Healthcare & Life Sciences'
'Law & Public Service'
'Software Development'
'UI/UX Design'

Extra Trees pipeline classes:
np.int64(0)
np.int64(1)
np.int64(2)
np.int64(3)
np.int64(4)
np.int64(5)
np.int64(6)
np.int64(7)
np.int64(8)
np.int64(9)
np.int64(10)
np.int64(11)

Class counts:
Ensemble: 12
Extra Trees: 12


In [72]:
# Step 53: Inspect the model's preprocessing and target encoding

print("Preprocessor steps:")
for name, step in extra_trees_pipeline.steps:
    print(f"- {name}: {type(step).__name__}")

print("\nModel attributes related to target labels:")
for attribute in ["le_", "label_encoder_", "classes_"]:
    print(f"{attribute}: {hasattr(model, attribute)}")

print("\nExtra Trees numeric classes:")
print(extra_trees_pipeline.classes_)

Preprocessor steps:
- preprocessor: ColumnTransformer
- classifier: ExtraTreesClassifier

Model attributes related to target labels:
le_: True
label_encoder_: False
classes_: True

Extra Trees numeric classes:
[ 0  1  2  3  4  5  6  7  8  9 10 11]


In [73]:
# Step 54: Inspect and use the fitted label encoder

label_encoder = model.le_

print("Label encoder type:", type(label_encoder).__name__)
print("\nEncoded classes and their career names:")

for encoded_label, career_name in enumerate(label_encoder.classes_):
    print(f"{encoded_label} -> {career_name}")

Label encoder type: LabelEncoder

Encoded classes and their career names:
0 -> Business & Management
1 -> Creative Arts & Media
2 -> Cybersecurity
3 -> Data Science & Analytics
4 -> Digital Marketing
5 -> Education & Research
6 -> Engineering & Core Technology
7 -> Finance & Accounting
8 -> Healthcare & Life Sciences
9 -> Law & Public Service
10 -> Software Development
11 -> UI/UX Design


In [74]:
# Step 55: Compare career scores using the verified label encoder

# Ensemble probabilities use career names directly
ensemble_probabilities = model.predict_proba(student_profile_df)[0]

# Extra Trees probabilities use numeric labels
extra_trees_probabilities = extra_trees_pipeline.predict_proba(
    student_profile_df
)[0]

# Decode Extra Trees numeric labels into career names
extra_trees_career_names = model.le_.inverse_transform(
    extra_trees_pipeline.classes_
)

# Build the comparison table
comparison_df = pd.DataFrame({
    "Career Domain": model.classes_,
    "Ensemble Score (%)": ensemble_probabilities * 100
})

extra_trees_score_map = dict(zip(
    extra_trees_career_names,
    extra_trees_probabilities * 100
))

comparison_df["Extra Trees Score (%)"] = comparison_df[
    "Career Domain"
].map(extra_trees_score_map)

comparison_df = comparison_df.sort_values(
    "Ensemble Score (%)",
    ascending=False
).reset_index(drop=True)

print("Top 3 Career Domains: Ensemble vs Extra Trees")
display(comparison_df.head(3).round(2))

Top 3 Career Domains: Ensemble vs Extra Trees


,Career Domain,Ensemble Score (%),Extra Trees Score (%)
0,Data Science & Analytics,42.79,24.0
1,Education & Research,29.67,20.5
2,Engineering & Core Technology,11.18,8.0


In [75]:
# Step 56: Compare the rankings of both models

comparison_df["Ensemble Rank"] = (
    comparison_df["Ensemble Score (%)"]
    .rank(ascending=False, method="min")
    .astype(int)
)

comparison_df["Extra Trees Rank"] = (
    comparison_df["Extra Trees Score (%)"]
    .rank(ascending=False, method="min")
    .astype(int)
)

comparison_df["Rank Difference"] = (
    comparison_df["Ensemble Rank"]
    - comparison_df["Extra Trees Rank"]
).abs()

print("Ranking Comparison — All 12 Career Domains")
display(
    comparison_df[
        [
            "Career Domain",
            "Ensemble Rank",
            "Extra Trees Rank",
            "Rank Difference"
        ]
    ].sort_values("Ensemble Rank").reset_index(drop=True)
)

exact_rank_matches = (
    comparison_df["Ensemble Rank"]
    == comparison_df["Extra Trees Rank"]
).sum()

print(
    f"\nExact rank matches: {exact_rank_matches}/12"
)

Ranking Comparison — All 12 Career Domains


,Career Domain,Ensemble Rank,Extra Trees Rank,Rank Difference
0,Data Science & Analytics,1,1,0
1,Education & Research,2,2,0
2,Engineering & Core Technology,3,5,2
3,Law & Public Service,4,4,0
4,Digital Marketing,5,3,2
5,Software Development,6,6,0
6,Business & Management,7,7,0
7,Finance & Accounting,8,8,0
8,UI/UX Design,9,8,1
9,Healthcare & Life Sciences,10,10,0



Exact rank matches: 9/12


In [76]:
# Step 57: Analyze score differences between the two models

comparison_df["Score Difference (%)"] = (
    comparison_df["Ensemble Score (%)"]
    - comparison_df["Extra Trees Score (%)"]
).abs()

print("Model Agreement Summary")
print("-" * 40)

print(
    "Average absolute score difference:",
    round(comparison_df["Score Difference (%)"].mean(), 2),
    "%"
)

print(
    "Careers with identical ranks:",
    int((comparison_df["Ensemble Rank"] ==
         comparison_df["Extra Trees Rank"]).sum()),
    "out of",
    len(comparison_df)
)

print("\nLargest Score Differences")
display(
    comparison_df[
        [
            "Career Domain",
            "Ensemble Score (%)",
            "Extra Trees Score (%)",
            "Score Difference (%)"
        ]
    ]
    .sort_values("Score Difference (%)", ascending=False)
    .head(5)
    .round(2)
    .reset_index(drop=True)
)

Model Agreement Summary
----------------------------------------
Average absolute score difference: 5.19 %
Careers with identical ranks: 9 out of 12

Largest Score Differences


,Career Domain,Ensemble Score (%),Extra Trees Score (%),Score Difference (%)
0,Data Science & Analytics,42.79,24.0,18.79
1,Digital Marketing,3.59,13.5,9.91
2,Education & Research,29.67,20.5,9.17
3,Law & Public Service,5.86,11.0,5.14
4,Software Development,2.62,7.0,4.38


In [77]:
# Step 58: Validate the ensemble probability distribution

ensemble_probabilities = model.predict_proba(student_profile_df)[0]

print(f"Total probability: {ensemble_probabilities.sum() * 100:.2f}%")
print(f"Minimum class score: {ensemble_probabilities.min() * 100:.2f}%")
print(f"Maximum class score: {ensemble_probabilities.max() * 100:.2f}%")

assert abs(ensemble_probabilities.sum() - 1.0) < 1e-6, \
    "Probability scores do not sum to 100%."

assert (ensemble_probabilities >= 0).all(), \
    "Negative probability score detected."

print("\nProbability validation passed.")

Total probability: 100.00%
Minimum class score: 0.25%
Maximum class score: 42.79%

Probability validation passed.


In [78]:
# Step 59: Reusable career recommendation function

def recommend_careers(profile_df, top_n=3):
    # Validate the profile
    errors = validate_student_profile(profile_df)

    if errors:
        raise ValueError(
            "Invalid student profile:\n" + "\n".join(errors)
        )

    if not isinstance(top_n, int) or not 1 <= top_n <= len(model.classes_):
        raise ValueError(
            f"top_n must be an integer between 1 and {len(model.classes_)}."
        )

    # Predict the leading career and obtain class scores
    probabilities = model.predict_proba(profile_df)[0]
    predicted_career = model.predict(profile_df)[0]

    # Rank careers from highest to lowest score
    ranked_indices = probabilities.argsort()[::-1][:top_n]

    recommendations = pd.DataFrame({
        "Rank": range(1, top_n + 1),
        "Career Domain": model.classes_[ranked_indices],
        "Model Score (%)": (
            probabilities[ranked_indices] * 100
        ).round(2)
    })

    return predicted_career, recommendations


# Test the function using our existing sample profile
predicted_career, recommendations = recommend_careers(
    student_profile_df,
    top_n=3
)

print("Recommended Career:", predicted_career)
print("\nRanked Recommendations:")
display(recommendations)

Recommended Career: Data Science & Analytics

Ranked Recommendations:


,Rank,Career Domain,Model Score (%)
0,1,Data Science & Analytics,42.79
1,2,Education & Research,29.67
2,3,Engineering & Core Technology,11.18


In [79]:
# Step 60: Define information for the 12 career domains

career_information = {
    "Business & Management": {
        "description": "Focuses on business operations, planning, and organizational decision-making.",
        "skills": ["Leadership", "Communication", "Problem-solving"],
        "roles": ["Business Analyst", "Project Coordinator", "Management Trainee"]
    },
    "Creative Arts & Media": {
        "description": "Focuses on creative content, visual communication, and media production.",
        "skills": ["Creativity", "Visual Design", "Communication"],
        "roles": ["Content Creator", "Media Designer", "Creative Producer"]
    },
    "Cybersecurity": {
        "description": "Focuses on protecting computer systems, networks, and digital information.",
        "skills": ["Networking", "Security Analysis", "Problem-solving"],
        "roles": ["Security Analyst", "SOC Analyst", "Security Engineer"]
    },
    "Data Science & Analytics": {
        "description": "Uses data analysis, statistics, and machine learning to support decisions.",
        "skills": ["Data Analysis", "Numerical Reasoning", "Programming"],
        "roles": ["Data Analyst", "Junior Data Scientist", "BI Analyst"]
    },
    "Digital Marketing": {
        "description": "Focuses on online campaigns, audience engagement, and marketing analytics.",
        "skills": ["Communication", "Creativity", "Data Analysis"],
        "roles": ["Digital Marketing Analyst", "SEO Analyst", "Campaign Executive"]
    },
    "Education & Research": {
        "description": "Focuses on teaching, investigation, and developing new knowledge.",
        "skills": ["Research", "Communication", "Critical Thinking"],
        "roles": ["Research Assistant", "Teaching Assistant", "Academic Researcher"]
    },
    "Engineering & Core Technology": {
        "description": "Applies engineering and technical principles to design and improve systems.",
        "skills": ["Technical Analysis", "Problem-solving", "Mathematics"],
        "roles": ["Systems Engineer", "Technical Analyst", "Engineering Associate"]
    },
    "Finance & Accounting": {
        "description": "Focuses on financial records, budgeting, reporting, and analysis.",
        "skills": ["Numerical Reasoning", "Attention to Detail", "Data Analysis"],
        "roles": ["Financial Analyst", "Accounts Associate", "Audit Assistant"]
    },
    "Healthcare & Life Sciences": {
        "description": "Focuses on health services, biological sciences, and healthcare research.",
        "skills": ["Scientific Reasoning", "Communication", "Research"],
        "roles": ["Clinical Research Assistant", "Lab Assistant", "Health Data Analyst"]
    },
    "Law & Public Service": {
        "description": "Focuses on legal systems, public administration, and civic services.",
        "skills": ["Verbal Reasoning", "Research", "Critical Thinking"],
        "roles": ["Legal Assistant", "Policy Research Assistant", "Public Service Associate"]
    },
    "Software Development": {
        "description": "Focuses on designing, building, testing, and maintaining software applications.",
        "skills": ["Programming", "Logical Reasoning", "Problem-solving"],
        "roles": ["Junior Software Developer", "QA Engineer", "Application Developer"]
    },
    "UI/UX Design": {
        "description": "Focuses on designing usable, accessible, and intuitive digital experiences.",
        "skills": ["Creativity", "User Research", "Visual Design"],
        "roles": ["UI Designer", "UX Research Assistant", "Product Design Associate"]
    }
}

print("Career domains in model:", len(model.classes_))
print("Career domains in information dictionary:", len(career_information))

missing_careers = set(model.classes_) - set(career_information)
extra_careers = set(career_information) - set(model.classes_)

print("Missing career information:", missing_careers)
print("Unexpected career entries:", extra_careers)

assert not missing_careers and not extra_careers, \
    "Career information does not match the model's classes."

print("\nCareer information validation passed.")


Career domains in model: 12
Career domains in information dictionary: 12
Missing career information: set()
Unexpected career entries: set()

Career information validation passed.


In [80]:
# Step 61: Generate enriched career recommendations

def get_detailed_recommendations(profile_df, top_n=3):
    predicted_career, recommendations = recommend_careers(
        profile_df,
        top_n=top_n
    )

    # Attach career information to each recommendation
    recommendations["Description"] = recommendations[
        "Career Domain"
    ].map(lambda career: career_information[career]["description"])

    recommendations["Relevant Skills"] = recommendations[
        "Career Domain"
    ].map(lambda career: ", ".join(
        career_information[career]["skills"]
    ))

    recommendations["Possible Job Roles"] = recommendations[
        "Career Domain"
    ].map(lambda career: ", ".join(
        career_information[career]["roles"]
    ))

    return predicted_career, recommendations


# Test using the existing sample student profile
predicted_career, detailed_recommendations = (
    get_detailed_recommendations(student_profile_df, top_n=3)
)

print("PRIMARY RECOMMENDATION:", predicted_career)
print("\nDETAILED CAREER RECOMMENDATIONS")
display(detailed_recommendations)

PRIMARY RECOMMENDATION: Data Science & Analytics

DETAILED CAREER RECOMMENDATIONS


,Rank,Career Domain,Model Score (%),Description,Relevant Skills,Possible Job Roles
0,1,Data Science & Analytics,42.79,"Uses data analysis, statistics, and machine le...","Data Analysis, Numerical Reasoning, Programming","Data Analyst, Junior Data Scientist, BI Analyst"
1,2,Education & Research,29.67,"Focuses on teaching, investigation, and develo...","Research, Communication, Critical Thinking","Research Assistant, Teaching Assistant, Academ..."
2,3,Engineering & Core Technology,11.18,Applies engineering and technical principles t...,"Technical Analysis, Problem-solving, Mathematics","Systems Engineer, Technical Analyst, Engineeri..."


In [81]:
# Step 62: Define profile features relevant to each career domain

career_feature_mapping = {
    "Data Science & Analytics": [
        "interest_data", "interest_problem_solving",
        "aptitude_numerical", "aptitude_logical",
        "skill_data_analysis", "skill_programming"
    ],
    "Education & Research": [
        "interest_helping", "interest_people",
        "aptitude_verbal", "aptitude_critical_thinking",
        "skill_research", "skill_communication"
    ],
    "Engineering & Core Technology": [
        "interest_technology", "interest_problem_solving",
        "aptitude_logical", "aptitude_numerical",
        "skill_programming", "skill_research"
    ],
    "Software Development": [
        "interest_technology", "interest_problem_solving",
        "aptitude_logical", "skill_programming",
        "skill_data_analysis"
    ],
    "Cybersecurity": [
        "interest_technology", "interest_problem_solving",
        "aptitude_logical", "aptitude_critical_thinking",
        "skill_programming", "skill_research"
    ],
    "Creative Arts & Media": [
        "interest_creativity", "interest_people",
        "trait_openness", "skill_communication"
    ],
    "UI/UX Design": [
        "interest_creativity", "interest_people",
        "trait_openness", "skill_communication"
    ],
    "Digital Marketing": [
        "interest_creativity", "interest_people",
        "interest_data", "skill_communication"
    ],
    "Business & Management": [
        "interest_people", "interest_problem_solving",
        "trait_extraversion", "skill_leadership",
        "skill_communication"
    ],
    "Finance & Accounting": [
        "interest_data", "aptitude_numerical",
        "aptitude_logical", "skill_data_analysis"
    ],
    "Healthcare & Life Sciences": [
        "interest_helping", "interest_data",
        "aptitude_critical_thinking", "skill_research",
        "skill_communication"
    ],
    "Law & Public Service": [
        "interest_people", "interest_helping",
        "aptitude_verbal", "aptitude_critical_thinking",
        "skill_communication"
    ]
}

# Check that every mapped feature exists in the trained model
invalid_features = {
    career: [
        feature for feature in features
        if feature not in model_features
    ]
    for career, features in career_feature_mapping.items()
}

invalid_features = {
    career: features
    for career, features in invalid_features.items()
    if features
}

print("Career mappings:", len(career_feature_mapping))
print("Invalid feature mappings:", invalid_features)

assert set(career_feature_mapping) == set(model.classes_)
assert not invalid_features

print("Career-profile feature mapping validation passed.")

Career mappings: 12
Invalid feature mappings: {}
Career-profile feature mapping validation passed.


In [82]:
# Step 63: Calculate heuristic profile-match scores

def calculate_profile_match_scores(profile_df):
    if len(profile_df) != 1:
        raise ValueError("Expected exactly one student profile.")

    student = profile_df.iloc[0]
    results = []

    for career, features in career_feature_mapping.items():
        valid_scores = []

        for feature in features:
            value = student[feature]

            if pd.notna(value):
                score = float(value)

                if 1 <= score <= 10:
                    valid_scores.append(score)

        if valid_scores:
            match_score = sum(valid_scores) / len(valid_scores) * 10
        else:
            match_score = None

        results.append({
            "Career Domain": career,
            "Profile Match Score (%)": (
                round(match_score, 2)
                if match_score is not None else None
            ),
            "Features Used": len(valid_scores),
            "Features Expected": len(features)
        })

    return pd.DataFrame(results).sort_values(
        "Profile Match Score (%)",
        ascending=False,
        na_position="last"
    ).reset_index(drop=True)


profile_match_df = calculate_profile_match_scores(student_profile_df)

print("Career Profile-Match Scores")
display(profile_match_df)

Career Profile-Match Scores


,Career Domain,Profile Match Score (%),Features Used,Features Expected
0,Finance & Accounting,100.00,4,4
1,Data Science & Analytics,95.00,6,6
2,Software Development,91.34,5,5
3,Digital Marketing,87.50,4,4
4,Cybersecurity,86.12,6,6
5,Engineering & Core Technology,86.12,6,6
6,Law & Public Service,86.00,5,5
7,UI/UX Design,85.00,4,4
8,Creative Arts & Media,85.00,4,4
9,Healthcare & Life Sciences,84.00,5,5


In [83]:
# Step 64: Compare ML recommendations with profile-match scores

ml_scores_df = comparison_df[
    ["Career Domain", "Ensemble Score (%)", "Ensemble Rank"]
].copy()

match_scores_df = profile_match_df[
    ["Career Domain", "Profile Match Score (%)"]
].copy()

combined_results_df = ml_scores_df.merge(
    match_scores_df,
    on="Career Domain",
    how="left",
    validate="one_to_one"
)

combined_results_df = combined_results_df.sort_values(
    "Ensemble Rank"
).reset_index(drop=True)

combined_results_df["Profile Match Rank"] = (
    combined_results_df["Profile Match Score (%)"]
    .rank(ascending=False, method="min")
    .astype(int)
)

print("ML Recommendation vs Profile Match")
display(
    combined_results_df[
        [
            "Career Domain",
            "Ensemble Rank",
            "Ensemble Score (%)",
            "Profile Match Rank",
            "Profile Match Score (%)"
        ]
    ].round(2)
)

ML Recommendation vs Profile Match


,Career Domain,Ensemble Rank,Ensemble Score (%),Profile Match Rank,Profile Match Score (%)
0,Data Science & Analytics,1,42.79,2,95.00
1,Education & Research,2,29.67,11,81.67
2,Engineering & Core Technology,3,11.18,5,86.12
3,Law & Public Service,4,5.86,7,86.00
4,Digital Marketing,5,3.59,4,87.50
5,Software Development,6,2.62,3,91.34
6,Business & Management,7,1.54,12,78.00
7,Finance & Accounting,8,0.76,1,100.00
8,UI/UX Design,9,0.75,8,85.00
9,Healthcare & Life Sciences,10,0.50,10,84.00


In [84]:
# Step 65: Inspect mapped feature scores for Finance & Accounting

career_to_inspect = "Finance & Accounting"
features = career_feature_mapping[career_to_inspect]

finance_feature_scores = pd.DataFrame({
    "Feature": features,
    "Student Score": [
        student_profile_df.iloc[0][feature]
        for feature in features
    ]
})

print(f"Feature scores for: {career_to_inspect}")
display(finance_feature_scores)

print(
    "\nMean feature score:",
    round(finance_feature_scores["Student Score"].mean(), 2),
    "/ 10"
)

Feature scores for: Finance & Accounting


,Feature,Student Score
0,interest_data,10.0
1,aptitude_numerical,10.0
2,aptitude_logical,10.0
3,skill_data_analysis,10.0



Mean feature score: 10.0 / 10


In [85]:
# Step 66: Analyze feature reuse across career mappings

from collections import Counter

feature_usage = Counter(
    feature
    for features in career_feature_mapping.values()
    for feature in features
)

feature_usage_df = pd.DataFrame([
    {
        "Feature": feature,
        "Number of Career Mappings": count
    }
    for feature, count in feature_usage.items()
]).sort_values(
    "Number of Career Mappings",
    ascending=False
).reset_index(drop=True)

print("Most Reused Features Across Career Domains")
display(feature_usage_df.head(15))

print("\nTotal unique mapped features:", len(feature_usage))
print("Total feature-to-career assignments:", sum(feature_usage.values()))

Most Reused Features Across Career Domains


,Feature,Number of Career Mappings
0,skill_communication,7
1,interest_people,6
2,aptitude_logical,5
3,interest_problem_solving,5
4,interest_data,4
5,aptitude_critical_thinking,4
6,skill_research,4
7,skill_programming,4
8,skill_data_analysis,3
9,interest_creativity,3



Total unique mapped features: 17
Total feature-to-career assignments: 60


In [86]:
# Step 67: Inspect the career guidance questionnaire workbook

from pathlib import Path
import pandas as pd

questionnaire_path = project_dir / "AI_Career_Guidance_Questions.xlsx"

excel_file = pd.ExcelFile(questionnaire_path)

print("Workbook sheets:")
print(excel_file.sheet_names)

for sheet_name in excel_file.sheet_names:
    sheet_df = pd.read_excel(questionnaire_path, sheet_name=sheet_name)

    print(f"\nSheet: {sheet_name}")
    print("Shape:", sheet_df.shape)
    print("Columns:", sheet_df.columns.tolist())
    display(sheet_df.head(5))

Workbook sheets:
['Questions Dataset']

Sheet: Questions Dataset
Shape: (123, 8)
Columns: ['Question_ID', 'Section', 'Question_Text', 'Answer_Type', 'Answer_Options', 'Scoring_Method', 'Feature_Mapping', 'Purpose']


,Question_ID,Section,Question_Text,Answer_Type,Answer_Options,Scoring_Method,Feature_Mapping,Purpose
0,Q001,Interest,How interested are you in writing scripts or c...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in core programming and logic
1,Q002,Interest,How interested are you in analyzing large data...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in data science and statisti...
2,Q003,Interest,How interested are you in finding vulnerabilit...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in cybersecurity and risk mi...
3,Q004,Interest,How interested are you in sketching out wirefr...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in UI/UX and user-centric de...
4,Q005,Interest,How interested are you in coordinating team me...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in business management and l...


In [87]:
# Step 68: Analyze questionnaire sections and scoring metadata

questions_df = pd.read_excel(
    questionnaire_path,
    sheet_name="Questions Dataset"
)

print("Question count by section:")
display(
    questions_df.groupby("Section")
    .agg(
        Total_Questions=("Question_ID", "count"),
        Answer_Types=("Answer_Type", "nunique")
    )
    .sort_values("Total_Questions", ascending=False)
)

print("\nQuestion ID ranges by section:")
display(
    questions_df.groupby("Section").agg(
        First_Question=("Question_ID", "min"),
        Last_Question=("Question_ID", "max"),
        Total_Questions=("Question_ID", "count")
    )
)

print("\nScoring metadata completeness:")
display(
    questions_df.groupby("Section").agg(
        Total_Questions=("Question_ID", "count"),
        Scoring_Method_Missing=("Scoring_Method", lambda x: x.isna().sum()),
        Feature_Mapping_Missing=("Feature_Mapping", lambda x: x.isna().sum()),
        Purpose_Missing=("Purpose", lambda x: x.isna().sum())
    )
)

Question count by section:


,Total_Questions,Answer_Types
Section,,
Education & Career Preferences,80,2
Aptitude,12,1
Skills,11,1
Interest,10,1
Personality,10,1



Question ID ranges by section:


,First_Question,Last_Question,Total_Questions
Section,,,
Aptitude,Q011,Q122,12
Education & Career Preferences,Q041,Q120,80
Interest,Q001,Q010,10
Personality,Q021,Q030,10
Skills,Q031,Q123,11



Scoring metadata completeness:


,Total_Questions,Scoring_Method_Missing,Feature_Mapping_Missing,Purpose_Missing
Section,,,,
Aptitude,12,10,10,0
Education & Career Preferences,80,80,80,0
Interest,10,10,10,0
Personality,10,10,10,0
Skills,11,10,10,0


In [88]:
# Step 69: Verify question ID and section consistency

for section in questions_df["Section"].unique():
    section_df = questions_df[
        questions_df["Section"] == section
    ].copy()

    print(f"\n{'=' * 70}")
    print(f"SECTION: {section}")
    print(f"Total questions: {len(section_df)}")
    print("Question IDs:")

    print(", ".join(section_df["Question_ID"].astype(str)))

print("\nDuplicate Question IDs:")
display(
    questions_df[
        questions_df["Question_ID"].duplicated(keep=False)
    ].sort_values("Question_ID")
)

print("\nQuestions without scoring metadata:")
display(
    questions_df[
        questions_df["Scoring_Method"].isna()
        | questions_df["Feature_Mapping"].isna()
    ][
        ["Question_ID", "Section", "Question_Text",
         "Scoring_Method", "Feature_Mapping"]
    ]
)


SECTION: Interest
Total questions: 10
Question IDs:
Q001, Q002, Q003, Q004, Q005, Q006, Q007, Q008, Q009, Q010

SECTION: Aptitude
Total questions: 12
Question IDs:
Q011, Q012, Q013, Q014, Q015, Q016, Q017, Q018, Q019, Q020, Q121, Q122

SECTION: Personality
Total questions: 10
Question IDs:
Q021, Q022, Q023, Q024, Q025, Q026, Q027, Q028, Q029, Q030

SECTION: Skills
Total questions: 11
Question IDs:
Q031, Q032, Q033, Q034, Q035, Q036, Q037, Q038, Q039, Q040, Q123

SECTION: Education & Career Preferences
Total questions: 80
Question IDs:
Q041, Q042, Q043, Q044, Q045, Q046, Q047, Q048, Q049, Q050, Q051, Q052, Q053, Q054, Q055, Q056, Q057, Q058, Q059, Q060, Q061, Q062, Q063, Q064, Q065, Q066, Q067, Q068, Q069, Q070, Q071, Q072, Q073, Q074, Q075, Q076, Q077, Q078, Q079, Q080, Q081, Q082, Q083, Q084, Q085, Q086, Q087, Q088, Q089, Q090, Q091, Q092, Q093, Q094, Q095, Q096, Q097, Q098, Q099, Q100, Q101, Q102, Q103, Q104, Q105, Q106, Q107, Q108, Q109, Q110, Q111, Q112, Q113, Q114, Q115, Q116, Q1

,Question_ID,Section,Question_Text,Answer_Type,Answer_Options,Scoring_Method,Feature_Mapping,Purpose



Questions without scoring metadata:


,Question_ID,Section,Question_Text,Scoring_Method,Feature_Mapping
0,Q001,Interest,How interested are you in writing scripts or c...,NaN,NaN
1,Q002,Interest,How interested are you in analyzing large data...,NaN,NaN
2,Q003,Interest,How interested are you in finding vulnerabilit...,NaN,NaN
3,Q004,Interest,How interested are you in sketching out wirefr...,NaN,NaN
4,Q005,Interest,How interested are you in coordinating team me...,NaN,NaN
...,...,...,...,...,...
115,Q116,Education & Career Preferences,Do you prefer arguing cases in a courtroom or ...,NaN,NaN
116,Q117,Education & Career Preferences,How important is it for your work to directly ...,NaN,NaN
117,Q118,Education & Career Preferences,Are you comfortable defending clients or polic...,NaN,NaN
118,Q119,Education & Career Preferences,Do you prefer the fast pace of a corporate law...,NaN,NaN


In [89]:
# Step 70: Inspect remaining questions and answer formats

print("Questions Q121–Q123:")
display(
    questions_df[
        questions_df["Question_ID"].isin(["Q121", "Q122", "Q123"])
    ][
        ["Question_ID", "Section", "Question_Text",
         "Answer_Type", "Answer_Options", "Purpose"]
    ]
)

print("\nAnswer types by section:")
display(
    questions_df.groupby(["Section", "Answer_Type"], dropna=False)
    .size()
    .reset_index(name="Question_Count")
    .sort_values(["Section", "Answer_Type"])
)

print("\nScoring methods currently present:")
display(
    questions_df["Scoring_Method"]
    .value_counts(dropna=False)
    .rename_axis("Scoring_Method")
    .reset_index(name="Count")
)

print("\nFeature mappings currently present:")
display(
    questions_df["Feature_Mapping"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("Feature_Mapping")
    .reset_index(name="Count")
)

Questions Q121–Q123:


,Question_ID,Section,Question_Text,Answer_Type,Answer_Options,Purpose
120,Q121,Aptitude,Read the passage: 'Regular exercise can improv...,MCQ,"{""A"": ""Exercise only improves sleep quality."",...",Preliminary assessment of verbal comprehension...
121,Q122,Aptitude,"A square has vertices labeled clockwise A, B, ...",MCQ,"{""A"": ""A"", ""B"": ""B"", ""C"": ""C"", ""D"": ""D""}",Preliminary assessment of spatial reasoning.
122,Q123,Skills,How would you rate your ability to explain com...,Skill Scale (1-5 + N/A),"1-Beginner to 5-Expert, N/A",Collect a self-reported communication skill ra...



Answer types by section:


,Section,Answer_Type,Question_Count
0,Aptitude,MCQ,12
1,Education & Career Preferences,Dropdown,3
2,Education & Career Preferences,MCQ,77
3,Interest,Rating scale (1-5),10
4,Personality,Agreement Scale (1-5),10
5,Skills,Skill Scale (1-5 + N/A),11



Scoring methods currently present:


,Scoring_Method,Count
0,NaN,120
1,"Correct option: B; correct=1, incorrect=0; pro...",1
2,"Correct option: A; correct=1, incorrect=0; pro...",1
3,Convert a valid 1-5 rating to the model's 1-10...,1



Feature mappings currently present:


,Feature_Mapping,Count
0,NaN,120
1,aptitude_verbal,1
2,aptitude_spatial,1
3,skill_communication,1


In [90]:
# Step 71: Inspect core assessment questions

core_questions_df = questions_df[
    questions_df["Question_ID"].apply(
        lambda q: int(q[1:]) <= 40
    )
].copy()

display(
    core_questions_df[
        [
            "Question_ID",
            "Section",
            "Question_Text",
            "Answer_Type",
            "Answer_Options",
            "Scoring_Method",
            "Feature_Mapping",
            "Purpose"
        ]
    ].to_string(index=False)
)

'Question_ID     Section                                                                                                                                                                                                    Question_Text             Answer_Type                                                                                                                                          Answer_Options Scoring_Method Feature_Mapping                                                               Purpose\n       Q001    Interest                                                                                                                         How interested are you in writing scripts or code to automate complex, repetitive tasks?      Rating scale (1-5)                                                                                                   1 - Not interested at all to 5 - Extremely interested            NaN             NaN                       Measures interest in core 

In [91]:
# Step 72: Define proposed mappings for Q001–Q040

question_feature_mapping = {
    # Interest questions
    "Q001": ["interest_technology", "interest_problem_solving"],
    "Q002": ["interest_data"],
    "Q003": ["interest_technology", "interest_problem_solving"],
    "Q004": ["interest_creativity"],
    "Q005": ["interest_people"],
    "Q006": ["interest_helping"],
    "Q007": ["interest_people", "interest_helping"],
    "Q008": ["interest_creativity", "interest_data"],
    "Q009": ["interest_data", "interest_problem_solving"],
    "Q010": ["interest_technology", "interest_problem_solving"],

    # Aptitude questions
    "Q011": ["aptitude_numerical"],
    "Q012": ["aptitude_logical"],
    "Q013": ["aptitude_numerical", "aptitude_logical"],
    "Q014": ["aptitude_logical"],
    "Q015": ["aptitude_numerical", "aptitude_critical_thinking"],
    "Q016": ["aptitude_logical", "aptitude_critical_thinking"],
    "Q017": ["aptitude_numerical"],
    "Q018": ["aptitude_critical_thinking"],
    "Q019": ["aptitude_logical", "aptitude_critical_thinking"],
    "Q020": ["aptitude_numerical"],

    # Personality questions
    "Q021": ["trait_extraversion", "trait_agreeableness"],
    "Q022": ["trait_conscientiousness"],
    "Q023": ["trait_extraversion"],
    "Q024": ["trait_conscientiousness"],
    "Q025": ["trait_openness"],
    "Q026": ["trait_emotional_stability"],
    "Q027": ["trait_conscientiousness"],
    "Q028": ["trait_extraversion"],
    "Q029": ["trait_conscientiousness"],
    "Q030": ["trait_openness"],

    # Skills questions
    "Q031": ["skill_programming"],
    "Q032": ["skill_data_analysis"],
    "Q033": [],  # Design skill; no dedicated model feature
    "Q034": [],  # Finance skill; no dedicated model feature
    "Q035": ["skill_programming"],  # Proxy; validate before using
    "Q036": ["skill_leadership"],
    "Q037": [],  # Marketing skill; no dedicated model feature
    "Q038": [],  # Legal skill; no dedicated model feature
    "Q039": [],  # Engineering skill; no dedicated model feature
    "Q040": ["skill_research"],
}

# Validate mapping IDs and feature names
valid_question_ids = set(questions_df["Question_ID"])
valid_model_features = set(model_features)

assert set(question_feature_mapping).issubset(valid_question_ids)

invalid_features = {
    feature
    for features in question_feature_mapping.values()
    for feature in features
    if feature not in valid_model_features
}

print("Mapped questions:", len(question_feature_mapping))
print("Invalid model features:", invalid_features)

print("\nQuestions with no direct model-feature mapping:")
display(
    questions_df[
        questions_df["Question_ID"].isin(
            qid for qid, features in question_feature_mapping.items()
            if not features
        )
    ][["Question_ID", "Question_Text", "Purpose"]]
)

Mapped questions: 40
Invalid model features: set()

Questions with no direct model-feature mapping:


,Question_ID,Question_Text,Purpose
32,Q033,How would you rate your proficiency with desig...,Measures UI/UX and graphic design capability
33,Q034,How would you rate your ability to read financ...,Measures finance and accounting capability
36,Q037,How would you rate your ability to set up targ...,Measures digital marketing capability
37,Q038,How would you rate your ability to comprehend ...,Measures legal and policy capability
38,Q039,How would you rate your proficiency with core ...,Measures core engineering capability


In [92]:
# Step 73: Inspect all aptitude questions and answer options

aptitude_questions_df = questions_df[
    questions_df["Section"] == "Aptitude"
].copy()

display(
    aptitude_questions_df[
        [
            "Question_ID",
            "Question_Text",
            "Answer_Options",
            "Scoring_Method",
            "Feature_Mapping"
        ]
    ].to_string(index=False)
)

'Question_ID                                                                                                                                                                                                      Question_Text                                                                                                                                                                                 Answer_Options                                                                 Scoring_Method  Feature_Mapping\n       Q011                                         A project requires 3 developers to complete in 12 days. If the deadline is moved up and it must be done in 4 days, how many developers of equal skill are needed in total?                                                                                                                                                                        A) 6, B) 9, C) 12, D) 4                                                                           

In [93]:
# Step 74: Proposed aptitude answer key

aptitude_answer_key = {
    "Q011": "C",  # 3 * 12 / 4 = 9 developers; correct option is B, verify below
    "Q012": "D",
    "Q013": "C",  # Differences: +3, +4, +5, +6, next +7 => 40
    "Q014": "A",  # Each letter moves forward by 2
    "Q015": "C",  # Costs: 70% of 10 = 7; 80% of 12 = 9.6
    "Q016": "B",  # n iterations * log n inner loop
    "Q017": "B",  # 60 / 5 * 20 = 240 minutes = 4 hours
    "Q018": "A",
    "Q019": "C",
    "Q020": "C",
    "Q121": "B",
    "Q122": "A",
}

print("Proposed answer key:")
display(
    pd.DataFrame(
        aptitude_answer_key.items(),
        columns=["Question_ID", "Proposed_Correct_Option"]
    )
)

Proposed answer key:


,Question_ID,Proposed_Correct_Option
0,Q011,C
1,Q012,D
2,Q013,C
3,Q014,A
4,Q015,C
5,Q016,B
6,Q017,B
7,Q018,A
8,Q019,C
9,Q020,C


In [94]:
# Step 75: Correct Q011 and validate the answer key

aptitude_answer_key["Q011"] = "B"

assert aptitude_answer_key["Q011"] == "B"
assert len(aptitude_answer_key) == 12
assert set(aptitude_answer_key).issubset(
    set(questions_df["Question_ID"])
)

print("Q011 corrected:", aptitude_answer_key["Q011"])
print("Total keyed aptitude questions:", len(aptitude_answer_key))
print("All answer-key IDs exist in the questionnaire.")

display(
    pd.DataFrame(
        aptitude_answer_key.items(),
        columns=["Question_ID", "Correct_Option"]
    )
)

Q011 corrected: B
Total keyed aptitude questions: 12
All answer-key IDs exist in the questionnaire.


,Question_ID,Correct_Option
0,Q011,B
1,Q012,D
2,Q013,C
3,Q014,A
4,Q015,C
5,Q016,B
6,Q017,B
7,Q018,A
8,Q019,C
9,Q020,C


In [95]:
# Step 76: MCQ scoring function

def score_mcq_answer(question_id, student_answer):
    if question_id not in aptitude_answer_key:
        raise ValueError(f"No answer key available for {question_id}")

    if student_answer is None or pd.isna(student_answer):
        return None

    answer = str(student_answer).strip().upper()

    if answer not in {"A", "B", "C", "D"}:
        raise ValueError(
            f"Invalid answer '{student_answer}' for {question_id}. "
            "Expected A, B, C, or D."
        )

    return int(answer == aptitude_answer_key[question_id])


# Test correct, incorrect, and unanswered responses
test_answers = [
    ("Q011", "B"),
    ("Q011", "A"),
    ("Q011", None),
]

test_results = pd.DataFrame([
    {
        "Question_ID": question_id,
        "Student_Answer": answer,
        "Score": score_mcq_answer(question_id, answer)
    }
    for question_id, answer in test_answers
])

display(test_results)

,Question_ID,Student_Answer,Score
0,Q011,B,1.0
1,Q011,A,0.0
2,Q011,NaN,NaN


In [96]:
# Step 77: Group aptitude questions by model feature

aptitude_feature_mapping = {
    feature: [
        question_id
        for question_id, features in question_feature_mapping.items()
        if feature in features
    ]
    for feature in [
        "aptitude_logical",
        "aptitude_numerical",
        "aptitude_critical_thinking",
        "aptitude_verbal",
        "aptitude_spatial"
    ]
}

for feature, question_ids in aptitude_feature_mapping.items():
    print(f"{feature}: {question_ids}")
    print(f"Number of mapped questions: {len(question_ids)}\n")

aptitude_logical: ['Q012', 'Q013', 'Q014', 'Q016', 'Q019']
Number of mapped questions: 5

aptitude_numerical: ['Q011', 'Q013', 'Q015', 'Q017', 'Q020']
Number of mapped questions: 5

aptitude_critical_thinking: ['Q015', 'Q016', 'Q018', 'Q019']
Number of mapped questions: 4

aptitude_verbal: []
Number of mapped questions: 0

aptitude_spatial: []
Number of mapped questions: 0



In [97]:
# Step 78: Add the additional aptitude questions

question_feature_mapping["Q121"] = ["aptitude_verbal"]
question_feature_mapping["Q122"] = ["aptitude_spatial"]

# Rebuild aptitude feature groups
aptitude_feature_mapping = {
    feature: [
        question_id
        for question_id, features in question_feature_mapping.items()
        if feature in features
    ]
    for feature in [
        "aptitude_logical",
        "aptitude_numerical",
        "aptitude_critical_thinking",
        "aptitude_verbal",
        "aptitude_spatial"
    ]
}

for feature, question_ids in aptitude_feature_mapping.items():
    print(f"{feature}: {question_ids}")

aptitude_logical: ['Q012', 'Q013', 'Q014', 'Q016', 'Q019']
aptitude_numerical: ['Q011', 'Q013', 'Q015', 'Q017', 'Q020']
aptitude_critical_thinking: ['Q015', 'Q016', 'Q018', 'Q019']
aptitude_verbal: ['Q121']
aptitude_spatial: ['Q122']


In [98]:
# Step 79: Calculate aptitude feature scores on a 1–10 scale

def calculate_aptitude_feature_scores(student_answers):
    results = {}

    for feature, question_ids in aptitude_feature_mapping.items():
        scores = [
            score_mcq_answer(qid, student_answers.get(qid))
            for qid in question_ids
        ]

        valid_scores = [score for score in scores if score is not None]

        if not valid_scores:
            results[feature] = None
        else:
            accuracy = sum(valid_scores) / len(valid_scores)
            results[feature] = round(1 + accuracy * 9, 2)

    return results


# Test using sample answers only (not real student responses)
sample_aptitude_answers = {
    "Q011": "B",
    "Q012": "D",
    "Q013": "C",
    "Q014": "A",
    "Q015": "C",
    "Q016": "B",
    "Q017": "B",
    "Q018": "A",
    "Q019": "C",
    "Q020": "C",
    "Q121": "B",
    "Q122": "A"
}

sample_aptitude_scores = calculate_aptitude_feature_scores(
    sample_aptitude_answers
)

display(
    pd.DataFrame(
        sample_aptitude_scores.items(),
        columns=["Aptitude Feature", "Score (1–10)"]
    )
)

,Aptitude Feature,Score (1–10)
0,aptitude_logical,10.0
1,aptitude_numerical,10.0
2,aptitude_critical_thinking,10.0
3,aptitude_verbal,10.0
4,aptitude_spatial,10.0


In [99]:
# Step 80: Test partial and unanswered aptitude responses

test_aptitude_answers = {
    "Q011": "A",   # Incorrect
    "Q012": "D",   # Correct
    "Q013": None,  # Unanswered
    "Q014": "B",   # Incorrect
    "Q016": "B",   # Correct
    "Q019": None,  # Unanswered
    "Q121": None,  # Verbal unanswered
    "Q122": "A"    # Spatial correct
}

test_aptitude_scores = calculate_aptitude_feature_scores(
    test_aptitude_answers
)

display(
    pd.DataFrame(
        test_aptitude_scores.items(),
        columns=["Aptitude Feature", "Score (1–10)"]
    )
)

print("\nMissing feature scores:")
print([
    feature
    for feature, score in test_aptitude_scores.items()
    if score is None
])

,Aptitude Feature,Score (1–10)
0,aptitude_logical,7.0
1,aptitude_numerical,1.0
2,aptitude_critical_thinking,10.0
3,aptitude_verbal,NaN
4,aptitude_spatial,10.0



Missing feature scores:
['aptitude_verbal']


In [100]:
# Step 81: Add minimum-answer requirements for aptitude scores

def calculate_aptitude_feature_scores(student_answers):
    results = {}

    minimum_answers = {
        "aptitude_logical": 3,            # Out of 5 questions
        "aptitude_numerical": 3,          # Out of 5 questions
        "aptitude_critical_thinking": 3,  # Out of 4 questions
        "aptitude_verbal": 1,             # Only 1 mapped question currently
        "aptitude_spatial": 1             # Only 1 mapped question currently
    }

    for feature, question_ids in aptitude_feature_mapping.items():
        scores = [
            score_mcq_answer(qid, student_answers.get(qid))
            for qid in question_ids
        ]

        valid_scores = [score for score in scores if score is not None]

        if len(valid_scores) < minimum_answers[feature]:
            results[feature] = None
        else:
            accuracy = sum(valid_scores) / len(valid_scores)
            results[feature] = round(1 + accuracy * 9, 2)

    return results


# Test the improved scoring function
test_aptitude_scores = calculate_aptitude_feature_scores(
    test_aptitude_answers
)

display(
    pd.DataFrame(
        list(test_aptitude_scores.items()),
        columns=["Aptitude Feature", "Score (1–10)"]
    )
)

print("\nFeatures requiring more responses:")
print([
    feature
    for feature, score in test_aptitude_scores.items()
    if score is None
])

,Aptitude Feature,Score (1–10)
0,aptitude_logical,7.0
1,aptitude_numerical,NaN
2,aptitude_critical_thinking,NaN
3,aptitude_verbal,NaN
4,aptitude_spatial,10.0



Features requiring more responses:
['aptitude_numerical', 'aptitude_critical_thinking', 'aptitude_verbal']


In [102]:
# Step 82: Inspect the trained model's preprocessing steps

print("Model type:", type(model).__name__)

if hasattr(model, "named_steps"):
    print("\nPreprocessing pipeline steps:")
    for step_name, step_object in model.named_steps.items():
        print(f"{step_name}: {type(step_object).__name__}")
else:
    print("\nModel does not expose named preprocessing steps.")

print("\nModel expected features:")
print(model_features)

print("\nModel classes:")
print(career_labels)

Model type: VotingClassifier

Model does not expose named preprocessing steps.

Model expected features:
['age_group', 'education_level', 'experience_stage', 'region_type', 'interest_technology', 'interest_data', 'interest_problem_solving', 'interest_creativity', 'interest_people', 'interest_helping', 'aptitude_logical', 'aptitude_numerical', 'aptitude_verbal', 'aptitude_spatial', 'aptitude_critical_thinking', 'trait_openness', 'trait_conscientiousness', 'trait_extraversion', 'trait_agreeableness', 'trait_emotional_stability', 'skill_programming', 'skill_data_analysis', 'skill_research', 'skill_communication', 'skill_leadership']

Model classes:
['Business & Management', 'Creative Arts & Media', 'Cybersecurity', 'Data Science & Analytics', 'Digital Marketing', 'Education & Research', 'Engineering & Core Technology', 'Finance & Accounting', 'Healthcare & Life Sciences', 'Law & Public Service', 'Software Development', 'UI/UX Design']


In [103]:
print("Voting strategy:", model.voting)

for name, estimator in model.named_estimators_.items():
    print(f"\n{name}: {type(estimator).__name__}")

    if hasattr(estimator, "steps"):
        print("Pipeline steps:")
        for step_name, step_object in estimator.steps:
            print(f"  {step_name}: {type(step_object).__name__}")
    else:
        print("No pipeline detected.")

Voting strategy: soft

gnb: Pipeline
Pipeline steps:
  preprocessor: ColumnTransformer
  classifier: GaussianNB

lr: Pipeline
Pipeline steps:
  preprocessor: ColumnTransformer
  classifier: LogisticRegression

mlp: Pipeline
Pipeline steps:
  preprocessor: ColumnTransformer
  classifier: MLPClassifier

extra_trees: Pipeline
Pipeline steps:
  preprocessor: ColumnTransformer
  classifier: ExtraTreesClassifier


In [104]:
# Step 84: Inspect preprocessing configurations

for name, estimator in model.named_estimators_.items():
    print(f"\n{'=' * 15} {name} {'=' * 15}")

    preprocessor = estimator.named_steps["preprocessor"]

    for transformer_name, transformer, columns in preprocessor.transformers_:
        print(f"\nTransformer: {transformer_name}")
        print("Columns:", list(columns) if not isinstance(columns, str) else columns)
        print("Transformer type:", type(transformer).__name__)

        if hasattr(transformer, "steps"):
            for step_name, step_object in transformer.steps:
                print(f"  {step_name}: {type(step_object).__name__}")


=============== gnb ===============

Transformer: numeric
Columns: ['interest_technology', 'interest_data', 'interest_problem_solving', 'interest_creativity', 'interest_people', 'interest_helping', 'aptitude_logical', 'aptitude_numerical', 'aptitude_verbal', 'aptitude_spatial', 'aptitude_critical_thinking', 'trait_openness', 'trait_conscientiousness', 'trait_extraversion', 'trait_agreeableness', 'trait_emotional_stability', 'skill_programming', 'skill_data_analysis', 'skill_research', 'skill_communication', 'skill_leadership']
Transformer type: Pipeline
  imputer: SimpleImputer
  scaler: StandardScaler

Transformer: categorical
Columns: ['age_group', 'education_level', 'experience_stage', 'region_type']
Transformer type: Pipeline
  imputer: SimpleImputer
  encoder: OneHotEncoder

=============== lr ===============

Transformer: numeric
Columns: ['interest_technology', 'interest_data', 'interest_problem_solving', 'interest_creativity', 'interest_people', 'interest_helping', 'aptitude_l

In [105]:
# Step 85: Inspect imputer strategies and fill values

for name, estimator in model.named_estimators_.items():
    print(f"\n{'=' * 15} {name} {'=' * 15}")

    preprocessor = estimator.named_steps["preprocessor"]

    for transformer_name, transformer, columns in preprocessor.transformers_:
        if not hasattr(transformer, "steps"):
            continue

        imputer = transformer.named_steps.get("imputer")

        if imputer is not None:
            print(f"\n{transformer_name} imputer:")
            print("Strategy:", imputer.strategy)
            print("Fill value:", imputer.fill_value)
            print("Missing-value indicator:", imputer.add_indicator)


=============== gnb ===============

numeric imputer:
Strategy: median
Fill value: None
Missing-value indicator: False

categorical imputer:
Strategy: most_frequent
Fill value: None
Missing-value indicator: False

=============== lr ===============

numeric imputer:
Strategy: median
Fill value: None
Missing-value indicator: False

categorical imputer:
Strategy: most_frequent
Fill value: None
Missing-value indicator: False

=============== mlp ===============

numeric imputer:
Strategy: median
Fill value: None
Missing-value indicator: False

categorical imputer:
Strategy: most_frequent
Fill value: None
Missing-value indicator: False

=============== extra_trees ===============

numeric imputer:
Strategy: median
Fill value: None
Missing-value indicator: False

categorical imputer:
Strategy: most_frequent
Fill value: None
Missing-value indicator: False


In [106]:
# Step 86: Test prediction with missing aptitude scores

sample_profile = train_df[model_features].iloc[[0]].copy()

sample_profile.loc[:, [
    "aptitude_logical",
    "aptitude_numerical",
    "aptitude_verbal",
    "aptitude_spatial",
    "aptitude_critical_thinking"
]] = float("nan")

print("Missing values in sample profile:")
print(sample_profile.isna().sum().sum())

prediction = model.predict(sample_profile)[0]
probabilities = model.predict_proba(sample_profile)[0]

print("\nPredicted career domain:", prediction)

print("\nTop 3 predicted career domains:")
top_indices = probabilities.argsort()[::-1][:3]

for index in top_indices:
    print(
        f"{model.classes_[index]}: "
        f"{probabilities[index] * 100:.2f}%"
    )

Missing values in sample profile:
5

Predicted career domain: Engineering & Core Technology

Top 3 predicted career domains:
Engineering & Core Technology: 98.07%
Data Science & Analytics: 0.73%
Software Development: 0.68%


In [107]:
# Step 87: Check questionnaire coverage of model features

mapped_features = {
    feature
    for features in question_feature_mapping.values()
    for feature in features
}

mapped_features.update([
    "aptitude_verbal",
    "aptitude_spatial"
])

coverage_df = pd.DataFrame({
    "Model Feature": model_features,
    "Mapped to Questionnaire": [
        feature in mapped_features
        for feature in model_features
    ]
})

display(coverage_df)

print("\nTotal model features:", len(model_features))
print("Mapped features:", len(mapped_features))
print("Unmapped features:", len(model_features) - len(mapped_features))

print("\nFeatures not mapped to questionnaire:")
print(coverage_df.loc[
    ~coverage_df["Mapped to Questionnaire"],
    "Model Feature"
].tolist())

,Model Feature,Mapped to Questionnaire
0,age_group,False
1,education_level,False
2,experience_stage,False
3,region_type,False
4,interest_technology,True
5,interest_data,True
6,interest_problem_solving,True
7,interest_creativity,True
8,interest_people,True
9,interest_helping,True



Total model features: 25
Mapped features: 20
Unmapped features: 5

Features not mapped to questionnaire:
['age_group', 'education_level', 'experience_stage', 'region_type', 'skill_communication']


In [108]:
# Step 88: Map Q123 to the communication skill feature

question_feature_mapping["Q123"] = ["skill_communication"]

print("Q123 details:")
display(
    questions_df.loc[
        questions_df["Question_ID"] == "Q123",
        ["Question_ID", "Section", "Question_Text",
         "Answer_Type", "Scoring_Method", "Feature_Mapping"]
    ]
)

print("\nQ123 feature mapping:")
print(question_feature_mapping["Q123"])

Q123 details:


,Question_ID,Section,Question_Text,Answer_Type,Scoring_Method,Feature_Mapping
122,Q123,Skills,How would you rate your ability to explain com...,Skill Scale (1-5 + N/A),Convert a valid 1-5 rating to the model's 1-10...,skill_communication



Q123 feature mapping:
['skill_communication']


In [109]:
# Step 89: Convert Q123 rating from 1–5 to 1–10

def score_communication_skill(answer):
    if answer is None or pd.isna(answer):
        return None

    answer = str(answer).strip().upper()

    if answer in {"N/A", "NA", "NOT APPLICABLE"}:
        return None

    try:
        rating = float(answer)
    except ValueError:
        raise ValueError("Enter a rating from 1 to 5, or N/A.")

    if rating not in [1, 2, 3, 4, 5]:
        raise ValueError("Rating must be an integer from 1 to 5.")

    return round(rating * 2, 2)


# Test all valid ratings
for rating in [1, 2, 3, 4, 5, "N/A"]:
    print(f"Response: {rating} -> Score: {score_communication_skill(rating)}")

Response: 1 -> Score: 2.0
Response: 2 -> Score: 4.0
Response: 3 -> Score: 6.0
Response: 4 -> Score: 8.0
Response: 5 -> Score: 10.0
Response: N/A -> Score: None


In [110]:
# Step 90: Inspect education and demographic questions

education_questions = questions_df[
    questions_df["Section"].astype(str).str.contains(
        "Education|Career|Demographic", case=False, na=False
    )
]

display(
    education_questions[
        [
            "Question_ID",
            "Section",
            "Question_Text",
            "Answer_Type",
            "Answer_Options",
            "Feature_Mapping"
        ]
    ].head(25)
)

print("\nTotal matching questions:", len(education_questions))

,Question_ID,Section,Question_Text,Answer_Type,Answer_Options,Feature_Mapping
40,Q041,Education & Career Preferences,What is your current or most recently complete...,Dropdown,"High School, Diploma, Bachelor's, Master's, Ph...",NaN
41,Q042,Education & Career Preferences,Which broad academic field does your primary d...,Dropdown,"Engineering, Business, Arts, Sciences, Medicin...",NaN
42,Q043,Education & Career Preferences,What is your cumulative academic performance/G...,MCQ,"Top 10%, Top 25%, Average, Below Average, Pref...",NaN
43,Q044,Education & Career Preferences,Have you completed any specialized bootcamps o...,MCQ,"Yes - Technical, Yes - Non-Technical, No",NaN
44,Q045,Education & Career Preferences,How many total years of formal education/train...,MCQ,"<1 year, 1-3 years, 3-5 years, 5+ years",NaN
45,Q046,Education & Career Preferences,How many years of full-time professional work ...,MCQ,"0, <1, 1-3, 3-5, 5+",NaN
46,Q047,Education & Career Preferences,"Have you ever completed an internship, apprent...",MCQ,"Yes, No",NaN
47,Q048,Education & Career Preferences,Have you ever held a leadership or management ...,MCQ,"Yes, No",NaN
48,Q049,Education & Career Preferences,Do you have experience working as a freelancer...,MCQ,"Yes, No",NaN
49,Q050,Education & Career Preferences,What is the largest team size you have consist...,MCQ,"Solo, 2-5, 6-15, 15+",NaN



Total matching questions: 80


In [111]:
# Step 91: Inspect candidate demographic questions

question_ids_to_check = ["Q041", "Q046"]

display(
    questions_df.loc[
        questions_df["Question_ID"].isin(question_ids_to_check),
        [
            "Question_ID",
            "Question_Text",
            "Answer_Type",
            "Answer_Options",
            "Scoring_Method"
        ]
    ].to_string(index=False)
)

print("\nQuestions mentioning age or location:")
age_region_questions = questions_df[
    questions_df["Question_Text"].str.contains(
        r"\bage\b|location|region|reside|district|city",
        case=False,
        na=False
    )
]

display(
    age_region_questions[
        ["Question_ID", "Question_Text", "Answer_Type", "Answer_Options"]
    ].to_string(index=False)
)

"Question_ID                                                                   Question_Text Answer_Type                                         Answer_Options Scoring_Method\n       Q041             What is your current or most recently completed level of education?    Dropdown High School, Diploma, Bachelor's, Master's, PhD, Other            NaN\n       Q046 How many years of full-time professional work experience do you currently have?         MCQ                                    0, <1, 1-3, 3-5, 5+            NaN"


Questions mentioning age or location:


'Question_ID                                                                                Question_Text Answer_Type                                               Answer_Options\n       Q053 Are you willing to relocate to a different city or country for the right career opportunity?         MCQ                               Yes, No, Only within my region\n       Q091             What age group do you most prefer teaching or designing educational content for?         MCQ Early Childhood, K-12, Higher Ed/University, Adult/Corporate'

In [112]:
# Step 92: Map education and work experience answers

def map_education_level(answer):
    if answer is None or pd.isna(answer):
        return None

    mapping = {
        "HIGH SCHOOL": "High School",
        "DIPLOMA": "Diploma",
        "BACHELOR'S": "Bachelor's",
        "BACHELORS": "Bachelor's",
        "MASTER'S": "Master's",
        "MASTERS": "Master's",
        "PHD": "PhD",
        "OTHER": "Other"
    }

    key = str(answer).strip().upper()

    if key not in mapping:
        raise ValueError(f"Unrecognized education level: {answer}")

    return mapping[key]


def map_experience_stage(answer):
    if answer is None or pd.isna(answer):
        return None

    mapping = {
        "0": "No Experience",
        "<1": "Less than 1 year",
        "1-3": "1-3 years",
        "3-5": "3-5 years",
        "5+": "5+ years"
    }

    key = str(answer).strip()

    if key not in mapping:
        raise ValueError(f"Unrecognized experience value: {answer}")

    return mapping[key]


# Test the mappings
for answer in ["Bachelor's", "Diploma", "PhD"]:
    print(f"Education: {answer} -> {map_education_level(answer)}")

for answer in ["0", "<1", "1-3", "3-5", "5+"]:
    print(f"Experience: {answer} -> {map_experience_stage(answer)}")

Education: Bachelor's -> Bachelor's
Education: Diploma -> Diploma
Education: PhD -> PhD
Experience: 0 -> No Experience
Experience: <1 -> Less than 1 year
Experience: 1-3 -> 1-3 years
Experience: 3-5 -> 3-5 years
Experience: 5+ -> 5+ years


In [113]:
# Step 93: Inspect actual categorical values in training data

categorical_features = [
    "age_group",
    "education_level",
    "experience_stage",
    "region_type"
]

for feature in categorical_features:
    print(f"\n{feature}")
    print(train_df[feature].value_counts(dropna=False).to_string())


age_group
age_group
31+      494
23-30    494
15-18    487
19-22    445

education_level
education_level
B.Tech IT                  338
MBA                        123
BBA                        100
B.Com                       97
BCA                         95
MCA                         86
B.Tech Computer Science     82
BA Economics                73
Diploma Design              70
BA                          63
B.Des                       63
Diploma IT                  59
BFA                         56
B.Sc Mathematics            52
B.Sc Statistics             47
B.Pharm                     40
B.Sc Nursing                38
MA                          37
M.Sc                        35
B.Sc Biology                34
LLB                         33
Diploma Engineering         32
BA Media                    32
B.Tech Electrical           32
B.Tech Civil                31
B.Ed                        31
MBBS                        31
B.Tech Mechanical           31
PhD                       

In [114]:
# Step 94: Map Q046 responses to the model's actual categories

def map_experience_stage(answer):
    if answer is None or pd.isna(answer):
        return None

    mapping = {
        "0": "Student",
        "<1": "Entry-level",
        "1-3": "1-3 years",
        "3-5": "4+ years",
        "5+": "4+ years"
    }

    key = str(answer).strip()

    if key not in mapping:
        raise ValueError(f"Unrecognized experience value: {answer}")

    return mapping[key]


# Test against the model's training categories
valid_experience_categories = set(
    train_df["experience_stage"].dropna().unique()
)

for answer in ["0", "<1", "1-3", "3-5", "5+"]:
    result = map_experience_stage(answer)
    print(
        f"{answer:>3} -> {result} "
        f"| Valid training category: {result in valid_experience_categories}"
    )

  0 -> Student | Valid training category: True
 <1 -> Entry-level | Valid training category: True
1-3 -> 1-3 years | Valid training category: True
3-5 -> 4+ years | Valid training category: True
 5+ -> 4+ years | Valid training category: True


In [115]:
# Step 95: Map broad education responses to training categories

def map_education_level(answer):
    if answer is None or pd.isna(answer):
        return None

    key = str(answer).strip().upper()

    mapping = {
        "HIGH SCHOOL": "B.Tech IT",
        "DIPLOMA": "Diploma IT",
        "BACHELOR'S": "B.Tech IT",
        "BACHELORS": "B.Tech IT",
        "MASTER'S": "MCA",
        "MASTERS": "MCA",
        "PHD": "PhD",
        "OTHER": None
    }

    if key not in mapping:
        raise ValueError(f"Unrecognized education level: {answer}")

    return mapping[key]


valid_education_categories = set(
    train_df["education_level"].dropna().unique()
)

for answer in [
    "High School", "Diploma", "Bachelor's",
    "Master's", "PhD", "Other"
]:
    result = map_education_level(answer)
    print(
        f"{answer:>12} -> {result} "
        f"| Valid training category: "
        f"{result in valid_education_categories if result else 'Missing'}"
    )

 High School -> B.Tech IT | Valid training category: True
     Diploma -> Diploma IT | Valid training category: True
  Bachelor's -> B.Tech IT | Valid training category: True
    Master's -> MCA | Valid training category: True
         PhD -> PhD | Valid training category: True
       Other -> None | Valid training category: Missing


In [116]:
# Step 96: Inspect education categories and their career labels

education_career_summary = (
    train_df.groupby(
        ["education_level", "recommended_career_domain"],
        observed=True
    )
    .size()
    .reset_index(name="Count")
    .sort_values(["education_level", "Count"], ascending=[True, False])
)

display(education_career_summary.head(30))

print("\nUnique education categories:", train_df["education_level"].nunique())
print("Unique career domains:", train_df["recommended_career_domain"].nunique())

,education_level,recommended_career_domain,Count
1,B.Com,Digital Marketing,33
2,B.Com,Finance & Accounting,33
0,B.Com,Business & Management,31
3,B.Des,Creative Arts & Media,43
4,B.Des,UI/UX Design,20
5,B.Ed,Education & Research,31
6,B.Pharm,Healthcare & Life Sciences,40
7,B.Sc Biology,Healthcare & Life Sciences,34
8,B.Sc Mathematics,Data Science & Analytics,31
9,B.Sc Mathematics,Finance & Accounting,21



Unique education categories: 31
Unique career domains: 12


In [117]:
# Step 97: Measure career-domain distribution by education

education_target_counts = pd.crosstab(
    train_df["education_level"],
    train_df["recommended_career_domain"]
)

education_target_percent = (
    pd.crosstab(
        train_df["education_level"],
        train_df["recommended_career_domain"],
        normalize="index"
    ) * 100
).round(1)

print("Counts for each education and career combination:")
display(education_target_counts)

print("\nCareer distribution within each education category (%):")
display(education_target_percent)

print("\nMost common career domain for each education category:")
print(education_target_counts.idxmax(axis=1).to_string())

Counts for each education and career combination:


recommended_career_domain,Business & Management,Creative Arts & Media,Cybersecurity,Data Science & Analytics,Digital Marketing,Education & Research,Engineering & Core Technology,Finance & Accounting,Healthcare & Life Sciences,Law & Public Service,Software Development,UI/UX Design
education_level,,,,,,,,,,,,
B.Com,31,0,0,0,33,0,0,33,0,0,0,0
B.Des,0,43,0,0,0,0,0,0,0,0,0,20
B.Ed,0,0,0,0,0,31,0,0,0,0,0,0
B.Pharm,0,0,0,0,0,0,0,0,40,0,0,0
B.Sc Biology,0,0,0,0,0,0,0,0,34,0,0,0
B.Sc Mathematics,0,0,0,31,0,0,0,21,0,0,0,0
B.Sc Nursing,0,0,0,0,0,0,0,0,38,0,0,0
B.Sc Statistics,0,0,0,47,0,0,0,0,0,0,0,0
B.Tech Biotechnology,0,0,0,0,0,0,0,0,23,0,0,0



Career distribution within each education category (%):


recommended_career_domain,Business & Management,Creative Arts & Media,Cybersecurity,Data Science & Analytics,Digital Marketing,Education & Research,Engineering & Core Technology,Finance & Accounting,Healthcare & Life Sciences,Law & Public Service,Software Development,UI/UX Design
education_level,,,,,,,,,,,,
B.Com,32.0,0.0,0.0,0.0,34.0,0.0,0.0,34.0,0.0,0.0,0.0,0.0
B.Des,0.0,68.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31.7
B.Ed,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
B.Pharm,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0
B.Sc Biology,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0
B.Sc Mathematics,0.0,0.0,0.0,59.6,0.0,0.0,0.0,40.4,0.0,0.0,0.0,0.0
B.Sc Nursing,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0
B.Sc Statistics,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
B.Tech Biotechnology,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0



Most common career domain for each education category:
education_level
B.Com                                  Digital Marketing
B.Des                              Creative Arts & Media
B.Ed                                Education & Research
B.Pharm                       Healthcare & Life Sciences
B.Sc Biology                  Healthcare & Life Sciences
B.Sc Mathematics                Data Science & Analytics
B.Sc Nursing                  Healthcare & Life Sciences
B.Sc Statistics                 Data Science & Analytics
B.Tech Biotechnology          Healthcare & Life Sciences
B.Tech Civil               Engineering & Core Technology
B.Tech Computer Science             Software Development
B.Tech Electrical          Engineering & Core Technology
B.Tech IT                                   UI/UX Design
B.Tech Mechanical          Engineering & Core Technology
BA                                  Law & Public Service
BA Economics                        Finance & Accounting
BA Media        

In [118]:
# Step 98: Compare predictions with and without education

X_test = test_df[model_features].copy()

# Original predictions
predictions_with_education = model.predict(X_test)

# Hide education to let the trained imputer fill missing values
X_without_education = X_test.copy()
X_without_education["education_level"] = None

predictions_without_education = model.predict(X_without_education)

# Compare results
comparison = pd.DataFrame({
    "Actual Career": test_df[target_column].values,
    "Prediction With Education": predictions_with_education,
    "Prediction Without Education": predictions_without_education
})

agreement = (
    comparison["Prediction With Education"]
    == comparison["Prediction Without Education"]
).mean() * 100

print(f"Prediction agreement: {agreement:.2f}%")
print(
    "Predictions changed after hiding education:",
    (comparison["Prediction With Education"]
     != comparison["Prediction Without Education"]).sum(),
    "out of", len(comparison)
)

display(comparison.head(15))

Prediction agreement: 72.50%
Predictions changed after hiding education: 132 out of 480


,Actual Career,Prediction With Education,Prediction Without Education
0,Healthcare & Life Sciences,Healthcare & Life Sciences,Law & Public Service
1,Software Development,Software Development,Software Development
2,Digital Marketing,Digital Marketing,Cybersecurity
3,Engineering & Core Technology,Engineering & Core Technology,Engineering & Core Technology
4,Digital Marketing,Digital Marketing,Digital Marketing
5,Software Development,Software Development,Software Development
6,Law & Public Service,Law & Public Service,Law & Public Service
7,Business & Management,Business & Management,Business & Management
8,Cybersecurity,Engineering & Core Technology,Finance & Accounting
9,Creative Arts & Media,UI/UX Design,Creative Arts & Media


In [119]:
# Step 99: Compare test accuracy with and without education

from sklearn.metrics import accuracy_score, f1_score

actual = test_df[target_column]

accuracy_with = accuracy_score(
    actual, predictions_with_education
)

accuracy_without = accuracy_score(
    actual, predictions_without_education
)

f1_with = f1_score(
    actual, predictions_with_education,
    average="macro", zero_division=0
)

f1_without = f1_score(
    actual, predictions_without_education,
    average="macro", zero_division=0
)

results = pd.DataFrame({
    "Evaluation": ["With Education", "Without Education"],
    "Accuracy (%)": [
        round(accuracy_with * 100, 2),
        round(accuracy_without * 100, 2)
    ],
    "Macro F1 (%)": [
        round(f1_with * 100, 2),
        round(f1_without * 100, 2)
    ]
})

display(results)

,Evaluation,Accuracy (%),Macro F1 (%)
0,With Education,83.33,83.14
1,Without Education,63.54,63.14


In [120]:
# Step 100: Inspect model and questionnaire education options

print("Education categories expected by the model:")
print(sorted(train_df["education_level"].dropna().unique()))

print("\nEducation question from the questionnaire:")
display(
    questions_df.loc[
        questions_df["Question ID"] == "Q041",
        ["Question ID", "Question Text", "Answer Options"]
    ]
)

Education categories expected by the model:
['B.Com', 'B.Des', 'B.Ed', 'B.Pharm', 'B.Sc Biology', 'B.Sc Mathematics', 'B.Sc Nursing', 'B.Sc Statistics', 'B.Tech Biotechnology', 'B.Tech Civil', 'B.Tech Computer Science', 'B.Tech Electrical', 'B.Tech IT', 'B.Tech Mechanical', 'BA', 'BA Economics', 'BA Media', 'BA Political Science', 'BBA', 'BCA', 'BFA', 'Diploma Design', 'Diploma Engineering', 'Diploma IT', 'LLB', 'M.Sc', 'MA', 'MBA', 'MBBS', 'MCA', 'PhD']

Education question from the questionnaire:


KeyError: 'Question ID'

In [121]:
# Step 101: Identify the actual questionnaire column names

print("Column names:")
print(questions_df.columns.tolist())

print("\nFirst five rows:")
display(questions_df.head())

print("\nRows containing Q041:")
display(
    questions_df[
        questions_df.astype(str)
        .apply(lambda column: column.str.contains("Q041", case=False, na=False))
        .any(axis=1)
    ]
)

Column names:
['Question_ID', 'Section', 'Question_Text', 'Answer_Type', 'Answer_Options', 'Scoring_Method', 'Feature_Mapping', 'Purpose']

First five rows:


,Question_ID,Section,Question_Text,Answer_Type,Answer_Options,Scoring_Method,Feature_Mapping,Purpose
0,Q001,Interest,How interested are you in writing scripts or c...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in core programming and logic
1,Q002,Interest,How interested are you in analyzing large data...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in data science and statisti...
2,Q003,Interest,How interested are you in finding vulnerabilit...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in cybersecurity and risk mi...
3,Q004,Interest,How interested are you in sketching out wirefr...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in UI/UX and user-centric de...
4,Q005,Interest,How interested are you in coordinating team me...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN,Measures interest in business management and l...



Rows containing Q041:


,Question_ID,Section,Question_Text,Answer_Type,Answer_Options,Scoring_Method,Feature_Mapping,Purpose
40,Q041,Education & Career Preferences,What is your current or most recently complete...,Dropdown,"High School, Diploma, Bachelor's, Master's, Ph...",NaN,NaN,General Education


In [122]:
# Step 102: Find questions related to education and qualifications

education_questions = questions_df[
    questions_df["Question_Text"].str.contains(
        r"education|qualification|degree|diploma|course|studying|completed|major|stream",
        case=False,
        na=False,
        regex=True
    )
]

display(
    education_questions[
        ["Question_ID", "Question_Text", "Answer_Type", "Answer_Options"]
    ].to_string(index=False)
)

'Question_ID                                                                                                                                                                    Question_Text        Answer_Type                                                   Answer_Options\n       Q006                                                                       How interested are you in studying human anatomy, diseases, or working directly to improve patient health? Rating scale (1-5)            1 - Not interested at all to 5 - Extremely interested\n       Q041                                                                                                              What is your current or most recently completed level of education?           Dropdown           High School, Diploma, Bachelor\'s, Master\'s, PhD, Other\n       Q042                                                                                                 Which broad academic field does your primary degree or certificatio

In [123]:
# Step 103: Count each education qualification in the training dataset

education_counts = (
    train_df["education_level"]
    .value_counts()
    .rename_axis("Qualification")
    .reset_index(name="Student Count")
)

education_counts["Percentage (%)"] = (
    education_counts["Student Count"]
    / len(train_df) * 100
).round(2)

display(education_counts)

,Qualification,Student Count,Percentage (%)
0,B.Tech IT,338,17.60
1,MBA,123,6.41
2,BBA,100,5.21
3,B.Com,97,5.05
4,BCA,95,4.95
5,MCA,86,4.48
6,B.Tech Computer Science,82,4.27
7,BA Economics,73,3.80
8,Diploma Design,70,3.65
9,BA,63,3.28


In [124]:
# Step 104: Prepare supported qualifications for the questionnaire

supported_qualifications = sorted(
    train_df["education_level"].dropna().unique().tolist()
)

print(f"Number of supported qualifications: {len(supported_qualifications)}")
print("\nOptions for the exact qualification dropdown:")

for qualification in supported_qualifications:
    print(f"- {qualification}")

print("\nAdditional options to handle:")
print("- Other / Not listed")
print("- Not yet enrolled / No formal qualification")

Number of supported qualifications: 31

Options for the exact qualification dropdown:
- B.Com
- B.Des
- B.Ed
- B.Pharm
- B.Sc Biology
- B.Sc Mathematics
- B.Sc Nursing
- B.Sc Statistics
- B.Tech Biotechnology
- B.Tech Civil
- B.Tech Computer Science
- B.Tech Electrical
- B.Tech IT
- B.Tech Mechanical
- BA
- BA Economics
- BA Media
- BA Political Science
- BBA
- BCA
- BFA
- Diploma Design
- Diploma Engineering
- Diploma IT
- LLB
- M.Sc
- MA
- MBA
- MBBS
- MCA
- PhD

Additional options to handle:
- Other / Not listed
- Not yet enrolled / No formal qualification


In [125]:
# Step 105: Validate education qualification input

supported_qualifications = set(
    train_df["education_level"].dropna().unique()
)

def validate_education_qualification(answer):
    if answer is None or pd.isna(answer):
        return None

    answer = str(answer).strip()

    if answer in supported_qualifications:
        return answer

    if answer in {
        "Other / Not listed",
        "Not yet enrolled / No formal qualification"
    }:
        return answer

    raise ValueError(
        f"Unsupported qualification: {answer}. "
        "Choose a listed qualification or a supported additional option."
    )


# Test the validation function
test_answers = [
    "B.Tech IT",
    "BCA",
    "Other / Not listed",
    "Not yet enrolled / No formal qualification",
    None
]

for answer in test_answers:
    print(f"{answer!r} -> {validate_education_qualification(answer)!r}")

'B.Tech IT' -> 'B.Tech IT'
'BCA' -> 'BCA'
'Other / Not listed' -> 'Other / Not listed'
'Not yet enrolled / No formal qualification' -> 'Not yet enrolled / No formal qualification'
None -> None


In [126]:
# Step 106: Inspect demographic feature categories

demographic_features = [
    "education_level",
    "age_group",
    "region_type",
    "experience_stage"
]

for feature in demographic_features:
    print(f"\n{feature}:")
    print(train_df[feature].value_counts(dropna=False).to_string())


education_level:
education_level
B.Tech IT                  338
MBA                        123
BBA                        100
B.Com                       97
BCA                         95
MCA                         86
B.Tech Computer Science     82
BA Economics                73
Diploma Design              70
BA                          63
B.Des                       63
Diploma IT                  59
BFA                         56
B.Sc Mathematics            52
B.Sc Statistics             47
B.Pharm                     40
B.Sc Nursing                38
MA                          37
M.Sc                        35
B.Sc Biology                34
LLB                         33
Diploma Engineering         32
BA Media                    32
B.Tech Electrical           32
B.Tech Civil                31
B.Ed                        31
MBBS                        31
B.Tech Mechanical           31
PhD                         30
BA Political Science        26
B.Tech Biotechnology        23

age_

In [127]:
# Step 107: Find questions about the respondent's age and location

patterns = r"\b(your age|how old are you|age group|where do you live|residence|living area|urban|rural|semi-urban)\b"

demographic_questions = questions_df[
    questions_df["Question_Text"].str.contains(
        patterns,
        case=False,
        na=False,
        regex=True
    )
]

display(
    demographic_questions[
        ["Question_ID", "Question_Text", "Answer_Type", "Answer_Options"]
    ].to_string(index=False)
)

C:\Users\susha\AppData\Local\Temp\ipykernel_19008\2434613604.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  questions_df["Question_Text"].str.contains(


'Question_ID                                                                    Question_Text Answer_Type                                               Answer_Options\n       Q091 What age group do you most prefer teaching or designing educational content for?         MCQ Early Childhood, K-12, Higher Ed/University, Adult/Corporate'

In [128]:
# Step 108: Demographic questions + safe mapping + validation

from pathlib import Path
import pandas as pd

# Load training data if it is not currently available
if "train_df" not in globals():
    train_path = Path.cwd() / "career_guidance_train.csv"
    if not train_path.exists():
        raise FileNotFoundError(
            f"Training CSV not found at {train_path}. "
            "Open the notebook from the project dataset directory."
        )
    train_df = pd.read_csv(train_path)

# Define new questions in memory; do not overwrite the Excel file
demographic_questions = pd.DataFrame([
    {
        "Question_ID": "Q124",
        "Section": "Student Profile",
        "Question_Text": "What is your age group?",
        "Answer_Type": "Dropdown",
        "Answer_Options": "15-18, 19-22, 23-30, 31+",
        "Feature_Mapping": "age_group",
        "Purpose": "Collect the student's own age group."
    },
    {
        "Question_ID": "Q125",
        "Section": "Student Profile",
        "Question_Text": "Which best describes your residential area?",
        "Answer_Type": "Dropdown",
        "Answer_Options": "Urban, Rural, Semi-urban, Prefer not to say",
        "Feature_Mapping": "region_type",
        "Purpose": "Collect location type; allow missing information."
    },
    {
        "Question_ID": "Q126",
        "Section": "Student Profile",
        "Question_Text": "Which best describes your current education/work stage?",
        "Answer_Type": "Dropdown",
        "Answer_Options": (
            "Currently studying, Recently graduated/seeking first job, "
            "Working for 1-3 years, Working for 4+ years, "
            "Other / Prefer not to say"
        ),
        "Feature_Mapping": "experience_stage",
        "Purpose": "Collect the student's current stage."
    }
])

allowed_age_groups = set(train_df["age_group"].dropna().unique())
allowed_regions = set(train_df["region_type"].dropna().unique())
allowed_experience_stages = set(
    train_df["experience_stage"].dropna().unique()
)

def map_age_group(answer):
    if answer is None or pd.isna(answer) or str(answer).strip() == "":
        return None
    answer = str(answer).strip().replace("–", "-")
    if answer not in allowed_age_groups:
        raise ValueError(f"Unsupported age group: {answer}")
    return answer

def map_region_type(answer):
    if answer is None or pd.isna(answer):
        return None
    answer = str(answer).strip()
    if answer in {"", "Prefer not to say"}:
        return None
    if answer not in allowed_regions:
        raise ValueError(f"Unsupported region type: {answer}")
    return answer

def map_experience_stage(answer):
    if answer is None or pd.isna(answer):
        return None

    mapping = {
        "Currently studying": "Student",
        "Recently graduated/seeking first job": "Entry-level",
        "Working for 1-3 years": "1-3 years",
        "Working for 4+ years": "4+ years",
        "Other / Prefer not to say": None
    }
    answer = str(answer).strip()

    if answer not in mapping:
        raise ValueError(f"Unsupported education/work stage: {answer}")

    result = mapping[answer]
    if result is not None and result not in allowed_experience_stages:
        raise ValueError(f"Mapped category is unsupported: {result}")
    return result

# Validation tests
assert map_age_group("15-18") in allowed_age_groups
assert map_age_group("23-30") in allowed_age_groups
assert map_region_type("Urban") in allowed_regions
assert map_region_type("Prefer not to say") is None
assert map_experience_stage("Currently studying") == "Student"
assert map_experience_stage("Working for 1-3 years") == "1-3 years"
assert map_experience_stage("Other / Prefer not to say") is None
assert map_age_group(None) is None

print("Demographic questions created:", len(demographic_questions))
display(demographic_questions)
print("\nAll mapping tests passed.")
print("Age groups:", sorted(allowed_age_groups))
print("Regions:", sorted(allowed_regions))
print("Experience stages:", sorted(allowed_experience_stages))

Demographic questions created: 3


,Question_ID,Section,Question_Text,Answer_Type,Answer_Options,Feature_Mapping,Purpose
0,Q124,Student Profile,What is your age group?,Dropdown,"15-18, 19-22, 23-30, 31+",age_group,Collect the student's own age group.
1,Q125,Student Profile,Which best describes your residential area?,Dropdown,"Urban, Rural, Semi-urban, Prefer not to say",region_type,Collect location type; allow missing information.
2,Q126,Student Profile,Which best describes your current education/wo...,Dropdown,"Currently studying, Recently graduated/seeking...",experience_stage,Collect the student's current stage.



All mapping tests passed.
Age groups: ['15-18', '19-22', '23-30', '31+']
Regions: ['Rural', 'Semi-urban', 'Urban']
Experience stages: ['1-3 years', '4+ years', 'Entry-level', 'Student']


In [129]:
# Step 1: Verify model, dataset, features, and Streamlit

from pathlib import Path
import importlib.util
import pandas as pd
import joblib

# Locate the project directory
project_dir = Path.cwd()

# If the notebook is running from another directory, locate the dataset folder
if not (project_dir / "career_guidance_train.csv").exists():
    candidates = list(Path(
        r"C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT"
    ).rglob("career_guidance_train.csv"))

    if not candidates:
        raise FileNotFoundError("career_guidance_train.csv not found.")

    project_dir = candidates[0].parent

train_path = project_dir / "career_guidance_train.csv"
test_path = project_dir / "career_guidance_test.csv"
model_path = project_dir / "artifacts" / "career_guidance_model.joblib"

print("Project directory:", project_dir)
print("Training CSV exists:", train_path.exists())
print("Test CSV exists:", test_path.exists())
print("Saved model exists:", model_path.exists())
print("Streamlit installed:",
      importlib.util.find_spec("streamlit") is not None)

# Load missing objects if necessary
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

if not model_path.exists():
    raise FileNotFoundError(f"Saved model not found: {model_path}")

model = joblib.load(model_path)

target_column = "recommended_career_domain"
model_features = [
    column for column in train_df.columns
    if column != target_column
]
career_labels = list(model.classes_)

# Verify feature alignment
missing_features = [
    feature for feature in model_features
    if feature not in test_df.columns
]
model_feature_count = getattr(model, "n_features_in_", None)

print("\nTraining data shape:", train_df.shape)
print("Test data shape:", test_df.shape)
print("Model type:", type(model).__name__)
print("Expected model input features:", model_feature_count)
print("Training feature count:", len(model_features))
print("Missing test features:", missing_features)
print("Career domains:", len(career_labels))
print("\nCareer labels:")
print(career_labels)

assert not missing_features, "Test dataset is missing model features."
assert model_feature_count == len(model_features), (
    "Model feature count does not match the training feature count."
)

print("\nPASS: Model and dataset feature counts match.")

Project directory: C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset
Training CSV exists: True
Test CSV exists: True
Saved model exists: True
Streamlit installed: False

Training data shape: (1920, 26)
Test data shape: (480, 26)
Model type: VotingClassifier
Expected model input features: 25
Training feature count: 25
Missing test features: []
Career domains: 12

Career labels:
['Business & Management', 'Creative Arts & Media', 'Cybersecurity', 'Data Science & Analytics', 'Digital Marketing', 'Education & Research', 'Engineering & Core Technology', 'Finance & Accounting', 'Healthcare & Life Sciences', 'Law & Public Service', 'Software Development', 'UI/UX Design']

PASS: Model and dataset feature counts match.


In [130]:
# STEP 2: Inspect model input requirements and install Streamlit if needed

from pathlib import Path
import sys
import subprocess
import importlib.util
import pandas as pd
import numpy as np

# Load training data
project_dir = Path(
    r"C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset"
)

train_df = pd.read_csv(project_dir / "career_guidance_train.csv")
target = "recommended_career_domain"
features = [c for c in train_df.columns if c != target]

# Separate numeric and categorical inputs
numeric_features = train_df[features].select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = [
    c for c in features if c not in numeric_features
]

print("=" * 65)
print("1. NUMERIC FEATURE RANGES")
print("=" * 65)

if numeric_features:
    print(
        train_df[numeric_features]
        .describe()
        .loc[["min", "mean", "max"]]
        .round(2)
        .T
        .to_string()
    )

print("\n" + "=" * 65)
print("2. CATEGORICAL FEATURE OPTIONS")
print("=" * 65)

for col in categorical_features:
    values = sorted(train_df[col].dropna().astype(str).unique().tolist())
    print(f"{col}: {values}")

print("\n" + "=" * 65)
print("3. DATA QUALITY")
print("=" * 65)
print("Duplicate training rows:", train_df.duplicated().sum())
print("Missing values in training data:", int(train_df[features].isna().sum().sum()))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total model features:", len(features))

# Install Streamlit in the SAME Python environment as this notebook
print("\n" + "=" * 65)
print("4. STREAMLIT SETUP")
print("=" * 65)

if importlib.util.find_spec("streamlit") is None:
    print("Installing Streamlit...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "streamlit"],
        check=False
    )
    if result.returncode != 0:
        raise RuntimeError(
            "Streamlit installation failed. Share the error output."
        )

print("Notebook Python:", sys.executable)
print("Streamlit installed:", importlib.util.find_spec("streamlit") is not None)

if importlib.util.find_spec("streamlit") is None:
    raise RuntimeError("Streamlit is not available in this environment.")

print("\nSTEP 2 PASSED: Feature inspection and Streamlit setup completed.")

1. NUMERIC FEATURE RANGES
                            min  mean   max
interest_technology         1.3  7.31  10.0
interest_data               1.0  6.94  10.0
interest_problem_solving    1.1  7.35  10.0
interest_creativity         1.0  7.18  10.0
interest_people             1.0  7.41  10.0
interest_helping            1.0  6.42  10.0
aptitude_logical            1.8  7.85  10.0
aptitude_numerical          2.1  7.51  10.0
aptitude_verbal             2.5  7.27  10.0
aptitude_spatial            2.7  7.56  10.0
aptitude_critical_thinking  2.3  7.34  10.0
trait_openness              3.0  7.47  10.0
trait_conscientiousness     3.2  7.61  10.0
trait_extraversion          2.0  7.29  10.0
trait_agreeableness         1.9  7.53  10.0
trait_emotional_stability   1.9  7.11  10.0
skill_programming           1.0  6.75  10.0
skill_data_analysis         1.0  6.72  10.0
skill_research              1.0  6.20  10.0
skill_communication         1.3  7.55  10.0
skill_leadership            1.0  7.17  10.0

2. CA

In [131]:

from pathlib import Path

project_dir = Path(
    r"C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset"
)

app_code = r'''
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import streamlit as st

st.set_page_config(
    page_title="CareerCompass AI",
    page_icon="🎯",
    layout="wide"
)

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "artifacts" / "career_guidance_model.joblib"
TRAIN_PATH = BASE_DIR / "career_guidance_train.csv"

st.markdown("""
<style>
.block-container {padding-top: 2rem; padding-bottom: 2rem;}
.hero {
    padding: 1.6rem; border-radius: 18px;
    background: linear-gradient(120deg, #172554, #1d4ed8);
    color: white; margin-bottom: 1.2rem;
}
.hero h1 {color: white; margin-bottom: 0.3rem;}
.hero p {color: #dbeafe; margin-bottom: 0;}
.small-note {color: #64748b; font-size: 0.9rem;}
</style>
<div class="hero">
<h1>🎯 CareerCompass AI</h1>
<p>Discover career domains aligned with your interests, abilities, personality and skills.</p>
</div>
""", unsafe_allow_html=True)

@st.cache_resource
def load_resources():
    model = joblib.load(MODEL_PATH)
    train = pd.read_csv(TRAIN_PATH)
    features = [c for c in train.columns
                if c != "recommended_career_domain"]
    return model, train, features

if not MODEL_PATH.exists() or not TRAIN_PATH.exists():
    st.error("Required model or training CSV is missing. Check the project folder.")
    st.stop()

try:
    model, train_df, FEATURES = load_resources()
except Exception as exc:
    st.error(f"Could not load the saved model: {exc}")
    st.stop()

if len(FEATURES) != getattr(model, "n_features_in_", -1):
    st.error("Model and dataset feature counts do not match.")
    st.stop()

CAREER_INFO = {
    "Business & Management": "Business operations, planning, project management and leadership.",
    "Creative Arts & Media": "Visual communication, content creation, media and creative production.",
    "Cybersecurity": "Security operations, risk analysis, threat detection and digital protection.",
    "Data Science & Analytics": "Data analysis, statistics, machine learning and evidence-based decisions.",
    "Digital Marketing": "Digital campaigns, marketing analytics, branding and audience engagement.",
    "Education & Research": "Teaching, academic research, learning design and knowledge development.",
    "Engineering & Core Technology": "Engineering systems, technical problem-solving and core technologies.",
    "Finance & Accounting": "Financial reporting, accounting, budgeting and financial analysis.",
    "Healthcare & Life Sciences": "Healthcare, biological sciences, clinical support and life-science research.",
    "Law & Public Service": "Legal services, policy, governance and public administration.",
    "Software Development": "Programming, software engineering, application development and testing.",
    "UI/UX Design": "User research, interface design, usability and digital product experiences."
}

st.caption("Complete your profile and rate each area honestly. You can adjust the ratings before generating recommendations.")

with st.form("career_profile_form"):
    st.subheader("1. Student profile")
    p1, p2 = st.columns(2)

    age_group = p1.selectbox(
        "Age group",
        ["15-18", "19-22", "23-30", "31+"]
    )

    education_options = sorted(
        train_df["education_level"].dropna().unique().tolist()
    )
    education_options += ["Other / Not listed or no formal qualification"]
    selected_education = p2.selectbox(
        "Highest/current qualification",
        education_options,
        help="Choose the closest exact qualification. If it is not listed, the model will receive a missing value for this feature."
    )
    education_level = (
        selected_education
        if selected_education in set(train_df["education_level"].dropna())
        else None
    )

    p3, p4 = st.columns(2)
    experience_stage = p3.selectbox(
        "Current education/work stage",
        ["Student", "Entry-level", "1-3 years", "4+ years"]
    )
    region_type = p4.selectbox(
        "Residential area",
        ["Urban", "Semi-urban", "Rural"]
    )

    def rating_group(title, descriptions):
        st.subheader(title)
        st.caption("Rate each statement from 1 (low) to 10 (high).")
        values = {}
        cols = st.columns(2)
        for i, (feature, label) in enumerate(descriptions.items()):
            with cols[i % 2]:
                values[feature] = st.slider(
                    label, min_value=1, max_value=10,
                    value=7, key=f"rating_{feature}"
                )
        return values

    interests = rating_group("2. Interests", {
        "interest_technology": "Interest in technology",
        "interest_data": "Interest in data and patterns",
        "interest_problem_solving": "Interest in problem-solving",
        "interest_creativity": "Interest in creativity and design",
        "interest_people": "Interest in working with people",
        "interest_helping": "Interest in helping others"
    })

    aptitude = rating_group("3. Self-assessed abilities", {
        "aptitude_logical": "Logical reasoning",
        "aptitude_numerical": "Numerical reasoning",
        "aptitude_verbal": "Verbal ability",
        "aptitude_spatial": "Spatial reasoning",
        "aptitude_critical_thinking": "Critical thinking"
    })

    personality = rating_group("4. Personality traits", {
        "trait_openness": "Openness to new ideas",
        "trait_conscientiousness": "Planning and responsibility",
        "trait_extraversion": "Comfort in social interaction",
        "trait_agreeableness": "Cooperation and consideration",
        "trait_emotional_stability": "Emotional stability"
    })

    skills = rating_group("5. Skills", {
        "skill_programming": "Programming",
        "skill_data_analysis": "Data analysis",
        "skill_research": "Research",
        "skill_communication": "Communication",
        "skill_leadership": "Leadership"
    })

    submitted = st.form_submit_button(
        "✨ Generate my career recommendations",
        type="primary",
        use_container_width=True
    )

if submitted:
    profile = {
        "age_group": age_group,
        "education_level": education_level,
        "experience_stage": experience_stage,
        "region_type": region_type,
        **interests,
        **aptitude,
        **personality,
        **skills
    }

    input_df = pd.DataFrame([profile], columns=FEATURES)

    if input_df.shape[1] != 25 or input_df.isna().all(axis=1).iloc[0]:
        st.error("Invalid input. Please review your profile.")
        st.stop()

    try:
        probabilities = model.predict_proba(input_df)[0]
        classes = list(model.classes_)
        order = np.argsort(probabilities)[::-1]
        top_indices = order[:3]

        st.divider()
        st.header("Your Top 3 Career Recommendations")
        st.write(
            "These are the model's highest-ranked career domains for the profile you entered."
        )

        cols = st.columns(3)
        for rank, idx in enumerate(top_indices, start=1):
            career = classes[idx]
            score = float(probabilities[idx]) * 100
            with cols[rank - 1]:
                st.metric(f"#{rank} Recommendation", career, f"{score:.1f}% model score")
                st.progress(float(np.clip(probabilities[idx], 0, 1)))
                st.write(CAREER_INFO.get(career, "Explore roles in this career domain."))
                if rank == 1:
                    st.success("Highest-ranked result")

        with st.expander("View all career-domain scores"):
            all_results = pd.DataFrame({
                "Career domain": [classes[i] for i in order],
                "Relative model score (%)": [
                    round(float(probabilities[i]) * 100, 2) for i in order
                ]
            })
            st.dataframe(all_results, use_container_width=True, hide_index=True)

        st.subheader("How to use these results")
        st.markdown("""
        - Explore the recommended domains and their typical roles.
        - Compare them with your own interests and educational goals.
        - Research course requirements, practical skills, internships and job opportunities.
        - Treat the results as guidance for exploration, not as a final career decision.
        """)

        st.info(
            "Important limitation: this prototype uses synthetic training data. "
            "Its scores are model outputs, not validated probabilities of career success. "
            "Real-world reliability requires testing with consented, representative data "
            "and evaluation by career-guidance professionals."
        )

    except Exception as exc:
        st.error(f"Recommendation generation failed: {exc}")

with st.sidebar:
    st.title("About the project")
    st.write("**AI-Powered Career Guidance Platform**")
    st.write("Psychometric Assessment & ML-Based Career Matching")
    st.divider()
    st.write("**Model:** Saved VotingClassifier")
    st.write("**Input features:** 25")
    st.write("**Career domains:** 12")
    st.write("**Dataset:** Synthetic")
    st.caption("Developed as an academic ML prototype.")
'''

app_path = project_dir / "app.py"
app_path.write_text(app_code, encoding="utf-8")

# Verify the file exists and has content
assert app_path.exists(), "app.py was not created."
assert app_path.stat().st_size > 1000, "app.py appears incomplete."

print("Frontend file created successfully.")
print("Location:", app_path)
print("File size:", app_path.stat().st_size, "bytes")
print("Model exists:", (project_dir / "artifacts" / "career_guidance_model.joblib").exists())
print("Training data exists:", (project_dir / "career_guidance_train.csv").exists())
print("\nNext: run the syntax check in Step 4.")


Frontend file created successfully.
Location: C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset\app.py
File size: 9387 bytes
Model exists: True
Training data exists: True

Next: run the syntax check in Step 4.


In [132]:
import ast

app_path = project_dir / "app.py"
source = app_path.read_text(encoding="utf-8")

ast.parse(source)
print("PASS: app.py has valid Python syntax.")
print("App file:", app_path)

PASS: app.py has valid Python syntax.
App file: C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset\app.py


In [133]:
# STEP 4: Inspect the actual questionnaire and scoring mappings

from pathlib import Path
import pandas as pd

project_dir = Path(
    r"C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset"
)

excel_path = project_dir / "AI_Career_Guidance_Questions.xlsx"

assert excel_path.exists(), f"Questionnaire not found: {excel_path}"

xls = pd.ExcelFile(excel_path)
print("Workbook sheets:", xls.sheet_names)

questions_df = pd.read_excel(
    excel_path,
    sheet_name="Questions Dataset"
)

print("\nQuestionnaire shape:", questions_df.shape)
print("Columns:", questions_df.columns.tolist())

print("\nQuestion counts by section:")
print(questions_df["Section"].value_counts(dropna=False).to_string())

print("\nAnswer types:")
print(questions_df["Answer_Type"].value_counts(dropna=False).to_string())

print("\nFirst five questions:")
display(
    questions_df[
        ["Question_ID", "Section", "Question_Text", "Answer_Type",
         "Answer_Options", "Scoring_Method", "Feature_Mapping"]
    ].head()
)

print("\nLast ten questions:")
display(
    questions_df[
        ["Question_ID", "Section", "Question_Text", "Answer_Type",
         "Answer_Options", "Scoring_Method", "Feature_Mapping"]
    ].tail(10)
)

print("\nDuplicate question IDs:",
      questions_df["Question_ID"].duplicated().sum())
print("Missing question text:",
      questions_df["Question_Text"].isna().sum())

print("\nPASS: Questionnaire loaded for integration.")

Workbook sheets: ['Questions Dataset']

Questionnaire shape: (123, 8)
Columns: ['Question_ID', 'Section', 'Question_Text', 'Answer_Type', 'Answer_Options', 'Scoring_Method', 'Feature_Mapping', 'Purpose']

Question counts by section:
Section
Education & Career Preferences    80
Aptitude                          12
Skills                            11
Interest                          10
Personality                       10

Answer types:
Answer_Type
MCQ                        89
Skill Scale (1-5 + N/A)    11
Rating scale (1-5)         10
Agreement Scale (1-5)      10
Dropdown                    3

First five questions:


,Question_ID,Section,Question_Text,Answer_Type,Answer_Options,Scoring_Method,Feature_Mapping
0,Q001,Interest,How interested are you in writing scripts or c...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN
1,Q002,Interest,How interested are you in analyzing large data...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN
2,Q003,Interest,How interested are you in finding vulnerabilit...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN
3,Q004,Interest,How interested are you in sketching out wirefr...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN
4,Q005,Interest,How interested are you in coordinating team me...,Rating scale (1-5),1 - Not interested at all to 5 - Extremely int...,NaN,NaN



Last ten questions:


,Question_ID,Section,Question_Text,Answer_Type,Answer_Options,Scoring_Method,Feature_Mapping
113,Q114,Education & Career Preferences,Are you able to separate your personal ego fro...,MCQ,"Yes easily, Sometimes difficult, No",NaN,NaN
114,Q115,Education & Career Preferences,Do you prefer working solo in a studio or on a...,MCQ,"Solo, Large crew",NaN,NaN
115,Q116,Education & Career Preferences,Do you prefer arguing cases in a courtroom or ...,MCQ,"Courtroom/Litigation, Corporate/Drafting, N/A",NaN,NaN
116,Q117,Education & Career Preferences,How important is it for your work to directly ...,MCQ,"Very important, Somewhat, Not important",NaN,NaN
117,Q118,Education & Career Preferences,Are you comfortable defending clients or polic...,MCQ,"Yes, No, Depends on the issue",NaN,NaN
118,Q119,Education & Career Preferences,Do you prefer the fast pace of a corporate law...,MCQ,"Corporate Law Firm, Government/Public Sector, ...",NaN,NaN
119,Q120,Education & Career Preferences,How comfortable are you reading and synthesizi...,MCQ,"Very comfortable, Somewhat, Uncomfortable",NaN,NaN
120,Q121,Aptitude,Read the passage: 'Regular exercise can improv...,MCQ,"{""A"": ""Exercise only improves sleep quality."",...","Correct option: B; correct=1, incorrect=0; pro...",aptitude_verbal
121,Q122,Aptitude,"A square has vertices labeled clockwise A, B, ...",MCQ,"{""A"": ""A"", ""B"": ""B"", ""C"": ""C"", ""D"": ""D""}","Correct option: A; correct=1, incorrect=0; pro...",aptitude_spatial
122,Q123,Skills,How would you rate your ability to explain com...,Skill Scale (1-5 + N/A),"1-Beginner to 5-Expert, N/A",Convert a valid 1-5 rating to the model's 1-10...,skill_communication



Duplicate question IDs: 0
Missing question text: 0

PASS: Questionnaire loaded for integration.


In [134]:

from pathlib import Path

project_dir = Path(
    r"C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset"
)

app_code = r'''
from pathlib import Path
from datetime import datetime, timezone
import json
import uuid
import ast

import numpy as np
import pandas as pd
import joblib
import streamlit as st

BASE = Path(__file__).resolve().parent
MODEL_PATH = BASE / "artifacts" / "career_guidance_model.joblib"
TRAIN_PATH = BASE / "career_guidance_train.csv"
QUESTIONS_PATH = BASE / "AI_Career_Guidance_Questions.xlsx"
RESULTS_PATH = BASE / "assessment_results.csv"
TARGET = "recommended_career_domain"

st.set_page_config(
    page_title="CareerCompass AI",
    page_icon="🎯",
    layout="wide"
)

st.title("🎯 CareerCompass AI")
st.subheader("Psychometric Assessment & ML-Based Career Matching")
st.write(
    "Answer the questionnaire honestly. Your mapped responses are converted "
    "into model features and used to rank career domains."
)

st.info(
    "Academic prototype: the model was trained on synthetic data. "
    "Recommendations are exploratory guidance, not a validated psychological "
    "assessment or a guarantee of career success."
)

if not all(p.exists() for p in [MODEL_PATH, TRAIN_PATH, QUESTIONS_PATH]):
    st.error("A required model, dataset, or questionnaire file is missing.")
    st.stop()

@st.cache_resource
def load_assets():
    model = joblib.load(MODEL_PATH)
    train = pd.read_csv(TRAIN_PATH)
    questions = pd.read_excel(QUESTIONS_PATH, sheet_name="Questions Dataset")
    return model, train, questions

model, train_df, questions_df = load_assets()
FEATURES = [c for c in train_df.columns if c != TARGET]
CATEGORICAL = ["age_group", "education_level", "experience_stage", "region_type"]
NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]

if len(FEATURES) != 25 or getattr(model, "n_features_in_", None) != 25:
    st.error("Model feature validation failed.")
    st.stop()

# Explicit, reviewable mappings: unmapped questions are stored but not
# arbitrarily converted into ML features.
QUESTION_FEATURES = {
    "Q001": ["interest_technology", "interest_problem_solving"],
    "Q002": ["interest_data"],
    "Q003": ["interest_technology", "interest_problem_solving"],
    "Q004": ["interest_creativity"],
    "Q005": ["interest_people"],
    "Q006": ["interest_helping"],
    "Q007": ["interest_people", "interest_helping"],
    "Q008": ["interest_creativity", "interest_data"],
    "Q009": ["interest_data", "interest_problem_solving"],
    "Q010": ["interest_technology", "interest_problem_solving"],
    "Q021": ["trait_extraversion", "trait_agreeableness"],
    "Q022": ["trait_conscientiousness"],
    "Q023": ["trait_extraversion"],
    "Q024": ["trait_conscientiousness"],
    "Q025": ["trait_openness"],
    "Q026": ["trait_emotional_stability"],
    "Q027": ["trait_conscientiousness"],
    "Q028": ["trait_extraversion"],
    "Q029": ["trait_conscientiousness"],
    "Q030": ["trait_openness"],
    "Q031": ["skill_programming"],
    "Q032": ["skill_data_analysis"],
    "Q036": ["skill_leadership"],
    "Q040": ["skill_research"],
    "Q123": ["skill_communication"],
}

APTITUDE_KEYS = {
    "Q011": "B", "Q012": "D", "Q013": "C", "Q014": "A",
    "Q015": "C", "Q016": "B", "Q017": "B", "Q018": "A",
    "Q019": "C", "Q020": "C", "Q121": "B", "Q122": "A"
}

APTITUDE_FEATURE_QUESTIONS = {
    "aptitude_logical": ["Q012", "Q014", "Q016", "Q019"],
    "aptitude_numerical": ["Q011", "Q013", "Q015", "Q017", "Q020"],
    "aptitude_critical_thinking": ["Q015", "Q016", "Q018", "Q019"],
    "aptitude_verbal": ["Q121"],
    "aptitude_spatial": ["Q122"],
}

MIN_ANSWERED = {
    "aptitude_logical": 3,
    "aptitude_numerical": 3,
    "aptitude_critical_thinking": 3,
    "aptitude_verbal": 1,
    "aptitude_spatial": 1,
}

CAREER_INFO = {
    "Business & Management": "Business operations, planning and management.",
    "Creative Arts & Media": "Creative production, media and visual communication.",
    "Cybersecurity": "Security operations, risk analysis and digital protection.",
    "Data Science & Analytics": "Data analysis, statistics and machine learning.",
    "Digital Marketing": "Digital campaigns, branding and marketing analytics.",
    "Education & Research": "Teaching, academic research and learning design.",
    "Engineering & Core Technology": "Engineering systems and technical problem-solving.",
    "Finance & Accounting": "Accounting, budgeting and financial analysis.",
    "Healthcare & Life Sciences": "Healthcare and biological or life-science fields.",
    "Law & Public Service": "Legal services, governance and public administration.",
    "Software Development": "Programming, software engineering and application development.",
    "UI/UX Design": "User research, interface design and usability."
}

def parse_options(raw):
    if raw is None or pd.isna(raw):
        return ["Not answered"]

    text = str(raw).strip()

    # Some questions store their options as a JSON object.
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return [f"{k}: {v}" for k, v in obj.items()]
        if isinstance(obj, list):
            return [str(x) for x in obj]
    except (json.JSONDecodeError, TypeError):
        pass

    # Otherwise split the plain-text options.
    values = [x.strip() for x in text.split(",") if x.strip()]
    return values or [text]

def clean_answer(value):
    if value is None:
        return None
    if isinstance(value, (np.integer, np.floating)):
        value = value.item()
    if isinstance(value, float) and np.isnan(value):
        return None
    return value

def score_assessment(answers):
    # Collect rating-derived feature values. A 1-5 response is mapped
    # linearly to the model's 2-10 scale.
    grouped = {feature: [] for feature in NUMERIC}

    for qid, mapped_features in QUESTION_FEATURES.items():
        answer = answers.get(qid)
        if answer is None or answer == "N/A":
            continue

        try:
            rating = float(answer)
        except (TypeError, ValueError):
            continue

        if 1 <= rating <= 5:
            score = rating * 2
            for feature in mapped_features:
                if feature in grouped:
                    grouped[feature].append(score)

    feature_scores = {}
    for feature, values in grouped.items():
        if values:
            feature_scores[feature] = round(float(np.mean(values)), 2)

    # Score aptitude MCQs using the stored answer key.
    for feature, qids in APTITUDE_FEATURE_QUESTIONS.items():
        attempted = [
            qid for qid in qids
            if answers.get(qid) in ["A", "B", "C", "D"]
        ]
        correct = sum(
            answers[qid] == APTITUDE_KEYS[qid] for qid in attempted
        )

        if len(attempted) >= MIN_ANSWERED[feature]:
            accuracy = correct / len(attempted)
            feature_scores[feature] = round(1 + accuracy * 9, 2)
        else:
            feature_scores[feature] = np.nan

    # Ensure every model numeric feature is present.
    for feature in NUMERIC:
        feature_scores.setdefault(feature, np.nan)

    return feature_scores

def append_result(record):
    results = pd.DataFrame([record])
    if RESULTS_PATH.exists():
        results.to_csv(
            RESULTS_PATH, mode="a", header=False,
            index=False, encoding="utf-8-sig"
        )
    else:
        results.to_csv(
            RESULTS_PATH, index=False, encoding="utf-8-sig"
        )

# Three additional demographic prompts required by the trained model.
st.sidebar.header("Assessment")
st.sidebar.write(f"Original questionnaire: {len(questions_df)} questions")
st.sidebar.write("Demographic prompts: 3")
st.sidebar.write(f"Total prompts: {len(questions_df) + 3}")
st.sidebar.caption("Your submitted answers are saved locally in a CSV file.")

with st.form("full_assessment"):
    st.header("Part A — Student profile")

    c1, c2 = st.columns(2)
    age_group = c1.selectbox(
        "Q124 — Your age group",
        ["15-18", "19-22", "23-30", "31+"]
    )
    region_type = c2.selectbox(
        "Q125 — Your residential area",
        ["Urban", "Semi-urban", "Rural", "Prefer not to say"]
    )

    education_options = sorted(
        train_df["education_level"].dropna().astype(str).unique().tolist()
    )
    education_options += ["Other / Not listed", "No formal qualification"]
    education_choice = st.selectbox(
        "Highest/current qualification",
        education_options
    )
    education_level = (
        education_choice
        if education_choice in set(train_df["education_level"].dropna())
        else None
    )

    experience_stage = st.selectbox(
        "Q126 — Your current education/work stage",
        [
            "Student",
            "Entry-level",
            "1-3 years",
            "4+ years",
            "Other / Prefer not to say"
        ]
    )
    if experience_stage == "Other / Prefer not to say":
        experience_stage = None

    st.header("Part B — Questionnaire")
    st.caption(
        "Questions are grouped by their original sections. "
        "Questions without a defined model mapping are still recorded."
    )

    answers = {}
    for section, section_df in questions_df.groupby("Section", sort=False):
        with st.expander(f"{section} ({len(section_df)} questions)", expanded=(section == "Interest")):
            for _, q in section_df.iterrows():
                qid = str(q["Question_ID"]).strip()
                question_text = str(q["Question_Text"]).strip()
                answer_type = str(q["Answer_Type"]).strip()
                options = parse_options(q.get("Answer_Options"))

                if answer_type == "MCQ":
                    # Display option labels, but store the option key when
                    # the workbook supplies a JSON A/B/C/D object.
                    raw = q.get("Answer_Options")
                    try:
                        parsed = json.loads(str(raw))
                    except (json.JSONDecodeError, TypeError):
                        parsed = None

                    if isinstance(parsed, dict):
                        labels = list(parsed.values())
                        keys = list(parsed.keys())
                        selected = st.radio(
                            f"{qid}. {question_text}",
                            ["Not answered"] + labels,
                            key=qid
                        )
                        if selected == "Not answered":
                            answers[qid] = None
                        else:
                            answers[qid] = keys[labels.index(selected)]
                    else:
                        selected = st.radio(
                            f"{qid}. {question_text}",
                            ["Not answered"] + options,
                            key=qid
                        )
                        answers[qid] = None if selected == "Not answered" else selected

                elif answer_type in [
                    "Rating scale (1-5)",
                    "Agreement Scale (1-5)",
                    "Skill Scale (1-5 + N/A)"
                ]:
                    choices = ["Not answered", "1", "2", "3", "4", "5"]
                    if "N/A" in answer_type:
                        choices.append("N/A")
                    selected = st.radio(
                        f"{qid}. {question_text}",
                        choices,
                        horizontal=True,
                        key=qid
                    )
                    if selected in ["Not answered", "N/A"]:
                        answers[qid] = None if selected == "Not answered" else "N/A"
                    else:
                        answers[qid] = int(selected)

                else:
                    selected = st.selectbox(
                        f"{qid}. {question_text}",
                        ["Not answered"] + options,
                        key=qid
                    )
                    answers[qid] = None if selected == "Not answered" else selected

    submitted = st.form_submit_button(
        "Calculate scores and recommend careers",
        type="primary",
        use_container_width=True
    )

if submitted:
    # Validate required demographic inputs and minimum MCQ coverage.
    answered_count = sum(v is not None for v in answers.values())
    if answered_count == 0:
        st.error("Please answer the questionnaire before submitting.")
        st.stop()

    scores = score_assessment(answers)

    # Map the complete response set into exactly the trained 25 features.
    profile = {
        "age_group": age_group,
        "education_level": education_level,
        "experience_stage": experience_stage,
        "region_type": None if region_type == "Prefer not to say" else region_type,
        **scores
    }

    input_df = pd.DataFrame([profile], columns=FEATURES)

    try:
        ranked = model.predict_proba(input_df)[0]
        classes = list(model.classes_)
        order = np.argsort(ranked)[::-1]
        top = [
            {
                "rank": rank,
                "career_domain": classes[i],
                "model_score_percent": round(float(ranked[i]) * 100, 2)
            }
            for rank, i in enumerate(order[:3], start=1)
        ]

        assessment_id = str(uuid.uuid4())
        saved_at = datetime.now(timezone.utc).isoformat()

        # Save every question response, derived features and recommendations.
        record = {
            "assessment_id": assessment_id,
            "submitted_at_utc": saved_at,
            "answered_question_count": answered_count,
            "total_original_questions": len(questions_df),
            "age_group": age_group,
            "education_level": education_level,
            "experience_stage": experience_stage,
            "region_type": region_type,
            "answers_json": json.dumps(
                {k: clean_answer(v) for k, v in answers.items()},
                ensure_ascii=False
            ),
            "feature_scores_json": json.dumps(
                {k: clean_answer(v) for k, v in scores.items()},
                ensure_ascii=False
            ),
            "recommendations_json": json.dumps(top, ensure_ascii=False)
        }
        append_result(record)

        st.success("Assessment calculated and saved successfully.")
        st.caption(f"Assessment ID: {assessment_id}")
        st.metric("Questions answered", f"{answered_count}/{len(questions_df)}")

        st.header("Your Top 3 Career Recommendations")
        columns = st.columns(3)
        for rank, item in enumerate(top):
            with columns[rank]:
                st.markdown(f"### #{item['rank']}")
                st.subheader(item["career_domain"])
                st.metric("Relative model score", f"{item['model_score_percent']:.2f}%")
                st.progress(float(np.clip(
                    item["model_score_percent"] / 100, 0, 1
                )))
                st.write(CAREER_INFO.get(item["career_domain"], ""))

        with st.expander("View calculated model features"):
            score_table = pd.DataFrame(
                [{"Feature": k, "Calculated score": clean_answer(v)}
                 for k, v in scores.items()]
            )
            st.dataframe(score_table, use_container_width=True, hide_index=True)

        with st.expander("View all career-domain rankings"):
            all_scores = pd.DataFrame({
                "Career domain": [classes[i] for i in order],
                "Relative model score (%)": [
                    round(float(ranked[i]) * 100, 2) for i in order
                ]
            })
            st.dataframe(all_scores, use_container_width=True, hide_index=True)

        st.download_button(
            "Download this assessment result",
            data=json.dumps(record, ensure_ascii=False, indent=2),
            file_name=f"career_assessment_{assessment_id[:8]}.json",
            mime="application/json"
        )

        st.warning(
            "Scoring limitation: only explicitly mapped questions contribute "
            "to feature scores. Some features may be missing and are imputed by "
            "the saved model. The current aptitude MCQ key and score conversion "
            "are provisional and require subject-matter validation. Model scores "
            "are not validated probabilities of career success."
        )

    except Exception as exc:
        st.error(f"Assessment processing failed: {exc}")

st.divider()
st.caption(
    "Data storage: local CSV. This prototype is not a secure multi-user "
    "production system; avoid entering sensitive personal information."
)
'''

app_path = project_dir / "app.py"
app_path.write_text(app_code, encoding="utf-8")

# Syntax verification
import ast
ast.parse(app_code)

print("PASS: Questionnaire-based app.py created.")
print("Path:", app_path)
print("Original questionnaire questions:", len(
    pd.read_excel(project_dir / "AI_Career_Guidance_Questions.xlsx",
                  sheet_name="Questions Dataset")
))
print("Additional demographic prompts: 3")
print("Syntax check: passed")
print("Original Excel and saved ML model were not modified.")


PASS: Questionnaire-based app.py created.
Path: C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset\app.py
Original questionnaire questions: 123
Additional demographic prompts: 3
Syntax check: passed
Original Excel and saved ML model were not modified.


In [135]:
import sys
import subprocess
from pathlib import Path

project_dir = Path(
    r"C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset"
)

checks = {
    "NumPy": "import numpy; print(numpy.__version__)",
    "Pandas": "import pandas; print(pandas.__version__)",
    "SciPy": "import scipy; print(scipy.__version__)",
    "Scikit-learn": "import sklearn; print(sklearn.__version__)",
    "Joblib": "import joblib; print(joblib.__version__)",
    "Streamlit": "import streamlit; print(streamlit.__version__)",
    "Model prediction": r"""
import pandas as pd, joblib
from pathlib import Path
p = Path(r'C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset')
train = pd.read_csv(p / 'career_guidance_train.csv')
model = joblib.load(p / 'artifacts' / 'career_guidance_model.joblib')
X = train.drop(columns=['recommended_career_domain']).iloc[[0]]
print('Input shape:', X.shape)
print('Prediction:', model.predict(X))
print('Scores:', model.predict_proba(X)[0])
"""
}

print("Python executable:", sys.executable)
print("=" * 65)

for name, code in checks.items():
    print(f"\n--- {name} ---")
    result = subprocess.run(
        [sys.executable, "-c", code],
        capture_output=True,
        text=True,
        timeout=90
    )
    print("Exit code:", result.returncode)
    if result.stdout.strip():
        print("OUTPUT:\n", result.stdout.strip())
    if result.stderr.strip():
        print("ERROR:\n", result.stderr.strip()[-3500:])
    if result.returncode == 0:
        print("STATUS: PASS")
    else:
        print("STATUS: FAILED")

Python executable: C:\B-TECH\AnacondaFiles\envs\ml\python.exe

--- NumPy ---
Exit code: 0
OUTPUT:
 2.4.6
STATUS: PASS

--- Pandas ---
Exit code: 0
OUTPUT:
 3.0.5
STATUS: PASS

--- SciPy ---
Exit code: 0
OUTPUT:
 1.17.1
STATUS: PASS

--- Scikit-learn ---
Exit code: 0
OUTPUT:
 1.9.0
STATUS: PASS

--- Joblib ---
Exit code: 0
OUTPUT:
 1.5.3
STATUS: PASS

--- Streamlit ---
Exit code: 0
OUTPUT:
 1.66.1
STATUS: PASS

--- Model prediction ---
Exit code: 0
OUTPUT:
 Input shape: (1, 25)
Prediction: ['Engineering & Core Technology']
Scores: [2.31798035e-08 9.00149361e-10 1.70713985e-07 1.77309859e-03
 3.41083239e-09 1.52697954e-05 9.96537906e-01 5.82893872e-04
 7.52894716e-08 3.92117252e-09 1.09055206e-03 1.84153610e-09]
STATUS: PASS


In [136]:
from pathlib import Path
import ast

project_dir = Path(
    r"C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset"
)
app_path = project_dir / "app.py"
source = app_path.read_text(encoding="utf-8")

old_feature_display = '''            st.dataframe(score_table, use_container_width=True, hide_index=True)'''
new_feature_display = '''            st.markdown("| Feature | Calculated score |")
            st.markdown("|---|---:|")
            for _, row in score_table.iterrows():
                value = row["Calculated score"]
                display_value = "Not calculated" if pd.isna(value) else f"{float(value):.2f}"
                st.markdown(f"| {row['Feature']} | {display_value} |")'''

old_rank_display = '''            st.dataframe(all_scores, use_container_width=True, hide_index=True)'''
new_rank_display = '''            st.markdown("| Career domain | Relative model score (%) |")
            st.markdown("|---|---:|")
            for _, row in all_scores.iterrows():
                st.markdown(f"| {row['Career domain']} | {row['Relative model score (%)']:.2f} |")'''

if old_feature_display not in source:
    print("Feature display code was not found exactly; no change made to that section.")
else:
    source = source.replace(old_feature_display, new_feature_display, 1)
    print("Feature display updated.")

if old_rank_display not in source:
    print("Ranking display code was not found exactly; no change made to that section.")
else:
    source = source.replace(old_rank_display, new_rank_display, 1)
    print("Ranking display updated.")

ast.parse(source)
app_path.write_text(source, encoding="utf-8")

print("Syntax check: PASS")
print("Updated file:", app_path)

Feature display updated.
Ranking display updated.
Syntax check: PASS
Updated file: C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset\app.py


In [137]:
from pathlib import Path
import ast

project_dir = Path(
    r"C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset"
)
app_path = project_dir / "app.py"
source = app_path.read_text(encoding="utf-8")

old = '''                    else:
                        selected = st.radio(
                            f"{qid}. {question_text}",
                            ["Not answered"] + options,
                            key=qid
                        )
                        answers[qid] = None if selected == "Not answered" else selected
'''

new = '''                    else:
                        selected = st.radio(
                            f"{qid}. {question_text}",
                            ["Not answered"] + options,
                            key=qid
                        )

                        if selected == "Not answered":
                            answers[qid] = None
                        elif qid in APTITUDE_KEYS:
                            # Convert the selected option's position to A/B/C/D.
                            # This allows the existing answer key to score the response.
                            option_index = options.index(selected)
                            answers[qid] = (
                                "ABCD"[option_index]
                                if option_index < 4
                                else selected
                            )
                        else:
                            answers[qid] = selected
'''

if old not in source:
    raise RuntimeError(
        "Could not find the expected MCQ code. No changes were written."
    )

source = source.replace(old, new, 1)
ast.parse(source)
app_path.write_text(source, encoding="utf-8")

print("PASS: MCQ answer-storage logic updated.")
print("PASS: Python syntax validated.")
print("Updated:", app_path)

PASS: MCQ answer-storage logic updated.
PASS: Python syntax validated.
Updated: C:\B-TECH\ACADEMIC\MLLAB\ML_PROJECT\AI_Career_Guidance_Dataset_Package\AI_Career_Guidance_Dataset\app.py
